# RT-DETR 베이스라인

## 목적

- transformer 계열 검출 모델인 RT-DETR로 베이스라인을 만든다. 팀에서 돌리는 YOLO26(1단계 CNN), Faster R-CNN(2단계 CNN)과 계열이 다른 모델이다.
- 데이터 변환부터 학습, test 제출, val 분석까지 한 번 끝까지 이어 본다.

## 데이터

- 대회 train 데이터에 팀 공용 라벨 수정 목록(`data/label_fix/label_fixes.csv`)을 적용해서 쓴다.
- train/val 분할은 1차 EDA에서 고정한 분할(seed 410, `data/processed/split.csv`)을 그대로 쓴다. train 185장, val 47장이다.

## 목차

| 섹션 | 내용 |
|---|---|
| 0. 사전 설정 | 라이브러리, 경로, GPU, 한글 글꼴 |
| 1. RT-DETR를 고른 이유 | 팀 안에서의 위치, 다른 후보와 비교, 한계 |
| 2. 데이터 준비 | 라벨 읽기, 라벨 수정 적용, 분할 붙이기, YOLO 형식 변환, 변환 확인 |
| 3. 학습 설정 | 학습 인자, 확인 실행, W&B 기록 |
| 4. 학습 | GPU 메모리 확인, seed별 학습, 학습 곡선, train과 val 점수 비교 |
| 5. test 추론과 제출 | seed별 test 예측과 제출 CSV, 파일 점검, Kaggle 제출 결과, test 예측 분석 |
| 6. val 분석 | 클래스별 AP와 약의 성질, 박스 단위 에러 분류, PR 곡선 |
| 7. 예측 보기 | val 사진에 예측과 정답 그리기 |
| 8. 정리 | 결과 요약, 점수를 깎은 원인, 2차 실험의 방향 |

## 실행 방법

- 대회 데이터 압축을 `data/raw` 아래에 풀어 둔다.
- `data/processed/split.csv`와 `data/label_fix/label_fixes.csv`가 있어야 한다.
- W&B에 기록하려면 터미널에서 `wandb login`을 한 번 해 둔다. W&B 팀 `codeit-prj1-team2`의 멤버여야 한다.
- 처음부터 순서대로 실행한다. 3-1의 `SMOKE_TEST = True`로 짧은 확인 실행을 먼저 하고, 문제가 없으면 `False`로 바꿔 본 학습을 한다.
- 학습이 끝난 뒤에는 4번을 다시 실행하지 않는다(학습을 다시 시작한다). 5번부터는 0번 두 셀과 3-1, 3-2만 먼저 실행하면 된다.

## 0. 사전 설정

- 경로는 `pyproject.toml`이 있는 레포 최상위 폴더를 기준으로 잡는다. 노트북을 어느 폴더에서 열어도 같은 경로를 쓰기 위해서다. 출력할 때는 내 컴퓨터의 절대경로 대신 레포 기준 상대경로로 보여 준다.
- 학습에 필요한 파일(원본 데이터, 분할 파일, 라벨 수정 목록)이 있는지 먼저 확인한다. 없으면 학습 도중이 아니라 여기서 멈추게 한다.
- 실행 환경(Python, PyTorch, CUDA, Ultralytics, W&B, GPU)을 출력해 둔다. 결과가 다르게 나올 때 환경 차이부터 확인하기 위해서다.
- 사전학습 가중치는 레포의 `weights` 폴더에 둔다. 없으면 처음 학습할 때 내려받는다.
- 그래프의 한글이 깨지지 않게 한글 글꼴을 설정한다.

In [ ]:
import json
import platform
import shutil
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import RTDETR, settings

# 경로: pyproject.toml이 있는 폴더를 레포 최상위로 본다
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
assert ROOT is not None, "pyproject.toml이 있는 레포 폴더를 찾지 못했습니다. 레포 안에서 노트북을 실행하세요."


def rel(path):
    """레포 최상위 기준 상대경로를 / 구분자로 돌려준다. (내 컴퓨터의 절대경로를 출력하지 않기 위해)"""
    return Path(path).relative_to(ROOT).as_posix()


found = [p for p in sorted((ROOT / "data" / "raw").rglob("train_images")) if p.is_dir()]
assert len(found) == 1, f"data/raw 아래에 train_images 폴더가 1개여야 합니다. 찾은 수: {len(found)}"
RAW_DIR = found[0].parent
TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"

SPLIT_CSV = ROOT / "data" / "processed" / "split.csv"                          # 1차 EDA 분할 (seed 410)
FIX_CSV = ROOT / "data" / "label_fix" / "label_fixes.csv"                      # 팀 공용 라벨 수정 목록
YOLO_DIR = ROOT / "data" / "processed" / "yolo"                                # YOLO 형식 데이터를 만들 곳
DATA_YAML = YOLO_DIR / "data.yaml"                                             # YOLO 데이터 설정 파일 (2-4에서 만든다)
RUNS_DIR = ROOT / "runs"                                                       # 학습 결과를 쌓을 곳
WEIGHTS_DIR = ROOT / "weights"                                                 # 사전학습 가중치를 둘 곳

for path in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR, SPLIT_CSV, FIX_CSV]:
    assert path.exists(), f"없습니다: {rel(path)}"

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 0번 GPU. GPU가 없으면 CPU

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__, "/ CUDA:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__)
print("W&B        :", wandb.__version__)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU        :", gpu.name, f"/ 메모리 {gpu.total_memory / 1024**3:.1f} GiB")
else:
    print("GPU        : 없음 (CPU로 학습하면 매우 느립니다)")
print()
for name, path in [("원본 데이터", RAW_DIR), ("분할 파일", SPLIT_CSV), ("라벨 수정 목록", FIX_CSV),
                   ("YOLO 데이터", YOLO_DIR), ("학습 결과", RUNS_DIR), ("사전학습 가중치", WEIGHTS_DIR)]:
    print(f"{name:10}: {rel(path)}")

In [ ]:
KOREAN_FONTS = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
installed = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((name for name in KOREAN_FONTS if name in installed), None)
assert FONT is not None, f"한글 글꼴을 찾지 못했습니다. {KOREAN_FONTS} 중 하나를 설치한 뒤 커널을 다시 시작하세요."

plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False  # 한글 글꼴에서 음수 기호가 깨지지 않도록

print("운영체제:", platform.system(), "/ 글꼴:", FONT)

## 1. RT-DETR를 고른 이유

#### 팀 안에서의 위치

| 모델 | 계열 | 예측 방식 |
|---|---|---|
| YOLO26 | 1단계 CNN | 이미지를 한 번 보고 위치와 클래스를 함께 예측한다. |
| Faster R-CNN | 2단계 CNN | 알약이 있을 만한 영역을 먼저 고르고(1단계), 그 영역의 클래스와 bbox를 다듬는다(2단계). |
| RT-DETR | transformer | CNN으로 이미지 특징을 뽑은 뒤 transformer가 정해진 개수의 예측(쿼리)을 한 번에 낸다. 학습할 때 예측과 정답 bbox를 일대일로 짝지어(헝가리안 매칭) 오차를 계산한다. |

- RT-DETR는 미리 정한 기준 박스(앵커)와, 같은 물체에 겹친 예측을 지우는 후처리(NMS)를 쓰지 않는다. 앞의 두 모델과 예측 방식이 달라서, 세 모델을 비교하면 계열에 따른 차이를 볼 수 있다.

#### DETR 계열 중 RT-DETR를 고른 이유

- 데이터가 적은 상황에 맞다. 처음 나온 DETR은 COCO로 500 epoch을 학습할 만큼 수렴이 느리고 데이터가 많이 필요하다. 우리 train은 185장뿐이다. RT-DETR는 COCO로 사전학습된 가중치(`rtdetr-l.pt`)에서 시작하는 실시간 DETR이라 적은 데이터로 미세조정하기에 낫다.
- 같은 조건으로 비교할 수 있다. RT-DETR는 YOLO26과 같은 Ultralytics 라이브러리로 학습한다. 그래서 같은 분할, 같은 YOLO 형식 데이터, 같은 평가 방법을 쓸 수 있고, 점수 차이를 모델 차이로 보기 쉽다.

다른 transformer 후보와 비교

| 후보 | 고르지 않은 이유 |
|---|---|
| RF-DETR | 최신 모델이지만 COCO 형식 데이터가 필요해서 변환을 따로 만들어야 하고, 문서상 입력 크기가 384~640이다. |
| D-FINE | 위치 정밀도를 높인 모델이지만, 학습 반복문과 평가를 직접 짜야 해서 베이스라인으로는 품이 많이 든다. |

#### 한계

- Ultralytics가 제공하는 사전학습 가중치는 큰 모델(L, X)뿐이라 YOLO26의 작은 모델보다 무겁고 느리다.
- 공식 문서에 따르면 GPU에서 RT-DETR는 `deterministic=False`로 학습해야 한다. 그래서 seed가 같아도 결과가 조금씩 달라질 수 있다. 여러 seed로 학습해서 평균으로 비교한다.
- 2023년에 나온 구조라, 이후에 나온 DETR 계열(RF-DETR, D-FINE 등)보다 성능이 낮을 수 있다.

## 2. 데이터 준비

Ultralytics는 YOLO 형식 데이터만 읽는다. 지금 데이터는 알약 하나가 JSON 하나인 COCO 형식이라, 아래 순서로 바꾼다. 방법은 팀의 YOLO26 베이스라인과 같아서, 두 모델이 같은 데이터로 학습한다.

| 단계 | 하는 일 |
|---|---|
| 2-1 | 원본 JSON을 읽어 bbox 표를 만들고, 클래스 번호표(category_id ↔ 0~55)를 만든다. |
| 2-2 | 라벨 수정 목록을 적용한다. |
| 2-3 | 분할 파일을 붙인다. |
| 2-4 | YOLO 형식(사진, txt 라벨, `data.yaml`)으로 바꾼다. |
| 2-5 | 바꾼 라벨을 사진에 다시 그려서 맞는지 본다. |

### 2-1. 라벨 읽기와 클래스 번호표

- `train_annotations`의 JSON을 모두 읽어 bbox 하나가 한 줄인 표를 만든다. bbox는 왼쪽 위 꼭짓점 기준 픽셀 `[x, y, w, h]`다.
- YOLO 형식은 클래스를 0부터 차례로 붙인 번호로 쓴다. 약 번호(category_id)를 작은 순으로 0, 1, 2, …에 대응시킨다. 제출할 때는 같은 표로 약 번호로 되돌린다.

In [ ]:
rows = []
cat_names = {}  # 약 번호 → 약 이름
for jp in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    d = json.loads(jp.read_text(encoding="utf-8-sig"))  # utf-8-sig: 파일 앞에 BOM이 있어도 읽는다
    img = d["images"][0]
    for c in d["categories"]:
        cat_names[c["id"]] = c["name"]
    for a in d["annotations"]:
        rows.append({"file_name": img["file_name"], "img_w": img["width"], "img_h": img["height"],
                     "category_id": a["category_id"], "bbox": a["bbox"]})
ann = pd.DataFrame(rows)

CAT_IDS = sorted(cat_names)
CAT2YOLO = {c: i for i, c in enumerate(CAT_IDS)}  # 약 번호 → 0~55
YOLO2CAT = {i: c for c, i in CAT2YOLO.items()}    # 0~55 → 약 번호

print(f"bbox: {len(ann)}개 / 이미지: {ann['file_name'].nunique()}장 / 클래스: {len(CAT_IDS)}개")
print("번호표 앞부분:", list(CAT2YOLO.items())[:5])
print("같은 (사진, 클래스) bbox가 2개 이상:", int(ann.duplicated(["file_name", "category_id"]).sum()), "건")

### 2-2. 라벨 수정 목록 적용

- 팀 공용 라벨 수정 목록을 적용한다. 한 줄이 bbox 하나이고, 열은 `file_name, category_id, action, bbox, area`다. `bbox`와 `area`는 고친 뒤의 값이다.
- `add`: 빠진 bbox를 추가한다.
- `modify`: 같은 사진, 같은 클래스의 bbox를 찾아 목록의 bbox로 바꾼다.
    - 2-1에서 한 사진에 같은 클래스 bbox가 2개 이상인 경우가 없음을 확인했으므로, (사진, 클래스)로 bbox 하나를 찾을 수 있다.
    - 목록의 bbox가 지금 bbox와 같은 줄은 면적(area) 값만 고친 줄이다. 학습에는 bbox만 쓰고 area는 쓰지 않아서 건너뛴다.
- 목록에 고치기 전 bbox가 없어서, 목록이 같은 원본을 기준으로 만들어졌는지는 고칠 bbox를 모두 찾았는지와 적용 뒤 bbox 수로 확인한다. 적용 뒤 bbox는 771개(원본 763개 + 추가 8개)여야 한다.

In [ ]:
fixes = pd.read_csv(FIX_CSV, encoding="utf-8-sig")
assert {"file_name", "category_id", "action", "bbox"} <= set(fixes.columns), f"라벨 수정 목록의 열이 다릅니다: {list(fixes.columns)}"
img_size = ann.drop_duplicates("file_name").set_index("file_name")[["img_w", "img_h"]]

# 표의 칸에 리스트(bbox)를 직접 바꿔 넣지 않도록, 행 목록으로 바꿔서 고친 뒤 다시 표로 만든다
records = ann.to_dict("records")
row_of = {(r["file_name"], r["category_id"]): i for i, r in enumerate(records)}  # (사진, 클래스) → 행 번호

applied = Counter()
for f in fixes.itertuples():
    key = (f.file_name, int(f.category_id))
    bbox = json.loads(f.bbox)  # "[x, y, w, h]" 글자 → 리스트
    if f.action == "add":
        assert key not in row_of, f"이미 있는 bbox를 추가하려 합니다: {key}"
        w, h = img_size.loc[f.file_name]
        records.append({"file_name": f.file_name, "img_w": w, "img_h": h,
                        "category_id": int(f.category_id), "bbox": bbox})
        row_of[key] = len(records) - 1
        applied["add"] += 1
    elif f.action == "modify":
        assert key in row_of, f"고칠 bbox를 못 찾았습니다: {key}"
        r = records[row_of[key]]
        if r["bbox"] == bbox:
            applied["modify (bbox 같음, area만 수정 → 건너뜀)"] += 1
        else:
            r["bbox"] = bbox
            applied["modify (bbox 수정)"] += 1
    else:
        raise ValueError(f"알 수 없는 action: {f.action}")
ann = pd.DataFrame(records)

for k, v in applied.items():
    print(f"{k}: {v}줄")
print(f"목록 전체: {len(fixes)}줄")
print(f"bbox: {len(ann)}개 / 이미지: {ann['file_name'].nunique()}장")
assert sum(applied.values()) == len(fixes), "적용하지 않은 줄이 있습니다."
assert len(ann) == 771, f"bbox가 771개가 아닙니다: {len(ann)}"

### 2-3. 분할 붙이기

- `split.csv`는 사진마다 `train` 또는 `val`이 적힌 표다(열: `file_name`, `combo`, `split`).
- 다음을 확인한다.
    - 사진 232장이 모두 들어 있고, train 185장, val 47장인지
    - 같은 조합의 사진이 train과 val에 나뉘지 않았는지(조합 단위 분할)
    - 분할 정보가 빠진 bbox가 없는지

In [ ]:
split = pd.read_csv(SPLIT_CSV)
n_split = {k: int(v) for k, v in split["split"].value_counts().items()}
print("사진 수:", n_split)
combo_splits = split.groupby("combo")["split"].nunique()
print(f"train과 val에 나뉜 조합: {int((combo_splits > 1).sum())}개 / 조합 {len(combo_splits)}개")

ann = ann.merge(split[["file_name", "split"]], on="file_name", how="left")
print("분할 정보가 빠진 bbox:", int(ann["split"].isna().sum()))
display(ann.groupby("split").agg(사진=("file_name", "nunique"), bbox=("file_name", "size"), 클래스=("category_id", "nunique")))

assert len(split) == 232 and n_split == {"train": 185, "val": 47}, "분할이 1차 EDA(seed 410)와 다릅니다."
assert (combo_splits == 1).all(), "train과 val에 나뉜 조합이 있습니다."
assert ann["split"].notna().all(), "분할 정보가 빠진 bbox가 있습니다."

### 2-4. YOLO 형식으로 변환

- 사진 한 장에 txt 한 개를 만든다. 한 줄이 bbox 하나이고, `클래스번호 중심x 중심y 너비 높이`를 이미지 크기로 나눈 0~1 값으로 적는다.
- 사진은 `images/train`, `images/val`로 복사한다. Ultralytics는 사진 경로의 `images`를 `labels`로 바꿔 라벨을 찾기 때문이다.
- 이미지 밖으로 나간 bbox는 경계에서 자르고, 완전히 밖이면 버린다. 라벨 수정으로 이미지 밖 bbox(3-2의 18357)는 고쳐져서, 버리는 bbox는 0개여야 한다.
- `data.yaml`에는 폴더 경로와 클래스 이름을 적는다. 클래스 이름은 한글이 깨지지 않게 약 번호를 글자로 쓴다.
- 다시 실행할 때 예전 파일이 남지 않게 `data/processed/yolo`를 지우고 새로 만든다. 팀의 YOLO26 노트북과 같은 폴더, 같은 방법이라 만들어지는 내용도 같다.

In [ ]:
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for s in ["train", "val"]:
    (YOLO_DIR / "images" / s).mkdir(parents=True)
    (YOLO_DIR / "labels" / s).mkdir(parents=True)

dropped = 0
for name, g in ann.groupby("file_name"):
    s = g["split"].iloc[0]
    shutil.copy(TRAIN_IMG_DIR / name, YOLO_DIR / "images" / s / name)

    lines = []
    for r in g.itertuples():
        x, y, w, h = r.bbox
        W, H = r.img_w, r.img_h
        x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)  # 이미지 밖으로 나간 부분은 자른다
        if x2 <= x1 or y2 <= y1:
            dropped += 1  # 완전히 밖이면 버린다
            continue
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        lines.append(f"{CAT2YOLO[r.category_id]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    (YOLO_DIR / "labels" / s / (Path(name).stem + ".txt")).write_text("\n".join(lines))

data_cfg = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: str(c) for i, c in YOLO2CAT.items()},
}
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

for s in ["train", "val"]:
    n_img = len(list((YOLO_DIR / "images" / s).glob("*.png")))
    n_lbl = len(list((YOLO_DIR / "labels" / s).glob("*.txt")))
    print(f"{s}: 사진 {n_img}장 / 라벨 {n_lbl}개")
saved = sum(len(p.read_text().splitlines()) for p in (YOLO_DIR / "labels").rglob("*.txt"))
print(f"저장한 bbox: {saved}개 / 버린 bbox: {dropped}개 / 표의 bbox: {len(ann)}개")
print("data.yaml:", rel(DATA_YAML))
assert saved + dropped == len(ann), "저장한 bbox 수가 표와 맞지 않습니다."

### 2-5. 변환 확인

- YOLO txt를 다시 읽어 픽셀 좌표로 되돌린 뒤 사진에 그린다. bbox가 알약을 감싸면 변환 공식이 맞다.
- train과 val에서 2장씩 무작위로 고른다. seed를 고정해서 다시 실행해도 같은 사진이 나온다.

In [ ]:
SAMPLE_SEED = 0
sample_rng = np.random.default_rng(SAMPLE_SEED)
samples = []
for s in ["train", "val"]:
    paths = sorted((YOLO_DIR / "images" / s).glob("*.png"))
    samples += [(s, paths[i]) for i in sorted(sample_rng.choice(len(paths), 2, replace=False))]

fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 5.5), layout="constrained")
for ax, (s, img_path) in zip(axes, samples):
    with Image.open(img_path) as im:
        W, H = im.size
        ax.imshow(im)
    for line in (YOLO_DIR / "labels" / s / (img_path.stem + ".txt")).read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        bw, bh = float(bw) * W, float(bh) * H                     # 0~1 비율 → 픽셀
        x1, y1 = float(cx) * W - bw / 2, float(cy) * H - bh / 2   # 중심 → 왼쪽 위
        ax.add_patch(Rectangle((x1, y1), bw, bh, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1, y1 - 10, YOLO2CAT[int(c)], color="white", fontsize=8,
                bbox=dict(facecolor="green", edgecolor="none", pad=1))
    ax.set_title(f"{s}: {img_path.stem.split('_')[0]}", fontsize=8)
    ax.axis("off")
fig.suptitle("YOLO 라벨을 다시 그린 사진 (초록 사각형: bbox, 숫자: 약 번호)", fontsize=11)
plt.show()

### 2번 결과

| 단계 | 결과 |
|---|---|
| 2-1 라벨 읽기 | bbox 763개, 사진 232장, 클래스 56개. 한 사진에 같은 클래스 bbox가 2개 이상인 경우는 없다. |
| 2-2 라벨 수정 | add 8줄로 bbox를 추가하고, modify 13줄 중 bbox가 바뀐 9줄을 고쳤다. bbox가 같은 4줄(area만 수정)은 건너뛰었다. bbox는 771개가 되었다. |
| 2-3 분할 | train 185장(bbox 612개, 클래스 56개), val 47장(bbox 159개, 클래스 30개). train과 val에 나뉜 조합, 분할 정보가 빠진 bbox는 없다. |
| 2-4 YOLO 변환 | txt 라벨 232개(train 185, val 47)에 bbox 771개를 저장했고, 버린 bbox는 없다. |
| 2-5 변환 확인 | train, val에서 2장씩 YOLO 라벨을 다시 그렸을 때 4장 모두 bbox가 알약을 감쌌다. 변환 공식이 맞다. |

- val에는 56종 중 30종만 있다. Ultralytics는 val 정답에 있는 클래스만 평균하므로, 뒤에 나오는 val 점수는 이 30종에 대한 점수다. train에만 있는 26종은 val로 평가되지 않는다.

## 3. 학습 설정

### 3-1. 학습 인자

- 팀의 YOLO26 본 학습(Colab)과 같은 값을 쓴다. 다른 것은 모델, 그리고 아래 표에 이유를 적은 `workers`, `deterministic`뿐이다.
- 표에 없는 인자(학습률, 증강 세기 등)는 Ultralytics 기본값을 쓴다. 학습이 끝나면 실제로 쓴 값 전체가 run 폴더의 `args.yaml`에 저장된다.

| 인자 | 값 | 뜻 | 이 값을 쓰는 이유 |
|---|---|---|---|
| 모델 | `rtdetr-l.pt` | COCO로 사전학습된 RT-DETR-L 가중치 | Ultralytics가 주는 RT-DETR 가중치(L, X) 중 작은 쪽 |
| epochs | 100 | train 사진 전체를 몇 번 반복해서 볼지 | 팀 설정 |
| patience | 30 | val 성능(mAP50-95)이 30 epoch 동안 나아지지 않으면 일찍 멈춘다 | 팀 설정 |
| imgsz | 640 | 사진을 640×640 크기로 맞춰 넣는다. RT-DETR는 가로세로 비율을 유지하지 않고 늘이거나 줄여서 맞춘다(학습, val, 예측 모두 같다) | 팀 설정. 원본 976×1280이 640×640이 된다(976 쪽 0.66배, 1280 쪽 0.5배). 1차 EDA 5-4는 비율을 유지해 둘 다 0.5배로 줄인 경우를 봤고 각인이 보였다. 여기서는 976 쪽이 그보다 덜 줄어든다 |
| batch | 16 | 한 번에 GPU에 올리는 사진 수 | 팀 설정. GPU 메모리가 모자라면 8로 줄인다 |
| optimizer | auto | 가중치를 고치는 방법과 학습률을 Ultralytics가 고른다 | 팀 설정. 아래에 설명 |
| 증강 | 기본값 | 학습 사진을 무작위로 바꿔(좌우 뒤집기, 색, 크기, 모자이크 등) 다양하게 만든다 | 팀 설정 |
| seed | 0, 1, 2 | 학습의 무작위 요소(새로 붙는 층의 초깃값, 사진 순서, 증강)를 정하는 값 | 팀 설정. seed마다 점수가 얼마나 흔들리는지 본다 |
| workers | 0 | 사진을 읽는 보조 프로세스 수 | Windows 주피터에서는 0이 아니면 오류가 날 수 있다. 속도에만 영향이 있다(팀원은 Colab에서 2) |
| deterministic | False | 같은 seed면 같은 결과가 나오도록 강제할지 | RT-DETR가 쓰는 연산(`grid_sample`)이 이 옵션을 지원하지 않는다 |

optimizer=auto가 고르는 값 (Ultralytics 8.4.183 기준)
- 전체 반복 수가 10,000 이하이면 AdamW를 고른다. 반복 수는 `올림(train 사진 수 / max(batch, 64)) × epochs` = 올림(185 / 64) × 100 = 300이라 AdamW가 된다.
- 학습률도 `0.002 × 5 / (4 + 클래스 수)` = 0.002 × 5 / 60 ≈ 0.000167로 정한다. 이때 `lr0`(기본 0.01)는 쓰지 않는다.
- 64는 `nbs`(기준 batch)다. batch 16이면 4번 모아서 가중치를 한 번 고친다(기울기 누적). batch를 8로 줄여도 8번 모아서 고치므로, 한 번에 반영되는 사진 수는 64장으로 같다.

AMP
- AMP(자동 혼합 정밀도)는 기본값대로 켠다. 일부 계산을 16비트로 해서 메모리를 덜 쓰고 빨라진다.
- Ultralytics RT-DETR 학습 코드 주석에 AMP에서 손실이 NaN이 될 수 있다고 적혀 있어서, 확인 실행에서 손실이 숫자로 나오는지 본다.

짧은 확인 실행
- `SMOKE_TEST = True`이면 seed 0으로 1 epoch만 돈다. 본 학습(seed당 최대 100 epoch) 전에 batch 16이 GPU 메모리에 들어가는지, W&B에 기록되는지 먼저 보기 위해서다.
- 확인 실행은 이름 끝에 `_smoke`를 붙여 본 학습 결과와 섞이지 않게 한다.
- 문제가 없으면 `False`로 바꾸고 이 셀부터 다시 실행한다.

In [ ]:
SMOKE_TEST = False  # True: seed 0으로 1 epoch만 도는 확인 실행 / False: 본 학습

MODEL_NAME = "rtdetr-l"
WEIGHTS = WEIGHTS_DIR / f"{MODEL_NAME}.pt"  # COCO 사전학습 가중치. 없으면 처음 학습할 때 내려받는다
EPOCHS = 100
PATIENCE = 30
IMGSZ = 640
BATCH = 16
SEEDS = [0, 1, 2]


def make_run_name(seed):
    """run 폴더 이름. 팀원 실행 이름(yolo26s_e100_s0_fix)과 형식을 맞춘다. _fix는 라벨 수정 반영이라는 뜻"""
    name = f"{MODEL_NAME}_e{EPOCHS}_s{seed}_fix"
    return f"{name}_smoke" if SMOKE_TEST else name


def train_args(seed):
    """model.train()에 넘길 인자. 여기 없는 인자는 Ultralytics 기본값을 쓴다."""
    return dict(
        data=str(DATA_YAML),
        epochs=1 if SMOKE_TEST else EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        seed=seed,
        workers=0,            # Windows 주피터에서 사진 읽기 오류를 피하려고 0
        deterministic=False,  # RT-DETR의 grid_sample이 deterministic을 지원하지 않는다
        project=str(RUNS_DIR),
        name=make_run_name(seed),
        exist_ok=True,        # 같은 이름의 run 폴더가 있으면 덮어쓴다
    )


print("모드:", "확인 실행 (seed 0, 1 epoch)" if SMOKE_TEST else f"본 학습 (seed {SEEDS}, 최대 {EPOCHS} epoch)")
print("사전학습 가중치:", rel(WEIGHTS), "(있음)" if WEIGHTS.exists() else "(없음, 처음 학습할 때 내려받음)")
print()
for k, v in train_args(0).items():
    if k not in ("data", "project"):  # 경로는 0번에서 출력했다
        print(f"{k:14}: {v}")

### 3-2. W&B 기록

- 팀 W&B(`codeit-prj1-team2` / `pill-object-detection`)에 seed마다 실행 하나로 기록한다.
    - 실행 이름: `jw_` + run 폴더 이름. 예: `jw_rtdetr-l_e100_s0_fix`
    - 태그: `rtdetr`, `label-fixed`, `baseline`(확인 실행은 `smoke`)
    - config: 모델, 분할 seed(410), 데이터 설명, 3-1의 학습 인자
- Ultralytics가 자동으로 기록하는 것: epoch마다 손실, 학습률, `metrics/mAP50(B)`, `metrics/mAP50-95(B)` 등. 팀원 YOLO26 실행과 같은 이름이라 바로 비교할 수 있다. 끝에 그래프와 best.pt도 올린다.
- 대회 지표(mAP75-95)는 자동 기록에 없어서 콜백으로 넣는다. 콜백은 학습 중 정해진 시점에 Ultralytics가 불러 주는 함수다.
    - `on_fit_epoch_end`(val이 끝날 때마다): IoU 0.75~0.95 다섯 단계 AP의 평균을 `val/map_75_95`로 기록한다.
    - `on_train_end`(학습이 끝날 때): best.pt로 다시 잰 값을 요약값 `best_val/map_75_95`로 넣는다.
    - 두 이름은 팀원 Faster R-CNN 실행과 같다. 다만 여기 best.pt는 mAP50-95로 고른 것이고, Faster R-CNN 값은 mAP75-95가 가장 높은 epoch의 값이다. 계산 라이브러리도 다르다(Ultralytics, torchmetrics). 그래서 모델끼리는 Kaggle 점수로 비교한다.
- W&B 실행은 학습 전에 직접 시작한다(`wandb.init`). Ultralytics에 맡기면 `project`에 넣은 결과 폴더 경로가 W&B 프로젝트 이름이 되어, 사람마다 다른 프로젝트에 흩어지고 내 컴퓨터 경로도 W&B에 남는다.
- W&B 로그인은 터미널에서 `wandb login`으로 한 번 해 둔다. 노트북에는 키를 적지 않는다.

In [ ]:
WANDB_ENTITY = "codeit-prj1-team2"      # W&B 팀
WANDB_PROJECT = "pill-object-detection"
WANDB_PREFIX = "jw"                      # 실행 이름 앞에 붙이는 이니셜 (팀원: cy_)
SPLIT_SEED = 410                         # split.csv를 만든 분할 seed (기록용)


def map75_95(metrics):
    """대회 지표 근사: IoU 0.75, 0.80, 0.85, 0.90, 0.95에서 AP를 구해 평균한다.
    metrics.box.all_ap는 (val 정답에 있는 클래스 수, IoU 0.50~0.95 10단계) 표라 뒤 5칸만 쓴다."""
    ap = metrics.box.all_ap
    return float(ap[:, 5:].mean()) if len(ap) else float("nan")


def log_map75_95(trainer):
    """콜백: val이 끝날 때마다 불린다. Ultralytics의 W&B 기록과 같은 단계(epoch + 1)에 넣는다."""
    if wandb.run is not None:
        wandb.run.log({"val/map_75_95": map75_95(trainer.validator.metrics)}, step=trainer.epoch + 1)


def save_best_map75_95(trainer):
    """콜백: 학습이 끝날 때 불린다. 이때 validator에는 best.pt로 다시 잰 결과가 들어 있다."""
    if wandb.run is not None:
        wandb.run.summary["best_val/map_75_95"] = map75_95(trainer.validator.metrics)


def wandb_config(args):
    """W&B config. 팀원 YOLO26 실행과 같은 구조로 넣는다. 경로는 넣지 않는다."""
    return {
        "model": MODEL_NAME,
        "split_seed": SPLIT_SEED,
        "data": "라벨 수정 반영(bbox 771) + split.csv(seed 410)",
        **{k: v for k, v in args.items() if k not in ("data", "project", "name")},
    }


print("W&B:", f"{WANDB_ENTITY}/{WANDB_PROJECT}", "/ 실행 이름 예:", f"{WANDB_PREFIX}_{make_run_name(0)}")

## 4. 학습

### 4-1. GPU 메모리 확인

- 학습 전에 GPU에 남은 메모리를 본다. 다른 프로그램(브라우저, 게임 등)이 GPU 메모리를 쓰고 있으면 학습에 쓸 수 있는 양이 줄어든다.
- 학습 중 val은 batch의 2배(32장)로 돈다.
- Windows에서는 GPU 메모리가 모자라도 오류 없이 시스템 메모리를 빌려 쓰면서 학습이 크게 느려질 수 있다. 그래서 확인 실행에서 epoch 시간도 같이 본다.

In [ ]:
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU 메모리: 남은 {free / 1024**3:.1f} GiB / 전체 {total / 1024**3:.1f} GiB")
else:
    print("GPU가 없습니다.")

### 4-2. 학습 함수

seed 하나를 학습하는 함수 `train_one`을 만든다. 순서는 다음과 같다.

1. 사전학습 가중치로 모델을 만들고, 3-2의 콜백 두 개를 등록한다.
2. Ultralytics의 W&B 기록을 켜고, W&B 실행을 직접 시작한다.
3. 학습한다. 학습이 끝나면 Ultralytics가 best.pt로 val을 한 번 더 하고 W&B 실행을 닫는다.
4. Ultralytics의 W&B 설정을 원래 값으로 돌린다. 이 설정은 내 컴퓨터의 Ultralytics 설정 파일에 저장되어 다른 노트북에도 남기 때문이다. 학습 전에 오류가 나서 W&B 실행이 열려 있으면 여기서 닫는다.

끝나면 best.pt의 val 지표(mAP50, mAP50-95, mAP75-95)를 출력한다. 자세한 분석은 6번에서 한다.

In [ ]:
def train_one(seed):
    args = train_args(seed)

    model = RTDETR(str(WEIGHTS))
    model.add_callback("on_fit_epoch_end", log_map75_95)
    model.add_callback("on_train_end", save_best_map75_95)

    wandb_before = settings["wandb"]
    settings.update({"wandb": True})
    wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=f"{WANDB_PREFIX}_{args['name']}",
        tags=["rtdetr", "label-fixed", "smoke" if SMOKE_TEST else "baseline"],
        config=wandb_config(args),
    )
    try:
        metrics = model.train(**args)  # best.pt로 다시 잰 val 결과가 돌아온다
    finally:
        settings.update({"wandb": wandb_before})
        if wandb.run is not None:
            wandb.finish()

    print(f"\n[{args['name']}] best.pt val: mAP50 {metrics.box.map50:.4f} / "
          f"mAP50-95 {metrics.box.map:.4f} / mAP75-95 {map75_95(metrics):.4f}")
    return metrics

### 4-3. seed 0 학습

- 확인 실행(`SMOKE_TEST = True`)도 이 셀로 한다.
- 확인 실행에서 볼 것
    - 오류 없이 끝나는지. `CUDA out of memory`가 나면 3-1에서 batch를 8로 줄인다.
    - 출력 앞부분에 `optimizer: AdamW(lr=0.000167, ...)`가 나오는지
    - 진행 표시줄의 손실 값이 `nan`이 아닌지
    - 진행 표시줄의 `GPU_mem`(PyTorch가 잡은 GPU 메모리)과 epoch 시간. `GPU_mem`이 12G에 가깝거나 넘으면 시스템 메모리를 빌려 쓰고 있을 수 있다.
    - 출력에 나온 W&B 링크에서 팀 프로젝트에 실행이 생기고, `val/map_75_95`와 요약값 `best_val/map_75_95`가 들어갔는지
- 1 epoch로는 거의 학습되지 않으므로 확인 실행의 점수는 보지 않는다.

In [ ]:
metrics_s0 = train_one(0)

### 4-4. seed 1 학습

seed 0 본 학습이 끝난 뒤 실행한다. 확인 실행에서는 돌리지 않는다.

In [ ]:
assert not SMOKE_TEST, "확인 실행은 seed 0만 돈다. 3-1에서 SMOKE_TEST = False로 바꾼 뒤 실행하세요."
metrics_s1 = train_one(1)

### 4-5. seed 2 학습

seed 1 학습이 끝난 뒤 실행한다.

In [ ]:
assert not SMOKE_TEST, "확인 실행은 seed 0만 돈다. 3-1에서 SMOKE_TEST = False로 바꾼 뒤 실행하세요."
metrics_s2 = train_one(2)

### 4-6. 학습 곡선

- 세 seed의 `results.csv`(Ultralytics가 epoch마다 저장한 기록)를 읽어 epoch별 변화를 그린다. 학습은 다시 하지 않는다. 커널을 다시 시작했다면 0번 두 셀과 3-1을 먼저 실행한다.
- 그림은 세 칸이고, seed마다 색이 다르다. 손실 칸은 train이 실선, val이 점선이다. mAP 칸은 mAP50-95가 실선, mAP50이 점선이다. 학습률은 세 seed가 같아서 한 줄만 그린다.
- 아래 표는 seed별로 mAP50-95가 가장 높았던 epoch, val 손실이 가장 낮았던 epoch, 마지막 epoch의 train과 val 손실이다.

그래프 읽는 법

| 그래프 | 무엇인가 | 읽는 법 |
|---|---|---|
| 손실 | 예측이 정답과 얼마나 다른지를 나타낸 숫자로, 낮을수록 좋다. RT-DETR는 giou(박스가 겹친 정도의 오차), cls(약 종류를 틀린 정도), l1(박스 좌표 차이)를 더한다. train 손실은 학습하는 사진(증강됨)으로, val 손실은 학습에 쓰지 않은 val 사진(증강 없음)으로 잰다 | 둘 다 내려가면 배우는 중이다. train은 내려가는데 val이 다시 오르면 과적합(학습 사진에만 맞춰짐) 신호다. 둘 다 높은 채로 평평하면 과소적합(덜 배움) 신호다. 둘 다 낮은 채로 평평하면 수렴(더 배울 것이 거의 없음)이다 |
| val mAP | val 사진에서 알약을 찾고, 약 종류와 박스를 맞혔는지 매긴 점수로, 1에 가까울수록 좋다. mAP50은 박스가 절반 이상 겹치면 맞음으로 치는 느슨한 기준이고, mAP50-95는 겹침 기준을 0.50~0.95로 바꿔 가며 평균한 엄격한 기준이다 | 오르면 나아지는 중, 평평하면 수렴이다. 오르다 떨어지면 과적합 신호다. 점선(mAP50)과 실선(mAP50-95)이 가까울수록 박스 위치가 정확하다 |
| 학습률 | 가중치를 한 번 고칠 때의 보폭이다. 성능이 아니라 학습 방식을 보여 준다 | 처음 3 epoch(warmup)은 새로 붙인 층이 흔들리지 않게 작은 값에서 올리고, 그 뒤로는 직선으로 줄인다. 초반에는 크게 배우고 후반에는 작게 다듬는다. 다른 두 그래프의 모양이 바뀌는 이유를 찾을 때 같이 본다 |

- 손실 칸의 세로축은 로그 눈금이다. 값이 절반이 되는 거리가 같은 간격으로 그려져서, 초반의 큰 감소와 후반의 작은 변화를 한 그림에서 볼 수 있다.
- 91 epoch의 세로선은 모자이크 증강을 끈 시점이다. 모자이크는 학습 사진 4장을 2×2로 이어 붙여 한 장으로 만드는 증강으로, Ultralytics 기본값이라 1차 실험에서도 켜져 있었다. 모자이크 사진은 경계에서 알약이 잘리는 등 실제 사진과 모양이 달라서, 기본값 `close_mosaic=10`에 따라 마지막 10 epoch은 평범한 사진으로 학습해 실제 사진에 맞춰 마무리한다.
- 대회 지표(mAP75-95)의 epoch별 값은 `results.csv`에 없고 W&B에만 있다.
- 읽을 때 주의할 점: train 손실은 증강(모자이크, 색 변화 등)을 거친 사진으로 재고, val 손실은 증강 없는 사진으로 잰다. 그래서 train 손실이 val 손실보다 높게 나올 수 있고, 모자이크를 끄는 91 epoch부터 train 손실이 갑자기 내려갈 수 있다. 이것만 보고 과적합이나 과소적합이라고 하지 않는다.

In [ ]:
SEED_COLORS = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a"}  # seed별 고정 색 (색약 구분 확인)
LOSS_PARTS = ["giou_loss", "cls_loss", "l1_loss"]
MOSAIC_OFF = EPOCHS - 10 + 1  # close_mosaic 10 → 91 epoch부터 모자이크를 끈다

curves = {}
for seed in SEEDS:
    df = pd.read_csv(RUNS_DIR / make_run_name(seed) / "results.csv")
    assert df["epoch"].is_unique, f"seed {seed}: results.csv에 같은 epoch이 두 번 있습니다. 같은 이름으로 다시 학습한 기록이 섞였을 수 있습니다."
    df["train_loss"] = df[[f"train/{p}" for p in LOSS_PARTS]].sum(axis=1)
    df["val_loss"] = df[[f"val/{p}" for p in LOSS_PARTS]].sum(axis=1)
    curves[seed] = df
assert all(np.allclose(curves[s]["lr/pg0"], curves[SEEDS[0]]["lr/pg0"]) for s in SEEDS), "seed마다 학습률이 다릅니다."

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), layout="constrained")
for seed, df in curves.items():
    c = SEED_COLORS[seed]
    axes[0].plot(df["epoch"], df["train_loss"], color=c, lw=1.5, label=f"seed {seed} train")
    axes[0].plot(df["epoch"], df["val_loss"], color=c, lw=1.5, ls="--", label=f"seed {seed} val")
    axes[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], color=c, lw=1.5, label=f"seed {seed} mAP50-95")
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], color=c, lw=1.2, ls=":", label=f"seed {seed} mAP50")
first = curves[SEEDS[0]]
axes[2].plot(first["epoch"], first["lr/pg0"], color="#2a78d6", lw=1.5)

axes[0].set_yscale("log")
LOSS_TICKS = [0.2, 0.3, 0.5, 1, 2, 3]  # 로그 눈금을 일반 숫자로 적는다 (기본 지수 표기는 한글 글꼴에서 마이너스 기호가 깨진다)
axes[0].set_yticks(LOSS_TICKS, labels=[f"{t:g}" for t in LOSS_TICKS])
axes[0].tick_params(axis="y", which="minor", labelleft=False)
axes[1].set_ylim(0, 1.02)
panels = [("손실 합 (giou + cls + l1), 로그 눈금", "손실"), ("val mAP", "mAP"), ("학습률 (lr/pg0, 세 seed 같음)", "학습률")]
for ax, (title, ylabel) in zip(axes, panels):
    ax.axvline(MOSAIC_OFF, color="gray", lw=1, ls="-.")
    ax.text(MOSAIC_OFF - 1, 0.55, "모자이크 끔", transform=ax.get_xaxis_transform(), ha="right", va="center", fontsize=8, color="dimgray")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, ncol=2)
axes[1].legend(fontsize=8, ncol=2, loc="lower right")
fig.suptitle("seed별 학습 곡선 (손실: 실선 train, 점선 val / mAP: 실선 mAP50-95, 점선 mAP50)", fontsize=11)
plt.show()

rows = []
for seed, df in curves.items():
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    low = df.loc[df["val_loss"].idxmin()]
    last = df.iloc[-1]
    rows.append({
        "seed": seed,
        "epoch 수": len(df),
        "mAP50-95 최고 epoch": int(best["epoch"]),
        "mAP50-95 최고값": round(float(best["metrics/mAP50-95(B)"]), 4),
        "val 손실 최저 epoch": int(low["epoch"]),
        "val 손실 최저값": round(float(low["val_loss"]), 4),
        "마지막 train 손실": round(float(last["train_loss"]), 4),
        "마지막 val 손실": round(float(last["val_loss"]), 4),
    })
display(pd.DataFrame(rows).set_index("seed"))

### 4-7. train 점수와 val 점수 비교

- 왜 하는가: 4-6의 train 손실은 증강된 사진으로 잰 값이라 val과 바로 비교할 수 없었다. 여기서는 best.pt로 train 사진을 증강 없이, val과 같은 방법으로 평가해서 두 점수를 나란히 본다. train 점수가 val보다 훨씬 높으면 학습 사진에만 맞춰진 것(과적합)이고, 비슷하면 학습한 것이 val에도 통한다는 뜻이다.
- 방법
    - 세 seed의 best.pt로 `model.val()`을 train과 val에 각각 실행한다. 입력 크기 640, 증강 없음, 신뢰도 기준 0.001(Ultralytics val 기본값)로 조건이 같다. 학습은 다시 하지 않는다.
    - val에는 30종만 있어서, train 점수는 전체 56종과 "val과 같은 30종"으로 나눠 낸다. 같은 30종끼리 비교해야 train과 val의 차이를 공정하게 볼 수 있다.
    - 지표는 mAP50, mAP50-95, mAP75-95(대회 지표 근사)다. 클래스마다 IoU 단계별 AP(`all_ap`)를 골라 평균한다.
    - 평가 결과 폴더는 각 run 폴더 아래 `eval_train`, `eval_val`로 생긴다. 그래프는 저장하지 않는다.
- 실행: 0번 두 셀과 3-1을 실행한 뒤 이 셀을 실행한다. GPU로 몇 분 걸린다.
- 해석 기준(결과를 보기 전에 정해 둔다): 같은 30종에서 train과 val의 mAP75-95 차이가 seed끼리의 val 점수 차이(4번 결과에서 최대 약 0.007)와 비슷하면 우연 범위로, 그보다 훨씬 크면 과적합 신호로 본다.

In [ ]:
IOU_COLS = {"mAP50": slice(0, 1), "mAP50-95": slice(0, 10), "mAP75-95": slice(5, 10)}  # all_ap의 열: IoU 0.50, 0.55, ..., 0.95


def ap_means(metrics, classes=None):
    """all_ap(클래스 × IoU 10단계)에서 고른 클래스만 남겨 지표별 평균을 낸다. classes=None이면 정답에 있는 클래스 전체."""
    ap, idx = metrics.box.all_ap, list(metrics.box.ap_class_index)
    rows = [i for i, c in enumerate(idx) if classes is None or c in classes]
    return {name: round(float(ap[rows, cols].mean()), 4) for name, cols in IOU_COLS.items()}, len(rows)


score_rows = []
for seed in SEEDS:
    run_dir = RUNS_DIR / make_run_name(seed)
    model = RTDETR(str(run_dir / "weights" / "best.pt"))
    res = {split: model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                            plots=False, verbose=False, project=str(run_dir), name=f"eval_{split}", exist_ok=True)
           for split in ["train", "val"]}
    val_classes = set(res["val"].box.ap_class_index)  # val 정답에 있는 30종
    for label, metrics, classes in [("train 전체", res["train"], None),
                                    ("train (val과 같은 종)", res["train"], val_classes),
                                    ("val", res["val"], None)]:
        means, n_cls = ap_means(metrics, classes)
        score_rows.append({"seed": seed, "데이터": label, "클래스 수": n_cls, **means})

scores = pd.DataFrame(score_rows)
display(scores.set_index(["seed", "데이터"]))

gap = (scores[scores["데이터"] == "train (val과 같은 종)"].set_index("seed")[["mAP50-95", "mAP75-95"]]
       - scores[scores["데이터"] == "val"].set_index("seed")[["mAP50-95", "mAP75-95"]]).round(4)
print("같은 종에서 train 점수 − val 점수 (양수면 train이 높다)")
display(gap)

### 4번 결과

- 세 seed 모두 100 epoch을 끝까지 돌았다. patience 30에 걸려 일찍 멈춘 seed는 없다.
- 3-1에서 계산한 대로 optimizer는 AdamW(lr=0.000167)로 정해졌다. AMP 확인을 통과했고, 손실이 NaN으로 나온 epoch은 없다.
- 사전학습 가중치에서 941개 중 926개를 가져왔다. 못 가져온 15개는 클래스 수(COCO 80종 → 56종)에 따라 크기가 달라지는 층(클래스 점수를 내는 층 등)이라 새로 학습한다.

| seed | mAP50 | mAP50-95 | mAP75-95 | 학습 시간 |
|---|---|---|---|---|
| 0 | 0.995 | 0.987 | 0.980 | 1.07시간 |
| 1 | 0.993 | 0.985 | 0.978 | 1.55시간 |
| 2 | 0.980 | 0.976 | 0.973 | 2.76시간 |
| 평균 | 0.989 | 0.983 | 0.977 | |

- 점수는 학습이 끝난 뒤 best.pt(mAP50-95가 가장 높은 epoch의 가중치)로 val 47장을 다시 잰 값이다.
- seed에 따른 mAP75-95 차이는 0.01보다 작다(0.973~0.980).
- 학습 시간은 seed마다 크게 달랐다. 학습 중 `GPU_mem`이 12.8G로 전용 GPU 메모리(12GB)를 넘어 시스템 메모리를 같이 썼고, seed 2는 중간에 batch 하나에 10초 넘게 걸린 구간이 있었다. 학습 시간은 점수에 영향을 주지 않는다.

학습 곡선 (4-6)

| seed | mAP50-95 최고 epoch (값) | val 손실 최저 epoch (값) | 마지막 train 손실 | 마지막 val 손실 |
|---|---|---|---|---|
| 0 | 92 (0.9883) | 84 (0.3487) | 0.2182 | 0.3754 |
| 1 | 75 (0.9865) | 79 (0.3096) | 0.2314 | 0.3297 |
| 2 | 99 (0.9764) | 100 (0.3804) | 0.2255 | 0.3804 |

- 학습 중에 잰 mAP50-95 최고값(0.9883 / 0.9865 / 0.9764)은 4-7에서 같은 best.pt로 다시 잰 val 값과 같다. 위 첫 표의 값(0.987 / 0.985 / 0.976)은 학습이 끝난 직후 Ultralytics가 잰 값인데, seed 0, 1에서 소수 셋째 자리가 조금 다르다. 이유는 확인하지 못했다.
- 그림에서 보이는 것
    - 처음부터 약 40 epoch까지 train과 val 손실이 함께 떨어지고, 40~90 epoch에는 둘이 비슷한 높이(약 0.35~0.45)에서 천천히 내려간다.
    - val mAP는 20 epoch 무렵 0.9를 넘고, 40 epoch부터 0.97~0.98 근처에서 거의 평평하다. 최고값이 나온 epoch은 75~99로 늦지만, 40 epoch 이후 오른 폭은 작다.
    - 91 epoch(모자이크를 끈 시점)에 train 손실만 약 0.35에서 0.22로 떨어지고, val 손실은 거의 그대로다. 학습 사진이 모자이크에서 평범한 사진으로 바뀐 영향으로 보인다.
    - 세 seed의 곡선이 거의 겹친다. 학습 과정이 seed에 따라 크게 달라지지 않았다.
- 수렴 시점: 약 40 epoch 무렵으로 보인다. 그 뒤로는 점수가 거의 오르지 않아서, 2차 실험에서 epoch을 줄여도 val 성능은 크게 줄지 않을 가능성이 크다. 다만 데이터가 바뀌면 수렴 시점도 달라진다.

train과 val 점수 비교 (4-7)

| seed | train mAP75-95 (56종) | train mAP75-95 (val과 같은 30종) | val mAP75-95 (30종) | 차이 (train 30종 − val) |
|---|---|---|---|---|
| 0 | 0.9897 | 0.9931 | 0.9816 | 0.0115 |
| 1 | 0.9825 | 0.9873 | 0.9803 | 0.0070 |
| 2 | 0.9907 | 0.9929 | 0.9731 | 0.0198 |

- 같은 30종의 mAP50-95 차이는 0.0058 / 0.0046 / 0.0175다.
- train 점수는 세 seed 모두 0.98~0.99로, 계산상 최고값(0.995)에 가깝다. train에만 있는 26종도 train 사진에서는 0.977~0.988이다(56종 평균과 30종 평균에서 계산).
- 같은 30종에서 train이 val보다 0.007~0.020 높다. 4-7에 미리 정한 기준(seed끼리의 val 차이, 최대 약 0.007)과 비교하면, seed 1은 기준과 같고 seed 0은 약 1.6배, seed 2는 약 2.8배다.
- seed 2의 차이는 대부분 30308 한 클래스에서 온 것으로 보인다.
    - 4-5 학습 출력 끝의 클래스별 표에서 seed 2의 30308은 val mAP50-95가 0.537이었다. 이 클래스 하나가 최고값(0.995)이었다면 30종 평균 mAP50-95가 약 0.015 높아져, 차이 0.0175의 대부분이 사라진다.
    - 같은 seed의 train 30종 mAP75-95 평균이 0.9929라서, 나머지 29종이 모두 최고값이라고 해도 30308의 train mAP75-95는 0.932 이상이다. 즉 30308은 train 사진에서는 잘 맞히는데, 다른 조합의 val 사진에서는 seed에 따라 크게 틀린다.
    - 30308의 클래스별 mAP75-95는 6번에서 확인한다.

과적합과 과소적합 판단 (4-6, 4-7)

- 과적합: 전체적으로 학습 사진에만 맞춰진 모습은 보이지 않지만, 클래스에 따라서는 그런 신호가 있다. 단정하지는 않는다.
    - 없다고 보는 근거: val 손실이 최저점 뒤로 크게 오르지 않는다(마지막 값과 최저값의 차이가 seed 0은 0.027, seed 1은 0.020, seed 2는 0). val mAP도 떨어지지 않는다. 같은 30종의 train과 val 점수 차이가 0.007~0.020으로 작다.
    - 있을 수 있다고 보는 근거: seed 0, 2는 미리 정한 기준보다 train과 val 차이가 크고, seed 2는 그 차이가 30308 한 클래스에 몰려 있다. 30308처럼 train에서는 맞히고 val에서는 틀리는 클래스는, 그 클래스에 한해서는 학습 사진에 맞춰졌다고 볼 수 있다.
    - 단정할 수 없는 이유: val은 47장, 30종뿐이라 작은 차이는 우연과 구분하기 어렵다. train에만 있는 26종(bbox가 3개뿐인 약 17종 포함)은 val에 없어서 처음 보는 사진에서도 맞히는지 알 수 없다. 클래스별 mAP75-95는 아직 보지 않았다(6번).
- 과소적합: train 사진 기준으로는 아니라고 확실히 말할 수 있다.
    - 이유: 과소적합은 덜 배워서 학습한 사진조차 맞히지 못하는 상태를 말한다. 4-7에서 증강 없이 잰 train 점수가 56종 모두를 포함해 0.98~0.99로 최고값(0.995)에 가깝다.
    - 이것은 "덜 배웠는가"에 대한 답이다. test에 나오는 배우지 않은 약을 맞히는지와는 다른 문제다.

클래스별 결과 (학습 끝에 Ultralytics가 출력한 표, val에 있는 30종)

- Ultralytics 계산 방식에서는 AP의 최고값이 0.995다. 30종 대부분은 세 seed 모두 mAP50과 mAP50-95가 0.995다.
- 세 seed 모두 mAP50-95가 0.99보다 낮은 클래스는 16262(0.963 / 0.963 / 0.945), 18147(0.901 / 0.955 / 0.967), 25469(0.983 / 0.937 / 0.978)다. 셋 다 mAP50은 0.995라서, 약을 찾고 클래스도 맞혔지만 IoU 기준을 높이면 박스가 덜 맞는 것으로 보인다.
- 30308은 seed마다 크게 다르다(mAP50-95 0.995 / 0.924 / 0.537). seed 2는 mAP50도 0.537이라, 박스 위치보다 약을 놓쳤거나 다른 약으로 예측한 것으로 보인다.
- recall이 낮은데 mAP는 높은 클래스가 있다. 20238은 recall이 0.23 / 0.66 / 0.38인데 mAP50-95는 세 seed 모두 0.995다. Ultralytics의 precision과 recall은 전체 F1이 가장 높은 신뢰도 기준값 하나에서 잰 값이라, 이 약은 그 기준값보다 낮은 점수로 맞혔다는 뜻이다. 그래서 제출 CSV에는 점수가 낮은 예측도 넣어야 한다.
- precision이 낮은 클래스는 22347(0.41 / 0.74 / 0.50)이다. 22347로 예측한 박스 중 정답이 아닌 것이 많다는 뜻으로, 다른 약을 22347로 봤거나 같은 약에 박스를 겹쳐 냈을 수 있다.
- 위 해석은 숫자만 보고 한 것이다. 6번에서 val 예측을 다시 해서 박스 단위로 확인한다.

W&B 기록

- seed 1, 2를 학습하는 동안 W&B 서버 오류(Service Unavailable, 500)가 여러 번 났다. 학습은 영향 없이 끝났고 마지막에 동기화도 끝났다고 출력됐지만, W&B 화면의 seed 1 요약값은 89 epoch에서 멈춰 있었다. 결과는 로컬 run 폴더(`runs/rtdetr-l_e100_s*_fix`)를 기준으로 본다.

## 5. test 추론과 제출

- 세 seed의 best.pt로 test 사진 전체를 예측하고, 대회 형식 CSV로 저장한다. 제출은 Kaggle 웹사이트에서 CSV를 직접 올려서 한다.
- test에는 정답이 없다. 그래서 이 섹션에서는 제출 파일의 형식과 예측의 모양(사진당 박스 수)만 보고, 성능은 Kaggle 점수로 본다.
- 학습은 다시 하지 않는다. 커널을 다시 시작했다면 0번 두 셀과 3-1, 3-2를 실행한 뒤 이 섹션을 실행한다.

### 5-1. test 예측과 CSV 저장

- 입력 크기는 학습과 같은 640이다. 예측할 때도 사진을 640×640으로 늘려 맞추고, 예측한 박스는 Ultralytics가 원본 크기로 되돌린다.
- 신뢰도 0.01 이상인 예측을 모두 넣는다. AP는 클래스마다 예측을 점수 순으로 줄 세워 계산하므로, 정답보다 점수가 낮은 오답은 점수를 깎지 않고, 점수가 낮은 정답은 점수를 올린다. 4번 결과의 20238처럼 낮은 점수로 맞힌 약이 있었다.
- 클래스 번호(0~55)는 모델에 저장된 클래스 이름으로 약 번호로 되돌린다. 2-4에서 `data.yaml`의 클래스 이름을 약 번호로 적었기 때문이다.
- 박스는 꼭짓점 좌표를 사진 안으로 자른 뒤 `[x, y, w, h]` 픽셀로 바꾼다. 정답 박스는 사진 안에 있으므로, 사진 밖으로 나간 부분을 잘라도 겹치는 넓이는 그대로라 IoU가 줄지 않는다.
- `image_id`는 test 파일 이름의 숫자이고, `annotation_id`는 1부터 차례로 붙인다.
- 저장 위치는 `runs/<run 이름>/submission_<run 이름>.csv`다. 팀 제출 파일과 같은 이름 형식이다.
- 이 위치에 CSV가 이미 있으면 다시 예측하지 않고 그 파일을 읽는다. 5-3에서 Kaggle 점수를 받은 파일을 그대로 점검하고 분석하기 위해서다. 다시 예측하면 같은 모델이라도 점수의 끝자리가 달라질 수 있어, 제출한 파일과 어긋날 수 있다. 다시 예측하려면 `REPREDICT = True`로 바꾼다. 이때는 제출한 파일을 덮어쓴다.

In [ ]:
SUB_COLS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
PRED_CONF = 0.01  # 이 점수 이상인 예측을 모두 제출한다

test_paths = sorted(TEST_IMG_DIR.glob("*.png"))
assert all(p.stem.isdigit() for p in test_paths), "파일 이름이 숫자가 아닌 test 사진이 있습니다."
TEST_IDS = sorted(int(p.stem) for p in test_paths)
print("test 사진:", len(TEST_IDS), "장")


def submission_path(seed):
    run_name = make_run_name(seed)
    return RUNS_DIR / run_name / f"submission_{run_name}.csv"


def predict_test(seed):
    """seed의 best.pt로 test 사진을 예측해 제출 형식 표로 만든다."""
    model = RTDETR(str(RUNS_DIR / make_run_name(seed) / "weights" / "best.pt"))
    rows = []
    for r in model.predict(source=str(TEST_IMG_DIR), imgsz=IMGSZ, conf=PRED_CONF, device=DEVICE,
                           stream=True, verbose=False):  # stream: 사진을 한 장씩 처리해 메모리를 아낀다
        H, W = r.orig_shape
        image_id = int(Path(r.path).stem)
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(W, x2), min(H, y2)  # 사진 밖으로 나간 부분은 자른다
            if x2 <= x1 or y2 <= y1:
                continue
            rows.append([image_id, int(model.names[int(c)]),  # 0~55 → 약 번호
                         round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2), round(s, 4)])
    sub = pd.DataFrame(rows, columns=SUB_COLS[1:])
    sub.insert(0, "annotation_id", range(1, len(sub) + 1))
    return sub


REPREDICT = False  # False: 이미 있는 제출 CSV를 읽는다 / True: 다시 예측해서 덮어쓴다

subs = {}
for seed in SEEDS:
    path = submission_path(seed)
    if path.exists() and not REPREDICT:
        subs[seed] = pd.read_csv(path)
        print(f"seed {seed}: 저장된 CSV를 읽음, {len(subs[seed])}행 ← {rel(path)}")
    else:
        subs[seed] = predict_test(seed)
        subs[seed].to_csv(path, index=False)
        print(f"seed {seed}: 새로 예측해서 저장, {len(subs[seed])}행 → {rel(path)}")

### 5-2. 제출 파일 점검

- 형식: 열 순서, `annotation_id` 중복, 빈 값, 박스 너비와 높이가 0보다 큰지, score가 0~1인지, 약 번호가 학습한 56종 안에 있는지 확인한다. 하나라도 틀리면 멈춘다.
- 예측의 모양
    - 예측이 하나도 없는 test 사진 수
    - 사진당 박스 수(신뢰도 0.01 이상 전체)
    - 사진당 점수 0.5 이상인 박스 수. 0.5는 "모델이 자신 있게 낸 예측"을 세려고 정한 기준이다. 대회 사진에는 알약이 최대 4개라, 이 수가 알약 수보다 적으면 못 찾은 알약이 있고, 많으면 같은 알약에 겹친 예측이나 잘못 본 예측이 있다는 신호다.

In [ ]:
names = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))["names"]
TRAIN_CATS = {int(v) for v in names.values()}  # 학습한 56종의 약 번호

summary, conf_dist = [], []
for seed, sub in subs.items():
    assert list(sub.columns) == SUB_COLS, "열 순서가 다릅니다."
    assert sub["annotation_id"].is_unique, "annotation_id가 겹칩니다."
    assert sub.notna().all().all(), "빈 값이 있습니다."
    assert (sub[["bbox_w", "bbox_h"]] > 0).all().all(), "너비나 높이가 0 이하인 박스가 있습니다."
    assert sub["score"].between(0, 1).all(), "score가 0~1 밖에 있습니다."
    assert sub["category_id"].isin(TRAIN_CATS).all(), "학습하지 않은 약 번호가 있습니다."

    n_all = sub.groupby("image_id").size().reindex(TEST_IDS, fill_value=0)                         # 사진당 박스 수
    n_conf = sub[sub["score"] >= 0.5].groupby("image_id").size().reindex(TEST_IDS, fill_value=0)  # 사진당 0.5 이상 박스 수
    summary.append({"seed": seed, "행 수": len(sub), "예측 없는 사진": int((n_all == 0).sum()),
                    "사진당 박스(중앙값)": float(n_all.median()), "사진당 박스(최대)": int(n_all.max())})
    counts = n_conf.clip(upper=5).value_counts()
    conf_dist.append({"seed": seed, **{(f"{k}개" if k < 5 else "5개 이상"): int(counts.get(k, 0)) for k in range(6)}})

print("형식 점검 통과")
display(pd.DataFrame(summary).set_index("seed"))
print(f"사진당 점수 0.5 이상 박스 수별 test 사진 수 (전체 {len(TEST_IDS)}장)")
display(pd.DataFrame(conf_dist).set_index("seed"))

### 5-3. Kaggle 제출 결과

- 5-1에서 만든 CSV 세 개를 Kaggle 대회 페이지의 Submit Prediction에서 직접 올려 seed별로 제출했다(2026-10-29). 제출 설명에는 seed를 적었다.
- 점수는 Kaggle 제출 목록(Submissions) 화면에 나온 Public Score다. test 정답은 Kaggle에만 있어서, 이 점수가 test 성능을 볼 수 있는 유일한 숫자다.
- Kaggle의 public 점수는 보통 test의 일부로 매긴다. 이 대회가 test의 몇 %로 매기는지는 확인하지 않았다.

| seed | 제출 파일 | Kaggle public (mAP@[0.75:0.95]) |
|---|---|---|
| 0 | `submission_rtdetr-l_e100_s0_fix.csv` | 0.42152 |
| 1 | `submission_rtdetr-l_e100_s1_fix.csv` | 0.41761 |
| 2 | `submission_rtdetr-l_e100_s2_fix.csv` | 0.42012 |
| 평균 | | 0.41975 |

### 5-4. test 예측 분석: 겹친 박스

- 왜 하는가: 5-2에서 점수 0.5 이상 박스가 5개 이상인 사진이 seed마다 116~170장 나왔다. 대회 사진에는 알약이 최대 4개라, 한 알약에 자신 있는 박스가 겹쳐 있을 가능성이 크다. 겹친 박스가 어떤 약 번호끼리인지 보면, 모델이 어떤 알약을 확신하지 못하는지 알 수 있다.
- 방법
    - seed마다 점수 0.5 이상인 박스만 남기고, 같은 사진 안에서 박스끼리의 IoU(겹친 넓이 ÷ 합친 넓이)를 구한다.
    - IoU가 0.5 이상이면 같은 알약을 가리킨다고 본다. 1차 EDA 5-3에서 알약 중심이 정해진 6자리에 놓여 서로 떨어져 있었으므로, 서로 다른 알약의 박스가 절반 넘게 겹치는 경우는 드물다고 보았다.
    - 겹친 쌍을 둘로 나눈다.
        - 다른 약끼리: 한 알약을 두 가지 약으로 동시에 자신 있게 본 것이다. 배우지 않은 약이거나, 배운 약 중 비슷하게 생긴 약일 수 있다. 이 분석만으로는 둘 중 어느 쪽인지 알 수 없다.
        - 같은 약끼리: 같은 알약에 같은 약 번호로 박스를 두 번 낸 것(중복)이다. RT-DETR는 겹친 예측을 지우는 후처리(NMS)를 쓰지 않아서 생길 수 있다. 채점에서는 두 번째 박스가 오답이 된다.
- 출력
    - seed별 요약: 겹친 쌍 수(다른 약끼리, 같은 약끼리), 겹친 쌍이 있는 사진 수, 5개 이상인 사진 중 겹친 쌍이 있는 사진 수
    - 자주 겹친 약 번호 쌍(다른 약끼리) 상위 15개와 seed별 횟수
- 실행: 5-1을 실행한 뒤 실행한다(5-1이 읽은 제출 CSV를 쓴다).

In [ ]:
CONF_HI = 0.5   # 이 점수 이상을 "자신 있는 예측"으로 본다
IOU_SAME = 0.5  # 이 이상 겹치면 같은 알약을 가리킨다고 본다


def pair_iou(b):
    """b: (n, 4) [x, y, w, h] 픽셀 박스 → (n, n) IoU 표"""
    x1, y1 = b[:, 0], b[:, 1]
    x2, y2 = x1 + b[:, 2], y1 + b[:, 3]
    iw = np.clip(np.minimum(x2[:, None], x2) - np.maximum(x1[:, None], x1), 0, None)
    ih = np.clip(np.minimum(y2[:, None], y2) - np.maximum(y1[:, None], y1), 0, None)
    inter = iw * ih
    area = b[:, 2] * b[:, 3]
    return inter / (area[:, None] + area - inter)


OVERLAP_COLS = ["seed", "image_id", "iou", "약A", "점수A", "약B", "점수B", "같은 약"]
overlap_rows = []
for seed, sub in subs.items():
    hi = sub[sub["score"] >= CONF_HI]
    for image_id, g in hi.groupby("image_id"):
        b = g[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy(float)
        cat, sc = g["category_id"].to_numpy(), g["score"].to_numpy()
        iou = pair_iou(b)
        for i, j in zip(*np.triu_indices(len(g), k=1)):  # 한 사진 안의 모든 박스 쌍
            if iou[i, j] >= IOU_SAME:
                a, c = (i, j) if sc[i] >= sc[j] else (j, i)  # 점수가 높은 박스를 A로
                overlap_rows.append([seed, int(image_id), round(float(iou[i, j]), 3),
                                     int(cat[a]), float(sc[a]), int(cat[c]), float(sc[c]), bool(cat[a] == cat[c])])
overlaps = pd.DataFrame(overlap_rows, columns=OVERLAP_COLS)

summary = []
for seed, sub in subs.items():
    o = overlaps[overlaps["seed"] == seed]
    n_hi = sub[sub["score"] >= CONF_HI].groupby("image_id").size()
    many = set(n_hi[n_hi >= 5].index)  # 점수 0.5 이상 박스가 5개 이상인 사진
    with_pair = set(o["image_id"])
    summary.append({"seed": seed, "겹친 쌍": len(o),
                    "다른 약끼리": int((~o["같은 약"]).sum()), "같은 약끼리": int(o["같은 약"].sum()),
                    "겹친 쌍이 있는 사진": len(with_pair),
                    "5개 이상인 사진": len(many), "그중 겹친 쌍이 있는 사진": len(many & with_pair)})
print(f"점수 {CONF_HI} 이상 박스끼리 IoU {IOU_SAME} 이상으로 겹친 쌍")
display(pd.DataFrame(summary).set_index("seed"))

diff = overlaps[~overlaps["같은 약"]].copy()
diff["약 번호 쌍"] = [f"{min(a, b)} - {max(a, b)}" for a, b in zip(diff["약A"], diff["약B"])]
top_pairs = diff.groupby(["약 번호 쌍", "seed"]).size().unstack(fill_value=0)
top_pairs["합계"] = top_pairs.sum(axis=1)
print("자주 겹친 약 번호 쌍 (다른 약끼리, 상위 15개)")
display(top_pairs.sort_values("합계", ascending=False).head(15))

### 5-5. test 예측 분석: 예측한 약 분포

- 왜 하는가: 모델은 학습한 56종의 번호만 낼 수 있다. test에 배우지 않은 약이 있으면, 그 알약을 비슷한 배운 약의 번호로 예측할 수 있다. 그런 약은 주어진 라벨에서의 비율보다 test에서 훨씬 많이 예측될 것이다. 약 번호별로 test 예측 수를 라벨 수와 나란히 놓고 이런 약을 찾는다.
- 방법
    - test 예측 수: seed별로 점수 0.5 이상인 박스를 약 번호별로 센다(5-4와 같은 기준).
    - 라벨 수: `data/processed/yolo`의 txt 라벨(라벨 수정 반영, train 612개 + val 159개)을 약 번호별로 센다.
    - 배율 = (그 약의 test 예측 비율) ÷ (그 약의 라벨 비율). 세 seed의 예측 수 평균으로 계산한다. 1이면 주어진 데이터와 같은 비율로 예측했다는 뜻이고, 클수록 그 약을 더 많이 예측한 것이다.
    - 5-4에서 다른 약끼리 겹친 쌍에 그 약이 몇 번 나왔는지도 같이 붙인다.
    - 약 이름은 원본 JSON의 categories에서 가져온다.
- 출력
    - 배율 상위 15개 약
    - 세 seed 모두 점수 0.5 이상으로 한 번도 예측하지 않은 약
- 읽을 때 주의할 점: test의 실제 약 구성은 모른다. 배율이 큰 약은 test에 그 약이 정말 많아서일 수도 있고, 배우지 않은 약을 그 약으로 보고 있어서일 수도 있다. 한 번도 예측하지 않은 약도 test에 그 약이 없어서일 수 있다. 어느 쪽인지는 5-6에서 사진으로 확인한다.
- 실행: 5-1, 5-4를 실행한 뒤 실행한다.

In [ ]:
# 약 이름: 원본 JSON의 categories
CAT_NAME = {}
for p in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    for c in json.loads(p.read_text(encoding="utf-8")).get("categories", []):
        CAT_NAME.setdefault(int(c["id"]), c.get("name"))

# 라벨 수: YOLO txt를 약 번호별로 센다 (라벨 수정 반영)
yolo_names = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))["names"]
CAT_IDS_ALL = sorted(int(v) for v in yolo_names.values())
label_cnt = {"train": Counter(), "val": Counter()}
for s in label_cnt:
    for p in (YOLO_DIR / "labels" / s).glob("*.txt"):
        for line in p.read_text().splitlines():
            label_cnt[s][int(yolo_names[int(line.split()[0])])] += 1

dist = pd.DataFrame(index=pd.Index(CAT_IDS_ALL, name="약 번호"))
dist["약 이름"] = [CAT_NAME.get(c) for c in dist.index]
dist["train 라벨"] = [label_cnt["train"][c] for c in dist.index]
dist["val 라벨"] = [label_cnt["val"][c] for c in dist.index]
assert dist["train 라벨"].sum() + dist["val 라벨"].sum() == 771, "라벨 수가 771개가 아닙니다."

pred_cols = []
for seed, sub in subs.items():
    col = f"test 예측 s{seed}"
    dist[col] = sub[sub["score"] >= CONF_HI]["category_id"].value_counts().reindex(dist.index, fill_value=0)
    pred_cols.append(col)
dist["test 예측 평균"] = dist[pred_cols].mean(axis=1).round(1)

label_total = dist["train 라벨"] + dist["val 라벨"]
expected = label_total / label_total.sum() * dist["test 예측 평균"].sum()  # 라벨과 같은 비율이면 나올 예측 수
dist["배율"] = (dist["test 예측 평균"] / expected).round(2)
dist["겹친 쌍 등장(5-4)"] = pd.concat([diff["약A"], diff["약B"]]).value_counts().reindex(dist.index, fill_value=0)

show_cols = ["약 이름", "train 라벨", "val 라벨", *pred_cols, "test 예측 평균", "배율", "겹친 쌍 등장(5-4)"]
print(f"라벨 비율보다 많이 예측된 약 (배율 상위 15개, 점수 {CONF_HI} 이상 예측 기준)")
display(dist.sort_values("배율", ascending=False)[show_cols].head(15))

never = dist[(dist[pred_cols] == 0).all(axis=1)]
print(f"세 seed 모두 점수 {CONF_HI} 이상으로 한 번도 예측하지 않은 약: {len(never)}종")
display(never[["약 이름", "train 라벨", "val 라벨"]])

### 5-6. test 예측 분석: 사진으로 확인

- 왜 하는가: 5-4, 5-5는 숫자만 본 것이라, 그 번호로 예측한 알약이 실제로 그 약인지는 알 수 없었다. test에는 정답이 없어서, test 사진의 예측과 train의 같은 약 번호 알약을 나란히 놓고 사람이 직접 비교한다.
- 볼 사례 (사례마다 test 사진 2장)

| 사례 | 고른 이유 |
|---|---|
| 겹친 쌍 12778 - 19552 | 겹친 쌍 중 가장 많고(98번), 세 seed 모두에서 나왔다. 둘 다 val에 없는 약이다 |
| 겹친 쌍 12081 - 13395 | 두 번째로 많다(74번). 13395는 배율 6.6, 겹친 쌍 등장 79번이다 |
| 겹친 쌍 29345 - 30308 | seed 2에서만 많다. seed 2가 val에서 30308을 크게 틀린 것과 이어진다 |
| 겹친 쌍 20238 - 22347 | 둘 다 val에 있는 배운 약이다. 배운 약끼리 헷갈리는 경우를 비교 기준으로 본다 |
| 27777, 2483 | 배율은 크지만(5.0, 3.5) 겹친 쌍에는 한 번도 나오지 않았다 |
| 16551 | 배율 1위다(8.2) |

- 그림 읽는 법: 사례마다 그림 한 장이다.
    - 왼쪽 두 칸은 test 사진 전체다. 점수 0.5 이상 박스를 모두 그리고, 사례의 약 번호 박스는 빨강, 나머지는 파랑으로 그린다. 라벨은 약 번호와 점수다.
    - 오른쪽 칸은 train 예시다. 같은 약 번호의 train 라벨 박스를 잘라 낸 것으로, 생김새를 비교하기 위해 사진 전체 대신 알약 부분만 보여 준다. 약 번호마다 첫 번째 train 사진에서 가져온다.
- 사진 고르는 법: 겹친 쌍은 그 쌍이 가장 많이 나온 seed의 예측에서, 단일 약은 그 약을 가장 많이 예측한 seed에서 고른다. seed를 고정한 무작위로 2장을 뽑아서, 다시 실행해도 같은 사진이 나온다.
- 판단 기준 (그림을 보고 사람이 판단한다)
    - 겹친 쌍: 빨간 박스의 test 알약이 train 예시 둘 중 하나와 같아 보이면 배운 약끼리 헷갈린 것이다. 둘 다와 달라 보이면 배우지 않은 약일 가능성이 크다.
    - 단일 약: 빨간 박스의 test 알약이 train 예시와 같아 보이면 test에 그 약이 실제로 있는 것이다. 달라 보이면 다른 약을 그 번호로 본 것이다.
- 한계: 사례 14장만 보는 것이라 전체 비율은 알 수 없다. 사진마다 각도와 조명이 달라 같은 약도 달라 보일 수 있고, train 예시는 약마다 한 장뿐이다.
- 실행: 5-1, 5-4, 5-5를 실행한 뒤 실행한다.

In [ ]:
CASES = [  # (제목, 사례의 약 번호들)
    ("겹친 쌍 12778 - 19552", (12778, 19552)),
    ("겹친 쌍 12081 - 13395", (12081, 13395)),
    ("겹친 쌍 29345 - 30308", (29345, 30308)),
    ("겹친 쌍 20238 - 22347", (20238, 22347)),
    ("겹침 없이 많이 예측된 27777", (27777,)),
    ("겹침 없이 많이 예측된 2483", (2483,)),
    ("배율 1위 16551", (16551,)),
]
N_PER_CASE = 2
view_rng = np.random.default_rng(0)  # 사진을 고르는 무작위 seed (고정)

# train 예시: 약 번호마다 첫 번째 train 라벨 박스 (YOLO txt, 0~1 비율)
train_example = {}
for p in sorted((YOLO_DIR / "labels" / "train").glob("*.txt")):
    for line in p.read_text().splitlines():
        k, cx, cy, w, h = line.split()
        train_example.setdefault(int(yolo_names[int(k)]), (YOLO_DIR / "images" / "train" / f"{p.stem}.png",
                                                           float(cx), float(cy), float(w), float(h)))


def pick_images(cats):
    """사례에 맞는 seed와 test 사진 번호 N_PER_CASE장을 고른다."""
    if len(cats) == 2:  # 겹친 쌍: 그 쌍이 가장 많이 나온 seed
        m = overlaps[[{a, b} == set(cats) for a, b in zip(overlaps["약A"], overlaps["약B"])]]
        seed = int(m["seed"].value_counts().idxmax())
        ids = sorted(set(m.loc[m["seed"] == seed, "image_id"]))
    else:  # 단일 약: 그 약을 가장 많이 예측한 seed
        hi = {s: sub[(sub["score"] >= CONF_HI) & (sub["category_id"] == cats[0])] for s, sub in subs.items()}
        seed = max(hi, key=lambda s: len(hi[s]))
        ids = sorted(set(hi[seed]["image_id"]))
    chosen = view_rng.choice(ids, size=min(N_PER_CASE, len(ids)), replace=False)
    return seed, sorted(int(i) for i in chosen)


def draw_test(ax, seed, image_id, cats):
    with Image.open(TEST_IMG_DIR / f"{image_id}.png") as im:
        ax.imshow(np.asarray(im.convert("RGB")))
    g = subs[seed]
    g = g[(g["image_id"] == image_id) & (g["score"] >= CONF_HI)].sort_values("score", ascending=False)
    n_hit = 0
    for r in g.itertuples():
        hit = r.category_id in cats
        color = "red" if hit else "deepskyblue"
        ax.add_patch(Rectangle((r.bbox_x, r.bbox_y), r.bbox_w, r.bbox_h, fill=False, edgecolor=color, linewidth=2.5 if hit else 1.5))
        if hit and n_hit % 2:  # 같은 알약에 빨간 박스가 겹칠 때 라벨이 겹치지 않게 아래쪽에 적는다
            ax.text(r.bbox_x, r.bbox_y + r.bbox_h + 8, f"{r.category_id} {r.score:.2f}", color="white", fontsize=8, va="top",
                    bbox=dict(facecolor=color, edgecolor="none", pad=1))
        else:
            ax.text(r.bbox_x, r.bbox_y - 8, f"{r.category_id} {r.score:.2f}", color="white", fontsize=8, va="bottom",
                    bbox=dict(facecolor=color, edgecolor="none", pad=1))
        n_hit += hit
    ax.set_title(f"test {image_id}.png (seed {seed})", fontsize=9)
    ax.axis("off")


def draw_train(ax, cat):
    img_path, cx, cy, w, h = train_example[cat]
    with Image.open(img_path) as im:
        W, H = im.size
        crop = im.convert("RGB").crop((int((cx - w / 2) * W), int((cy - h / 2) * H), int((cx + w / 2) * W), int((cy + h / 2) * H)))
    ax.imshow(np.asarray(crop))
    ax.set_title(f"train 예시: {cat}\n{CAT_NAME.get(cat)}", fontsize=9)
    ax.axis("off")


for title, cats in CASES:
    seed, ids = pick_images(cats)
    print(f"{title}: seed {seed}, test 사진 {ids}")
    fig, axes = plt.subplots(1, len(ids) + len(cats), figsize=(4 * len(ids) + 3 * len(cats), 5.5), layout="constrained",
                             gridspec_kw={"width_ratios": [4] * len(ids) + [3] * len(cats)})
    for ax, image_id in zip(axes, ids):
        draw_test(ax, seed, image_id, cats)
    for ax, cat in zip(axes[len(ids):], cats):
        draw_train(ax, cat)
    fig.suptitle(f"{title} (빨강: 사례의 약 번호, 파랑: 다른 약 번호, 점수 {CONF_HI} 이상)", fontsize=11)
    plt.show()

### 5번 결과

제출 파일과 예측의 모양 (5-1, 5-2)

- 5-1은 세 seed 모두 `runs/`에 저장된 제출 CSV(5-3에서 점수를 받은 파일)를 읽었다. 5-2의 형식 점검은 모두 통과했다.
- 세 파일 모두 252,600행으로, test 842장마다 예측이 정확히 300개씩이다. RT-DETR는 사진마다 예측(쿼리) 300개를 내는데, 그 300개가 모두 신뢰도 0.01을 넘어서 하나도 걸러지지 않았다. 예측이 없는 사진은 없다.

| seed | 점수 0.5 이상 박스 2개 | 3개 | 4개 | 5개 이상 |
|---|---|---|---|---|
| 0 | 0장 | 147장 | 536장 | 159장 |
| 1 | 2장 | 148장 | 576장 | 116장 |
| 2 | 0장 | 145장 | 527장 | 170장 |

- 점수 0.5 이상 박스가 0개, 1개인 사진은 세 seed 모두 없다.
- 대회 사진에는 알약이 최대 4개다. 그런데 점수 0.5 이상 박스가 5개 이상인 사진이 seed마다 116~170장(전체의 14~20%)이다. 이 사진들에는 한 알약에 자신 있는 박스가 둘 이상 겹쳤거나, 알약이 없는 곳에 자신 있게 낸 예측이 있다. 어느 쪽인지는 5-4에서 본다.
- 3개, 4개인 사진은 알약 수와 맞는지 알 수 없다. test 사진마다 알약이 몇 개인지 모르기 때문이다. 3개인 사진은 알약이 3개인 사진일 수도 있고, 4개 중 하나를 자신 있게 찾지 못한 사진일 수도 있다.

val 점수와 Kaggle(test) 점수

| seed | val mAP75-95 (4번 결과) | Kaggle public mAP@[0.75:0.95] (5-3) | 차이 |
|---|---|---|---|
| 0 | 0.980 | 0.4215 | 0.558 |
| 1 | 0.978 | 0.4176 | 0.560 |
| 2 | 0.973 | 0.4201 | 0.553 |
| 평균 | 0.977 | 0.4198 | 0.557 |

- 두 점수는 같은 지표(IoU 0.75~0.95의 AP 평균)지만 계산하는 곳이 다르다. val은 Ultralytics가 val 정답에 있는 30종으로 계산했고, Kaggle 점수는 Kaggle이 test 정답으로 계산했다. 그래서 숫자를 그대로 빼는 것은 대략의 비교다.

보이는 것

- test 점수도 seed에 따라 거의 흔들리지 않는다(0.4176~0.4215, 차이 0.004). val의 seed 차이(0.007)보다도 작다.
- val 순위와 test 순위가 다르다. val은 seed 0 > 1 > 2, test는 seed 0 > 2 > 1이다. 차이가 워낙 작아서, 지금 val 점수로는 test에서 더 나을 seed를 고를 수 없다.
- val과 test의 차이가 세 seed 모두 0.55 정도로 크고, 크기도 거의 같다. 우연이 아니라 test 데이터가 train, val과 다르기 때문으로 보인다.

test 점수가 낮은 이유 (지울 수 있는 설명부터)

1. 덜 배워서(과소적합)는 아니다. 4-7에서 증강 없이 잰 train 점수가 0.98~0.99로 최고값에 가깝다.
2. 학습 사진만 외워서(과적합)라고 보기도 어렵다. 학습에 쓰지 않은 val 사진(train에 없는 조합)에서도 0.97~0.98이고, 4-7의 train과 val 차이(0.007~0.020)는 val과 test 차이(약 0.56)에 비하면 아주 작다.
3. 남는 설명은 test가 train, val과 다르다는 것이다. 대회 설명에 "Test Data에는 Train Data엔 없는 클래스가 존재한다"고 되어 있고, 1차 EDA 7번에서 test 사진을 눈으로 봤을 때도 train에 없는 약이 많았다. 모델은 학습한 56종의 번호만 낼 수 있어서, 배우지 않은 약은 맞힐 수 없다.

계산으로 본 범위

- Kaggle 지표가 COCO처럼 클래스마다 AP를 구해 평균한다면, 점수는 "배운 약들의 AP 합 ÷ test에 나온 약 종류 수"다. 배우지 않은 약의 AP는 0이다.
- AP는 1을 넘지 않으므로, 0.42가 나오려면 test에 나온 약 종류 중 적어도 42%는 배운 약이어야 한다. 반대로 최대 58%까지는 배우지 않은 약일 수 있다.
- 배운 약의 test AP가 val처럼 약 0.977이라고 가정하면, 배운 약의 비율은 약 43%(0.4198 ÷ 0.977)다.

단정할 수 없는 점

- 위 계산은 Kaggle 지표가 클래스별 평균이라는 가정에 기댄다. Kaggle의 계산 방식은 확인하지 않았다.
- test 정답이 없어서, 배우지 않은 약이 몇 종인지, 배운 약의 test 점수가 val처럼 높은지 직접 확인할 수 없다. 촬영 조건 같은 다른 차이가 함께 있을 수도 있다.
- val로 best.pt를 골랐기 때문에 val 점수는 실제보다 조금 좋게 나왔을 수 있다. 다만 4-6에서 40 epoch 이후 곡선이 거의 평평해서, 그 영향은 val과 test 차이에 비하면 작다고 본다.
- Kaggle public 점수는 test의 일부로 매긴 점수일 수 있다(5-3).
- 그래서 이 이유는 정답 없이 할 수 있는 test 예측 분석으로 간접 확인한다.

겹친 박스 (5-4)

| seed | 겹친 쌍 | 다른 약끼리 | 같은 약끼리 | 겹친 쌍이 있는 사진 | 5개 이상인 사진 | 그중 겹친 쌍이 있는 사진 |
|---|---|---|---|---|---|---|
| 0 | 131 | 111 | 20 | 126 | 159 | 119 (75%) |
| 1 | 132 | 102 | 30 | 109 | 116 | 105 (91%) |
| 2 | 208 | 193 | 15 | 160 | 170 | 158 (93%) |

- 점수 0.5 이상 박스가 5개 이상인 사진의 75~93%에 IoU 0.5 이상으로 겹친 쌍이 있다. 5개 이상이 나온 주된 원인은 한 알약에 박스가 겹친 것으로 보인다. 겹친 쌍이 없는 나머지 사진(seed 0은 40장, 1은 11장, 2는 12장)은 겹침이 IoU 0.5보다 작거나, 알약이 없는 곳에 낸 예측일 수 있다.
- 겹친 쌍의 77~93%가 다른 약끼리다. 알약의 위치는 잡았지만 어떤 약인지 확신하지 못하는 경우가 대부분이다. 같은 약끼리 겹친 중복은 seed마다 15~30쌍으로 적다.

자주 겹친 약 번호 쌍 (다른 약끼리, 합계 상위 5개)

| 약 번호 쌍 | seed 0 | seed 1 | seed 2 | 합계 | val에 있는 약 |
|---|---|---|---|---|---|
| 12778 - 19552 | 31 | 37 | 30 | 98 | 둘 다 없음 |
| 12081 - 13395 | 3 | 0 | 71 | 74 | 둘 다 없음 |
| 20238 - 22347 | 10 | 18 | 1 | 29 | 둘 다 있음 |
| 29345 - 30308 | 1 | 0 | 26 | 27 | 30308만 있음 |
| 16551 - 27926 | 0 | 14 | 7 | 21 | 둘 다 없음 |

- val에 있는 약은 4-3~4-5 학습 출력 끝의 클래스별 표(30종)로 확인했다.
- 12778 - 19552는 세 seed 모두 30번 넘게 나온다. seed와 관계없이 생기는 혼동이라, 이 두 약을 닮은 test 알약(배우지 않은 약일 수 있다)이 있거나 두 약 자체가 닮았을 수 있다. 두 약 모두 val에 없어서 val로는 확인할 수 없었던 약이다.
- 12081 - 13395(seed 2에서 71번), 29345 - 30308(seed 2에서 26번)은 seed 2에 몰려 있다. 어떤 약끼리 헷갈리는지가 seed에 따라 달라진다.
- 4번 결과와 이어지는 것
    - seed 2는 val에서 30308을 크게 틀렸고(mAP50-95 0.537), test에서 30308과 29345를 겹쳐 낸 경우도 seed 2에 몰려 있다. seed 2가 30308을 29345와 헷갈리는 것으로 보인다. 6번에서 val 예측으로 확인한다.
    - val에서 22347은 precision이, 20238은 recall이 낮았는데, test에서 20238 - 22347이 겹친 쌍으로 자주 나온다. 배우지 않은 약뿐 아니라 배운 약끼리도 헷갈리는 경우가 있다는 뜻이다.
- 이 분석은 박스가 겹쳤는지만 본 것이라, 둘 중 어느 약 번호가 맞는지는 알 수 없다. 사진으로 확인하는 것은 5-6에서 한다.

예측한 약 분포 (5-5)

| 약 번호 | 약 이름 | train 라벨 | val 라벨 | test 예측 평균 | 배율 | 겹친 쌍 등장(5-4) |
|---|---|---|---|---|---|---|
| 16551 | 동아가바펜틴정 800mg | 3 | 0 | 108.0 | 8.17 | 27 |
| 3743 | 알드린정 | 3 | 0 | 100.7 | 7.62 | 20 |
| 13395 | 써스펜8시간이알서방정 650mg | 3 | 0 | 87.3 | 6.60 | 79 |
| 34597 | 제미메트서방정 50/1000mg | 3 | 0 | 72.3 | 5.47 | 11 |
| 27777 | 카나브정 60mg | 3 | 0 | 66.0 | 4.99 | 0 |
| 31885 | 자누메트엑스알서방정 100/1000mg | 3 | 0 | 62.3 | 4.71 | 7 |
| 27926 | 울트라셋이알서방정 | 3 | 0 | 61.0 | 4.61 | 41 |
| 12778 | 다보타민큐정 10mg/병 | 6 | 0 | 118.0 | 4.46 | 99 |
| 12081 | 리렉스펜정 300mg/PTP | 3 | 0 | 49.3 | 3.73 | 74 |
| 19552 | 트루비타정 60mg/병 | 3 | 0 | 47.7 | 3.61 | 100 |
| 2483 | 뮤테란캡슐 100mg | 9 | 0 | 138.0 | 3.48 | 0 |
| 28763 | 트라젠타정(리나글립틴) | 3 | 6 | 134.0 | 3.38 | 21 |
| 3544 | 무코스타정(레바미피드)(비매품) | 6 | 0 | 87.3 | 3.30 | 9 |
| 29345 | 비모보정 500/20mg | 6 | 0 | 73.0 | 2.76 | 32 |
| 21771 | 라비에트정 20mg | 3 | 0 | 32.7 | 2.47 | 25 |

- 배율 상위 15개 중 14개는 val 라벨이 0인 약(train에만 있는 약)이고, 11개는 train 라벨이 3개뿐이다. 라벨이 3개뿐인 약이 test에서는 seed 평균 30~110번씩 점수 0.5 이상으로 예측된다(배율 2.5~8.2).
- 세 seed 모두 점수 0.5 이상으로 한 번도 예측하지 않은 약은 없다.
- 해석은 두 가지가 가능하고, 이 표만으로는 가를 수 없다.
    - test에 이 약들이 실제로 많을 수 있다. 1차 EDA 4-2, 4-3에서 라벨이 3개뿐인 약은 조합 하나에만 나오는 약이었다(한 조합을 세 각도로 찍어서 3개). train에는 조합이 하나뿐이어도 test에는 그 약이 든 조합이 더 있을 수 있어서, 라벨 비율이 test의 약 구성을 대표하지 못할 수 있다.
    - 배우지 않은 약을 이 약들의 번호로 예측하고 있을 수 있다. 라벨이 3개뿐이라 이 약들의 생김새를 좁게 배웠을 수 있다.
- 5-4와 같이 보면 단서가 있다.
    - 많이 예측되면서 다른 약과 겹친 쌍에도 자주 나온 약(13395 79번, 12778 99번, 12081 74번, 19552 100번, 27926 41번)은 다른 약과 경쟁하면서 예측됐다. 처음에는 확신하지 못한 알약, 즉 배우지 않은 약이 이 번호들로 들어간 것으로 보았지만, 5-6에서 사진으로 확인해 보니 겹친 쌍 사진 8장 중 7장은 실제로 둘 중 한 약이었다. 겹친 쌍이 많다는 것만으로는 배우지 않은 약이라고 볼 수 없다. 다만 5-6의 파란 박스에서는 13395와 12778이 다른 약과 겹치지 않고도 배우지 않은 닮은 약(타이레놀류, 비타민정)을 높은 점수로 예측했다. 약에 따라 두 해석이 함께 일어나고 있다.
    - 27777(카나브정 60mg)과 2483(뮤테란캡슐 100mg)은 많이 예측됐지만 겹친 쌍에는 한 번도 나오지 않았다. 다른 약과 경쟁 없이 예측한 것이라, test에 실제로 많을 가능성도 있다.
- test의 실제 약 구성을 모르기 때문에 위 해석은 확인되지 않았다. 5-6에서 사진으로 확인한다.

사진으로 확인 (5-6)

- 그림의 알약을 사람이 직접 보고, 각인을 약학정보원에서 찾아 실제 약을 정했다. 점수는 그림의 라벨에서 읽은 값이다. train에 있는 약인지는 1차 EDA 클래스 표의 약 이름(56종)으로 확인했다.

빨간 박스(사례의 약 번호) 14개

| 사례 | test 사진 (seed) | 실제 약 (각인 앞 / 뒤) | train에 있는 약인가 | 빨간 박스 예측 | 판단 |
|---|---|---|---|---|---|
| 12778 - 19552 | 936 (1) | 트루비타정 (YH / TV) | 있음 (19552) | 19552 0.81, 12778 0.67 | 맞는 약이 더 높고, 닮은 약도 0.5를 넘었다 |
| 12778 - 19552 | 1312 (1) | 메가파워정 (YH / MG) | 없음 | 12778 0.84, 19552 0.52 | 배우지 않은 약을 배운 약 번호로 예측했다 |
| 12081 - 13395 | 475 (2) | 써스펜8시간이알서방정 650mg (SUSPENER / 없음) | 있음 (13395) | 13395 0.79, 12081 0.61 | 맞는 약이 더 높다 |
| 12081 - 13395 | 482 (2) | 써스펜8시간이알서방정 650mg (SUSPENER / 없음) | 있음 (13395) | 13395 0.79, 12081 0.66 | 맞는 약이 더 높다 |
| 29345 - 30308 | 286 (2) | 트라젠타듀오정 2.5/850mg (문양 / D2/850) | 있음 (30308) | 30308 0.73, 29345 0.71 | 맞는 약이 근소하게 높다 |
| 29345 - 30308 | 288 (2) | 트라젠타듀오정 2.5/850mg (문양 / D2/850) | 있음 (30308) | 30308 0.80, 29345 0.57 | 맞는 약이 더 높다 |
| 20238 - 22347 | 945 (1) | 플라빅스정 75mg (75 / 1171) | 있음 (20238) | 22347 0.71, 20238 0.53 | 틀린 약이 더 높다 |
| 20238 - 22347 | 1198 (1) | 플라빅스정 75mg (75 / 1171) | 있음 (20238) | 20238 0.76, 22347 0.56 | 맞는 약이 더 높다 |
| 27777 | 521 (0) | 카나브정 60mg (B\|R / FMS6) | 있음 (27777) | 27777 0.85 | 맞다 |
| 27777 | 830 (0) | 카나브정 60mg (B\|R / FMS6) | 있음 (27777) | 27777 0.85 | 맞다 |
| 2483 | 743 (1) | 뮤테란캡슐 100mg (Hanwha MUC100 / 없음) | 있음 (2483) | 2483 0.93 | 맞다 |
| 2483 | 984 (1) | 뮤테란캡슐 100mg (Hanwha MUC100 / 없음) | 있음 (2483) | 2483 0.93 | 맞다 |
| 16551 | 750 (1) | 케이캡정 50mg (K\|50 / 없음) | 없음 | 16551 0.54 | 배우지 않은 약을 배운 약 번호로 예측했다 |
| 16551 | 1279 (1) | 동아가바펜틴정 800mg (DGTH / 800) | 있음 (16551) | 16551 0.80 | 맞다 |

파란 박스(사례가 아닌 약 번호) 중 눈에 띈 알약 6개

| test 사진 (seed) | 알약 | 실제 약 (각인 앞 / 뒤) | train에 있는 약인가 | 예측 |
|---|---|---|---|---|
| 936 (1) | 흰 알약 | 타이레놀정 500mg (TYLENOL / 500) | 없음 | 13395 0.77 |
| 1312 (1) | 흰 알약 | 타이레놀8시간이알서방정 (TYLENOL ER / 없음) | 없음 | 13395 0.77 |
| 743 (1) | 흰 알약 | 타이레놀정 500mg (TYLENOL / 500) | 없음 | 13395 0.82 |
| 482 (2) | 빨간 알약 | 메가파워정 (YH / MG) | 없음 | 12778 0.91 |
| 984 (1) | 빨간 알약 | 비타비백정 (YH / V100) | 없음 | 12778 0.87 |
| 1279 (1) | 노란 알약 | 낙소졸정 500/20mg (Hanmi / Naxozol 500/20) | 없음 | 29345 0.71 |

- 파란 박스 6개는 그림에서 예측이 이상해 보인 것만 골라 판단했다. 무작위로 고른 것이 아니라서, 이 표로 배우지 않은 약의 비율을 말할 수는 없다.

확인된 것

- 배우지 않은 약: 판단한 알약 20개 중 8개(6종: 메가파워정, 케이캡정 50mg, 타이레놀정 500mg, 타이레놀8시간이알서방정, 비타비백정, 낙소졸정 500/20mg)가 train에 없는 약이었다. 모델은 이 8개를 모두 배운 약의 번호로, 점수 0.54~0.91로 예측했다. 대부분 색, 모양, 각인이 닮은 약의 번호였다. 모델이 배우지 않은 약을 배운 약 번호로 예측한다는 5번 결과의 설명을 실제 사례로 확인했고, 그 예측이 꽤 높은 점수로 나온다는 것도 보였다. 모델이 "모르는 약"이라는 신호를 내지 못한다.
- 닮은 약끼리의 짝: 흰색 장방형 타이레놀류는 써스펜8시간이알서방정(13395)으로, YH 각인의 빨간 비타민정(메가파워정, 비타비백정)은 다보타민큐정(12778)으로, 낙소졸정 500/20mg은 함량 표기(500/20)가 같은 비모보정(29345)으로 예측됐다. 케이캡정은 동아가바펜틴정(16551)으로 예측됐는데, 길쭉한 모양 말고는 색과 각인이 달라 닮았다고 보기는 어렵다.
- 겹친 쌍(빨간 박스 8개): 7개는 실제 약이 겹친 두 번호 중 하나였다(배운 약끼리 헷갈림). 그중 6개는 맞는 약의 점수가 더 높았고, 945 한 개는 틀린 약(22347)이 더 높았다. 나머지 1개(1312)는 배우지 않은 약이었다. 겹친 쌍은 대부분 모델이 맞는 약을 찾으면서 닮은 다른 약 번호도 높은 점수로 함께 낸 경우였다.
- 많이 예측된 약(5-5)
    - 27777, 2483은 빨간 박스 4개 모두 맞았다. test에 이 약들이 실제로 많다.
    - 13395는 실제 써스펜(빨간 박스 2개)도 있었고, 배우지 않은 타이레놀류(파란 박스 3개)도 떠안았다.
    - 12778은 판단한 사례 중에 실제 다보타민큐정은 없었고, 배우지 않은 비타민정만 3개(메가파워정 2개, 비타비백정 1개) 있었다.
    - 16551은 1개는 맞았고, 1개는 배우지 않은 약(케이캡정)이었다.
    - 5-5의 두 해석(test에 실제로 많다, 배우지 않은 약을 떠안는다)이 약에 따라 둘 다 일어나고 있다.
- 4번 결과와 이어지는 것
    - seed 2의 30308: test 사진 2개에서는 30308을 더 높은 점수로 맞게 냈다. 다만 29345도 0.5 넘게 함께 냈다. val에서 30308을 크게 틀린 이유는 6번에서 val 예측으로 확인한다.
    - 20238 - 22347: 플라빅스정(20238)을 자누비아정(22347)으로도 보고, 1개는 틀린 쪽이 더 높았다. val에서 22347의 precision이 낮았던 것과 맞는다.

점수에 주는 영향

- 겹친 다른 약 박스는 그 약 번호에서는 오답이 된다. 맞는 약의 박스는 따로 있어서 맞는 약 번호의 점수는 크게 깎이지 않지만, 틀린 약 번호의 AP는 깎인다.
- 배우지 않은 약은 자기 AP가 0이 되는 데서 끝나지 않는다. 닮은 배운 약 번호(13395, 12778, 29345, 16551)에 점수가 높은 오답을 보태서, 그 배운 약의 AP도 깎는다.
- 그래서 5번 결과 앞부분의 "배운 약의 test AP가 val처럼 약 0.977"이라는 가정은 실제보다 낙관적일 수 있다. 배운 약의 test AP가 이보다 낮다면, 같은 점수 0.42를 설명하는 배운 약의 비율은 43%보다 높아진다. "적어도 42%"라는 아래 한계는 그대로다.

한계

- 빨간 박스는 사례마다 2장씩 14개, 파란 박스는 눈에 띈 6개만 봤다. 전체에서 배우지 않은 약이 얼마나 되는지는 알 수 없다.
- 실제 약은 사람이 각인을 보고 판단한 것이다.

## 6. val 분석

- 왜 하는가: 4번에서는 val 점수를 전체 평균과 Ultralytics가 출력한 클래스별 표로만 봤고, 5번의 test는 정답이 없어서 예측의 모양으로 추정만 했다. val에는 정답이 있으므로, 배운 약을 처음 보는 사진에서 어디를 어떻게 틀리는지 정답과 맞대서 확인한다.
- val은 47장, 정답 박스 159개, 30종이다(2-3). 30종 모두 train에도 있는 약이다. train에만 있는 26종과 test에만 있는 약은 이 섹션에서 볼 수 없다.
- 학습은 다시 하지 않는다. 커널을 다시 시작했다면 0번 두 셀, 3-1, 5-1, 5-4, 5-5를 먼저 실행한다. 5번에서 만든 값(점수 기준 `PRED_CONF`와 `CONF_HI`, 약 번호표, 약 이름, train 라벨 수)을 그대로 쓰기 때문이다. 5-1은 저장된 CSV를 읽기만 한다.

| 단계 | 내용 |
|---|---|
| 6-1 | val 예측과 정답 준비 |
| 6-2 | 클래스별 AP와 약의 성질 |
| 6-3 | 박스 단위 에러 분류 |
| 6-4 | PR 곡선 (IoU 0.75, 0.95) |

### 6-1. val 예측과 정답 준비

- 박스 하나하나를 정답과 맞대 보려면 박스별 예측이 필요하다. Ultralytics의 `model.val()`은 점수만 돌려주고 박스별 예측은 돌려주지 않아서, val 사진을 따로 예측한다.
- 예측: 세 seed의 best.pt로 val 47장을 예측한다. 조건은 test 제출(5-1)과 같다. 입력 크기는 640이고(사진을 640×640으로 늘려 맞춘다), 점수 0.01(`PRED_CONF`) 이상인 예측을 모두 남기고, 사진 밖으로 나간 부분은 자른다.
- 정답: 2-4에서 만든 val의 YOLO txt 라벨(라벨 수정 반영)을 픽셀 박스 `[x, y, w, h]`로 되돌려 쓴다. 모델이 학습과 평가에 쓴 정답과 같다.
- 결과는 두 표다. 정답 표 `val_gt`(사진 이름, 약 번호, 박스)와 예측 표 `val_pred`(seed, 사진 이름, 약 번호, 박스, 점수)다. 6-3부터 이 두 표를 쓴다. 6-2의 AP는 Ultralytics val로 따로 잰다.
- 점검: 사진마다 점수 0.5 이상(`CONF_HI`, 5-4와 같은 기준) 예측 수를 정답 수와 비교한다. 5-2에서 test는 점수 0.5 이상 박스가 5개 이상인 사진이 116~170장이었는데, val에서도 정답보다 많이 나오는지 본다.
    - 정답보다 많으면: 같은 알약에 박스가 겹쳤거나, 정답이 없는 곳을 예측했다.
    - 정답보다 적으면: 알약을 놓쳤거나, 그 알약의 점수가 0.5보다 낮다.
    - 수가 같아도 약 번호나 위치가 틀렸을 수 있다. 그것은 6-3에서 본다.
- 예측은 파일로 저장하지 않는다. 47장이라 test(842장)보다 훨씬 빨리 끝나서, 커널을 다시 시작하면 다시 예측한다.

In [ ]:
VAL_IMG_DIR = YOLO_DIR / "images" / "val"
VAL_LBL_DIR = YOLO_DIR / "labels" / "val"
val_paths = sorted(VAL_IMG_DIR.glob("*.png"))
assert len(val_paths) == 47, f"val 사진이 47장이 아닙니다: {len(val_paths)}"
BOX_COLS = ["x", "y", "w", "h"]  # 픽셀, 왼쪽 위 꼭짓점과 너비, 높이

# 정답: YOLO txt(클래스 번호, 0~1 비율의 중심과 크기) → 픽셀 [x, y, w, h]
gt_rows = []
for p in val_paths:
    with Image.open(p) as im:
        W, H = im.size
    for line in (VAL_LBL_DIR / f"{p.stem}.txt").read_text().splitlines():
        k, cx, cy, w, h = line.split()
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        gt_rows.append([p.name, int(yolo_names[int(k)]), cx - w / 2, cy - h / 2, w, h])  # 0~55 → 약 번호
val_gt = pd.DataFrame(gt_rows, columns=["file_name", "category_id", *BOX_COLS])
assert len(val_gt) == 159, f"val 정답 박스가 159개(2-3)가 아닙니다: {len(val_gt)}"


def predict_val(seed):
    """seed의 best.pt로 val 사진을 예측해 표로 만든다. 조건은 5-1의 test 예측과 같다."""
    model = RTDETR(str(RUNS_DIR / make_run_name(seed) / "weights" / "best.pt"))
    rows = []
    for r in model.predict(source=str(VAL_IMG_DIR), imgsz=IMGSZ, conf=PRED_CONF, device=DEVICE,
                           stream=True, verbose=False):
        H, W = r.orig_shape
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            x1, y1, x2, y2 = max(0.0, x1), max(0.0, y1), min(W, x2), min(H, y2)  # 사진 밖으로 나간 부분은 자른다
            if x2 <= x1 or y2 <= y1:
                continue
            rows.append([seed, Path(r.path).name, int(model.names[int(c)]), x1, y1, x2 - x1, y2 - y1, s])
    return pd.DataFrame(rows, columns=["seed", "file_name", "category_id", *BOX_COLS, "score"])


val_pred = pd.concat([predict_val(seed) for seed in SEEDS], ignore_index=True)
print(f"정답 표: {len(val_gt)}행 / 예측 표: {len(val_pred)}행 (seed {SEEDS})")

gt_n = val_gt.groupby("file_name").size()  # 사진당 정답 수
print("사진당 정답 수별 val 사진 수:", {int(k): int(v) for k, v in gt_n.value_counts().sort_index().items()})

check = []
for seed, g in val_pred.groupby("seed"):
    hi_n = g[g["score"] >= CONF_HI].groupby("file_name").size().reindex(gt_n.index, fill_value=0)  # 사진당 0.5 이상 예측 수
    check.append({"seed": seed, f"예측 (점수 {PRED_CONF} 이상)": len(g), f"예측 (점수 {CONF_HI} 이상)": int(hi_n.sum()),
                  "정답": int(gt_n.sum()), "정답 수와 같은 사진": int((hi_n == gt_n).sum()),
                  "정답보다 많은 사진": int((hi_n > gt_n).sum()), "정답보다 적은 사진": int((hi_n < gt_n).sum())})
print(f"사진마다 점수 {CONF_HI} 이상 예측 수와 정답 수 비교 (val {len(gt_n)}장)")
display(pd.DataFrame(check).set_index("seed"))

### 6-2. 클래스별 AP와 약의 성질

- 왜 하는가: 4번 결과의 클래스별 점수는 Ultralytics가 출력한 mAP50과 mAP50-95뿐이었다. 대회 지표에 가까운 mAP75-95를 클래스마다 보고, 점수가 낮은 약에 공통된 성질(학습한 박스 수, 모양과 색이 닮은 약의 수)이 있는지 표로 비교한다.
- AP 계산: 4-7과 같다. seed마다 best.pt로 `model.val()`을 val에 실행하고(점수 0.001 이상, Ultralytics 기본값), 클래스 × IoU 10단계 AP(`all_ap`)에서 클래스마다 IoU 단계를 골라 평균한다.
    - mAP50: IoU 0.5 한 단계. 박스가 정답과 절반쯤 겹쳐도 맞음으로 친다. 약을 찾고 약 번호를 맞혔는지를 본다.
    - mAP75-95: IoU 0.75, 0.80, ..., 0.95 다섯 단계의 평균. 박스가 정답과 얼마나 딱 맞는지까지 본다. 대회 지표와 IoU 범위가 같다. 계산 방식은 Ultralytics 방식이라 대회 점수의 근사값이다(4-7과 같다).
    - 읽는 법: mAP50은 높은데 mAP75-95만 낮으면 약은 맞혔지만 박스 위치가 덜 맞은 것이다. mAP50부터 낮으면 놓쳤거나 다른 약으로 본 박스가 있다.
    - Ultralytics 계산 방식에서는 AP의 최고값이 0.995다(4번 결과). 0.995면 그 IoU 기준에서 그 약을 모두 맞혔다는 뜻이다.
- 약의 성질 (표의 열)
    - val 정답: val에 있는 그 약의 정답 박스 수(6-1의 `val_gt`). 적을수록 박스 하나만 틀려도 AP가 크게 바뀐다.
    - train 라벨: 그 약을 학습한 박스 수(5-5와 같은 값).
    - 모양, 색: 원본 JSON의 `drug_shape`, `color_class1`이다. 1차 EDA 4-4와 같은 방법으로, 약마다 처음 읽은 JSON의 값을 쓴다(같은 약의 JSON끼리는 값이 모두 같다).
    - 같은 모양·색 약: 학습한 56종 중 모양과 색이 같은 다른 약의 수. 1차 EDA 4-4의 묶음 기준과 같다. 많을수록 식별 문자 말고는 구분할 단서가 적다. 0이면 모양과 색이 같은 약이 없다.
- 출력
    - 먼저 seed마다 30종의 AP를 평균한 표를 낸다. 4-7의 val 줄(mAP50 / mAP50-95 / mAP75-95가 seed 0은 0.9950 / 0.9883 / 0.9816, seed 1은 0.9926 / 0.9865 / 0.9803, seed 2는 0.9798 / 0.9764 / 0.9731)과 같으면 클래스별 값을 제대로 꺼낸 것이다.
    - 다음으로 30종 전체를 mAP75-95 세 seed 평균이 낮은 순으로 보여 준다. 값은 소수 셋째 자리까지다.
- Ultralytics의 진행 표시는 이 셀에서만 끈다. 진행 표시에 내 컴퓨터의 절대경로가 찍히기 때문이다. 평가 결과 폴더는 4-7과 같은 `eval_val`에 덮어쓰고, 그래프는 저장하지 않는다.
- 한계: val 정답이 몇 개뿐인 약은 우연에 크게 흔들리므로, seed 하나의 값보다 세 seed를 함께 본다. 30종뿐이라 약의 성질과 점수의 관계를 일반화하기는 어렵다.
- 실행: 6-1을 실행한 뒤 실행한다. seed마다 val을 한 번씩 돌린다.

In [ ]:
IOU_COLS = {"mAP50": slice(0, 1), "mAP50-95": slice(0, 10), "mAP75-95": slice(5, 10)}  # 4-7과 같다. 4-7을 다시 실행하지 않아도 되게 다시 정한다

# 약의 모양과 색: 원본 JSON의 images[0] (1차 EDA 4-4와 같은 방법)
look = {}  # 약 번호 → (모양, 색)
for p in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8-sig"))
    img = d["images"][0]
    for a in d["annotations"]:
        look.setdefault(int(a["category_id"]), (img.get("drug_shape"), img.get("color_class1")))
look = {c: look[c] for c in CAT_IDS_ALL}  # 학습한 56종
group_n = Counter(v for v in look.values() if None not in v)  # (모양, 색) 묶음별 약 수. 정보가 없는 약은 묶지 않는다

# 클래스별 AP: seed마다 Ultralytics val
ap = {}  # 약 번호 → {"mAP50 s0": 값, ...}
ul_logger = ultralytics.utils.LOGGER
prev_level = ul_logger.level
ul_logger.setLevel("WARNING")  # 진행 표시를 끈다. 진행 표시에 내 컴퓨터의 절대경로가 찍힌다
try:
    for seed in SEEDS:
        run_dir = RUNS_DIR / make_run_name(seed)
        model = RTDETR(str(run_dir / "weights" / "best.pt"))
        metrics = model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                            plots=False, verbose=False, project=str(run_dir), name="eval_val", exist_ok=True)
        for k, iou_ap in zip(metrics.box.ap_class_index, metrics.box.all_ap):  # iou_ap: 그 클래스의 IoU 10단계 AP
            cat = int(yolo_names[int(k)])
            for metric, cols in IOU_COLS.items():
                ap.setdefault(cat, {})[f"{metric} s{seed}"] = float(iou_ap[cols].mean())
finally:
    ul_logger.setLevel(prev_level)  # 진행 표시를 되돌린다
ap = pd.DataFrame.from_dict(ap, orient="index")
assert set(ap.index) == set(val_gt["category_id"]), "Ultralytics val의 클래스와 6-1 정답의 클래스가 다릅니다."

check_ap = pd.DataFrame({metric: [ap[f"{metric} s{seed}"].mean() for seed in SEEDS] for metric in IOU_COLS},
                        index=pd.Index(SEEDS, name="seed")).round(4)
print(f"클래스 {len(ap)}종의 AP 평균 (4-7의 val 줄과 같아야 한다)")
display(check_ap)

gt_cnt = val_gt["category_id"].value_counts()
m50_cols = [f"mAP50 s{seed}" for seed in SEEDS]
m75_cols = [f"mAP75-95 s{seed}" for seed in SEEDS]
class_ap = pd.DataFrame(index=pd.Index(sorted(ap.index), name="약 번호"))
class_ap["약 이름"] = [CAT_NAME.get(c) for c in class_ap.index]
class_ap["val 정답"] = [int(gt_cnt[c]) for c in class_ap.index]
class_ap["train 라벨"] = [label_cnt["train"][c] for c in class_ap.index]
class_ap["모양"] = [look[c][0] for c in class_ap.index]
class_ap["색"] = [look[c][1] for c in class_ap.index]
class_ap["같은 모양·색 약"] = [group_n[look[c]] - 1 if None not in look[c] else None for c in class_ap.index]  # 자기 자신은 뺀다
class_ap = class_ap.join(ap[m50_cols + m75_cols].round(3))
class_ap["mAP75-95 평균"] = ap[m75_cols].mean(axis=1).round(3)
class_ap = class_ap.sort_values("mAP75-95 평균", kind="stable")
print("클래스별 AP와 약의 성질 (mAP75-95 세 seed 평균이 낮은 순)")
display(class_ap)

### 6-3. 박스 단위 에러 분류

- 왜 하는가: 6-2의 AP는 점수 순위 전체로 계산한 값이라, 어떤 박스가 왜 틀렸는지는 알 수 없다. 점수 0.5 이상인 예측을 정답과 하나씩 짝지어, 틀린 이유를 나눠 센다.
- 기준값
    - 점수 0.5 이상(`CONF_HI`): test 분석(5-4~5-6)과 같은 기준이라 val과 test를 나란히 비교할 수 있다.
    - IoU 0.75 이상이면 맞음(`IOU_HIT`): 대회 지표가 맞음으로 치기 시작하는 IoU다.
    - IoU 0.5 이상이면 같은 알약을 가리킨다고 본다(`IOU_SAME`, 5-4와 같은 기준).
- 짝짓는 순서 (seed마다, 사진마다)
    1. 같은 약 짝: 정답마다, 약 번호가 같고 IoU 0.5 이상인 예측 중 점수가 가장 높은 것과 짝짓는다. 한 사진에 같은 약은 한 알뿐이라(2-1), 예측 하나가 두 정답과 짝지어지는 일은 없다.
    2. 다른 약 짝: 1에서 짝이 없는 정답마다, 아직 짝이 없는 예측 중 IoU 0.5 이상이고 점수가 가장 높은 것과 짝짓는다.
    3. 남은 예측은 짝 없는 예측이다.
- 정답 박스의 분류 (seed마다 159개)

| 분류 | 뜻 |
|---|---|
| 맞음 | 같은 약 짝이 있고, IoU 0.75 이상 |
| 위치 부정확 | 같은 약 짝이 있지만, IoU 0.5 이상 0.75 미만 |
| 다른 약으로 착각 | 같은 약 짝이 없고, 그 알약에 다른 약 번호의 예측이 있다(IoU 0.5 이상) |
| 놓침 | IoU 0.5 이상인 예측이 하나도 없다 |

- 짝 없는 예측의 분류 (가장 많이 겹친 정답을 기준으로 나눈다)

| 분류 | 뜻 |
|---|---|
| 같은 알약에 겹침 (같은 약) | 이미 짝이 있는 정답과 IoU 0.5 이상이고, 약 번호도 같다. 한 알약에 같은 약 박스가 두 번 나온 것이다 |
| 같은 알약에 겹침 (다른 약) | 이미 짝이 있는 정답과 IoU 0.5 이상인데, 약 번호가 다르다. test 5-4에서 본 "다른 약끼리 겹친 박스"와 같은 현상이다 |
| 정답 없는 곳 | 모든 정답과 IoU 0.5 미만이다. 배경이거나, 알약을 조금만 덮은 박스다 |

- 함께 적는 값
    - IoU, 점수: 짝지은 예측의 IoU와 점수.
    - 가장 많이 겹친 정답 약: 짝 없는 예측과 IoU가 가장 큰 정답의 약 번호. IoU가 0이면(어떤 정답과도 겹치지 않으면) 비워 둔다.
    - 같은 약 짝의 IoU 최소: 맞음이어도 IoU가 0.95보다 낮으면, mAP75-95의 높은 IoU 단계(0.80~0.95 중 일부)에서는 틀린 것으로 쳐서 점수가 깎인다. 6-2에서 mAP50은 0.995인데 mAP75-95만 낮은 약을 보려고 클래스별 표에 함께 낸다.
    - 0.5 미만 점수의 같은 약 박스: 다른 약으로 착각했거나 놓친 정답에, 점수 0.5 미만(0.01 이상)으로는 같은 약 박스(IoU 0.5 이상)가 있었는지와 그 최고 점수. AP는 점수 순위로 계산하므로, 점수가 낮은 맞은 박스도 그 약 번호의 오답보다 점수가 높으면 AP를 거의 깎지 않는다(4번 결과의 20238 이야기).
- 출력
    1. seed별 개수: 정답 박스 분류 4개(합 159)와 짝 없는 예측 분류 3개. 점수 0.5 이상 예측 수가 "같은 약 짝 + 다른 약 짝 + 짝 없는 예측"과 같은지 확인한다.
    2. 맞음이 아닌 정답 박스 목록
    3. 짝 없는 예측 목록
    4. 클래스별 표: 틀린 것이 하나라도 있거나 6-2의 mAP75-95 평균이 0.995보다 낮은 약만 보여 준다. 값은 seed별로 "s0 / s1 / s2"로 적는다.
        - 정답 쪽: 그 약이 정답인 박스 중 위치 부정확, 다른 약으로 착각, 놓침의 수와 같은 약 짝의 IoU 최소
        - 예측 쪽: 그 약 번호로 낸 오답 예측 수(다른 약으로 착각한 예측과 짝 없는 예측). val에 없는 약 번호(train에만 있는 약)도 오답 예측으로 나오면 넣는다.
- 한계: 점수 0.5 기준이라 AP와는 다르다. 점수 0.5 미만으로 맞힌 박스는 놓침이나 다른 약으로 착각으로 센다. seed마다 정답 159개뿐이라 몇 개 차이는 우연일 수 있다.
- 결과 표 `gt_eval`(정답 박스별), `extra_eval`(짝 없는 예측별)은 7번에서 쓴다.
- 실행: 6-1, 6-2를 실행한 뒤 실행한다. 예측은 다시 하지 않는다.

In [ ]:
IOU_HIT = 0.75  # 이 이상이면 맞음 (대회 지표의 가장 낮은 IoU 기준). CONF_HI, IOU_SAME(0.5)은 5-4에서 정했다
GT_KINDS = ["맞음", "위치 부정확", "다른 약으로 착각", "놓침"]
EXTRA_KINDS = ["같은 알약에 겹침 (같은 약)", "같은 알약에 겹침 (다른 약)", "정답 없는 곳"]


def cross_iou(a, b):
    """a: (n, 4), b: (m, 4) [x, y, w, h] → (n, m) IoU 표. 5-4의 pair_iou를 다시 쓴다"""
    return pair_iou(np.vstack([a, b]))[:len(a), len(a):]


def best_low_score(pred_img, gt):
    """점수 0.01 이상 예측 중 정답과 약 번호가 같고 IoU 0.5 이상인 박스의 최고 점수. 없으면 None"""
    same = pred_img[pred_img["category_id"] == gt.category_id]
    if same.empty:
        return None
    iou = cross_iou(same[BOX_COLS].to_numpy(float), np.array([[gt.x, gt.y, gt.w, gt.h]]))[:, 0]
    scores = same["score"].to_numpy()[iou >= IOU_SAME]
    return float(scores.max()) if len(scores) else None


gt_rows, extra_rows = [], []
for seed in SEEDS:
    pred_seed = val_pred[val_pred["seed"] == seed]
    for file_name, g in val_gt.groupby("file_name"):
        pred_img = pred_seed[pred_seed["file_name"] == file_name]
        p = pred_img[pred_img["score"] >= CONF_HI].sort_values("score", ascending=False)  # 점수 높은 순
        iou = cross_iou(p[BOX_COLS].to_numpy(float), g[BOX_COLS].to_numpy(float))       # (예측 수, 정답 수)
        cat_p, score_p, cat_g = p["category_id"].to_numpy(), p["score"].to_numpy(), g["category_id"].to_numpy()
        used = np.zeros(len(p), dtype=bool)  # 짝이 정해진 예측
        pair, kind = [None] * len(g), [None] * len(g)

        for j in range(len(g)):  # 1) 같은 약 짝
            cand = np.flatnonzero((cat_p == cat_g[j]) & (iou[:, j] >= IOU_SAME) & ~used)
            if len(cand):
                i = cand[0]  # 점수 높은 순으로 정렬해 두었으므로 첫 번째가 점수가 가장 높다
                used[i], pair[j] = True, i
                kind[j] = "맞음" if iou[i, j] >= IOU_HIT else "위치 부정확"
        for j in range(len(g)):  # 2) 다른 약 짝
            if kind[j] is None:
                cand = np.flatnonzero((iou[:, j] >= IOU_SAME) & ~used)
                if len(cand):
                    i = cand[0]
                    used[i], pair[j] = True, i
                    kind[j] = "다른 약으로 착각"
                else:
                    kind[j] = "놓침"

        for j, gt in enumerate(g.itertuples()):
            i = pair[j]
            gt_rows.append({"seed": seed, "file_name": file_name, "정답 약": int(gt.category_id), "분류": kind[j],
                            "예측 약": None if i is None else int(cat_p[i]),
                            "IoU": None if i is None else float(iou[i, j]),
                            "점수": None if i is None else float(score_p[i]),
                            "0.5 미만 점수의 같은 약 박스": best_low_score(pred_img, gt) if kind[j] in ("다른 약으로 착각", "놓침") else None})
        for i in np.flatnonzero(~used):  # 3) 짝 없는 예측
            j = int(iou[i].argmax())  # 가장 많이 겹친 정답
            if iou[i, j] < IOU_SAME:
                extra_kind = "정답 없는 곳"
            else:
                extra_kind = "같은 알약에 겹침 (같은 약)" if cat_p[i] == cat_g[j] else "같은 알약에 겹침 (다른 약)"
            extra_rows.append({"seed": seed, "file_name": file_name, "예측 약": int(cat_p[i]), "점수": float(score_p[i]),
                               "분류": extra_kind, "가장 많이 겹친 정답 약": int(cat_g[j]) if iou[i, j] > 0 else None,
                               "IoU": float(iou[i, j])})

gt_eval = pd.DataFrame(gt_rows)
gt_eval["예측 약"] = gt_eval["예측 약"].astype("Int64")  # 놓침은 빈 값이라, 정수와 빈 값을 함께 담는 형식으로
extra_eval = pd.DataFrame(extra_rows, columns=["seed", "file_name", "예측 약", "점수", "분류", "가장 많이 겹친 정답 약", "IoU"])
extra_eval["가장 많이 겹친 정답 약"] = extra_eval["가장 많이 겹친 정답 약"].astype("Int64")  # 겹친 정답이 없으면 빈 값

# 1. seed별 개수
gt_count = pd.crosstab(gt_eval["seed"], gt_eval["분류"]).reindex(index=SEEDS, columns=GT_KINDS, fill_value=0)
extra_count = pd.crosstab(extra_eval["seed"], extra_eval["분류"]).reindex(index=SEEDS, columns=EXTRA_KINDS, fill_value=0)
n_hi = val_pred[val_pred["score"] >= CONF_HI].groupby("seed").size().reindex(SEEDS)
assert (gt_count.sum(axis=1) == len(val_gt)).all(), "정답 박스 수가 맞지 않습니다."
assert (gt_count[GT_KINDS[:3]].sum(axis=1) + extra_count.sum(axis=1) == n_hi).all(), "점수 0.5 이상 예측 수가 맞지 않습니다."
print(f"seed별 개수 (점수 {CONF_HI} 이상 예측, 맞음은 IoU {IOU_HIT} 이상)")
display(pd.concat({f"정답 박스 ({len(val_gt)}개)": gt_count, "짝 없는 예측": extra_count}, axis=1))

# 2. 맞음이 아닌 정답 박스
print("맞음이 아닌 정답 박스")
display(gt_eval[gt_eval["분류"] != "맞음"].set_index(["seed", "file_name"])
        .round({"IoU": 3, "점수": 3, "0.5 미만 점수의 같은 약 박스": 3}))

# 3. 짝 없는 예측
print("짝 없는 예측")
display(extra_eval.set_index(["seed", "file_name"]).round({"점수": 3, "IoU": 3}))

# 4. 클래스별 표
n_kind = gt_eval.groupby(["정답 약", "분류", "seed"]).size()
matched = gt_eval[gt_eval["분류"].isin(["맞음", "위치 부정확"])]
iou_min = matched.groupby(["정답 약", "seed"])["IoU"].min()
wrong_pred = pd.concat([gt_eval.loc[gt_eval["분류"] == "다른 약으로 착각", ["seed", "예측 약"]],
                        extra_eval[["seed", "예측 약"]]])
n_wrong_pred = wrong_pred.groupby([wrong_pred["예측 약"].astype(int), "seed"]).size()


def seeds_text(get):
    """seed별 값을 's0 / s1 / s2' 글자로"""
    return " / ".join(get(seed) for seed in SEEDS)


rows = []
for c in sorted(set(val_gt["category_id"]) | set(wrong_pred["예측 약"].astype(int))):
    row = {"약 번호": c, "약 이름": CAT_NAME.get(c), "val 정답": int((val_gt["category_id"] == c).sum()),
           "mAP75-95 평균 (6-2)": class_ap["mAP75-95 평균"].get(c)}
    for k in GT_KINDS[1:]:
        row[k] = seeds_text(lambda s: str(n_kind.get((c, k, s), 0)))
    row["같은 약 짝의 IoU 최소"] = seeds_text(lambda s: f"{iou_min[(c, s)]:.3f}" if (c, s) in iou_min.index else "-")
    row["이 약 번호의 오답 예측"] = seeds_text(lambda s: str(n_wrong_pred.get((c, s), 0)))
    row["틀린 것 합계 (세 seed)"] = (sum(n_kind.get((c, k, s), 0) for k in GT_KINDS[1:] for s in SEEDS)
                               + sum(n_wrong_pred.get((c, s), 0) for s in SEEDS))
    rows.append(row)
by_class = pd.DataFrame(rows).set_index("약 번호")
show = by_class[(by_class["틀린 것 합계 (세 seed)"] > 0) | (by_class["mAP75-95 평균 (6-2)"] < 0.995)]
show = show.sort_values(["mAP75-95 평균 (6-2)", "틀린 것 합계 (세 seed)"], ascending=[True, False], na_position="last")
print("클래스별 표 (값은 s0 / s1 / s2, 틀린 것이 있거나 mAP75-95 평균이 0.995보다 낮은 약)")
display(show)
print(f"이 표에 없는 {len(by_class) - len(show)}종은 세 seed 모두 정답 박스가 다 맞음이고, "
      f"그 약 번호의 오답 예측이 없고, mAP75-95 평균이 0.995다.")

### 6-4. PR 곡선 (IoU 0.75, 0.95)

- 왜 하는가: 6-3은 점수 0.5라는 기준 하나에서 센 것이다. AP는 점수 기준을 높은 쪽에서 낮은 쪽으로 내리면서 바뀌는 precision과 recall 전체로 계산한다. PR 곡선으로 그 전체를 보면, 6-3에서 추정한 "AP가 깎인 이유"를 확인할 수 있다.
- PR 곡선이란
    - 한 약 번호로 낸 예측을 점수가 높은 순으로 줄 세운다. 위에서부터 하나씩 받아들이면서, 그때까지의 recall(가로축)과 precision(세로축)을 점으로 찍어 잇는다.
    - precision(정밀도): 지금까지 받아들인 예측 중 맞은 것의 비율. 오답을 받아들일 때마다 떨어진다.
    - recall(재현율): 그 약의 정답 중 지금까지 찾은 것의 비율. 맞은 예측을 받아들일 때마다 오른다.
    - AP는 이 곡선 아래의 넓이다. 정확히는 곡선의 각 지점에서 그보다 오른쪽에 있는 가장 높은 precision으로 위를 메운 계단의 넓이다.
- 읽는 법
    - 곡선이 precision 1에서 수평으로 오른쪽 끝(recall 1)까지 가면 AP가 최고값(0.995)이다.
    - recall 1에 닿기 전에 아래로 떨어지면: 점수가 높은 오답이 맞은 박스보다 앞에 있다(precision 문제). 떨어진 높이와 그 뒤의 길이만큼 AP가 깎인다.
    - recall 1에 닿지 못하면: 그 IoU 기준을 넘는 박스가 하나도 없는 정답이 있다(recall 문제).
    - 오른쪽 끝에서 수직으로 떨어지는 부분은 맞은 박스를 모두 받아들인 뒤에 오는 오답들이다. AP에는 영향이 없다.
- IoU 기준
    - 0.75: 대회 지표에서 가장 낮은 기준이다. 6-3에서 약 번호를 틀린 30308, 20238, 22347을 보려고 고른다.
    - 0.95: 대회 지표에서 가장 높은 기준이다. 6-3에서 박스가 정답과 IoU 0.93~0.95로 아주 조금 어긋난 18147, 16262를 보려고 고른다. 6-2에서 박스 쪽 문제로 mAP75-95가 가장 낮았던 두 약이다.
    - 다섯 약 모두 두 기준으로 그려서, 기준을 높이면 곡선이 어떻게 바뀌는지 같이 본다.
- 계산 방법
    - 6-1의 val 예측(점수 0.01 이상)과 정답을 쓴다. 6-1에서 사진마다 예측 300개가 모두 점수 0.01 이상이었으므로, Ultralytics val(점수 0.001 이상)과 쓰는 예측이 같다.
    - 짝짓기: 점수가 높은 예측부터, 같은 사진의 같은 약 정답과 IoU가 기준 이상이고 그 정답이 아직 짝이 없으면 맞음, 아니면 오답으로 센다(COCO 방식, 6-3과 같다). 한 사진에 같은 약은 한 알뿐이다(2-1).
    - AP는 Ultralytics의 `compute_ap` 함수로 계산한다. 6-2와 같은 식이라 최고값도 0.995다.
    - Ultralytics val은 짝을 지을 때 점수가 아니라 IoU가 가장 큰 예측을 고른다. 그래서 여기서 계산한 AP는 Ultralytics 값과 조금 다를 수 있다.
- 기대값: 6-2에서 미리 계산할 수 있는 값이다. 계산이 맞는지 확인하는 기준으로 쓴다.
    - 30308: 6-2에서 seed마다 mAP50과 mAP75-95가 같았다. IoU 기준을 높여도 AP가 그대로라는 뜻이라, 두 기준 모두 0.995 / 0.924 / 0.538 근처여야 한다.
    - 20238, 22347: 6-2에서 mAP50과 mAP75-95가 모두 0.995였으므로, 두 기준 모두 0.995 근처여야 한다.
    - 18147, 16262: IoU 0.75는 0.995 근처여야 한다. 6-3에서 찾은 박스가 모두 IoU 0.93 이상이었으므로, IoU 0.75~0.90의 네 단계가 0.995라고 보면 IoU 0.95의 AP는 "5 × mAP75-95 − 4 × 0.995"로 어림할 수 있다. 18147은 약 0.05 / 0.60 / 0.72, 16262는 약 0.68 / 0.68 / 0.50이다.
- 그림: 위 줄은 IoU 0.75, 아래 줄은 IoU 0.95이고, 칸마다 약 하나다. seed는 색으로 구분하고, 곡선이 겹쳐도 보이도록 굵기를 다르게 한다(seed 0이 가장 굵다). 범례에 seed별 AP를 적는다.
- 표: 약과 IoU 기준마다 seed별 AP와 최대 recall을 낸다. 최대 recall은 점수 0.01 이상 예측을 모두 받아들였을 때의 recall이다.
- 한계: 약마다 val 정답이 3~7개라 곡선이 몇 계단뿐이고, 정답 하나가 recall 0.14~0.33을 차지한다.
- 실행: 6-1, 6-3을 실행한 뒤 실행한다(6-3의 `cross_iou`를 쓴다).

In [ ]:
PR_CATS = [30308, 20238, 22347, 18147, 16262]  # 약 번호를 틀린 3종, 박스가 조금 어긋난 2종 (6-3)
PR_IOUS = [0.75, 0.95]
SEED_COLORS = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a"}  # 4-6과 같은 색. 4-6을 다시 실행하지 않아도 되게 다시 정한다
SEED_LW = {0: 2.8, 1: 1.9, 2: 1.0}  # 곡선이 겹쳐도 보이도록 seed마다 굵기를 다르게


def pr_curve(seed, cat, iou_thr):
    """seed, 약 번호, IoU 기준 하나의 PR 곡선. 점수 높은 순으로 받아들이며 recall과 precision을 잰다."""
    gt = val_gt[val_gt["category_id"] == cat]
    gt_box = {r.file_name: np.array([[r.x, r.y, r.w, r.h]]) for r in gt.itertuples()}  # 사진 → 그 약의 정답 박스 (한 사진에 한 알)
    pred = val_pred[(val_pred["seed"] == seed) & (val_pred["category_id"] == cat)].sort_values("score", ascending=False)
    found, is_tp = set(), []
    for r in pred.itertuples():
        hit = (r.file_name in gt_box and r.file_name not in found
               and cross_iou(np.array([[r.x, r.y, r.w, r.h]]), gt_box[r.file_name])[0, 0] >= iou_thr)
        if hit:
            found.add(r.file_name)  # 이 정답은 짝이 생겼다
        is_tp.append(hit)
    n_tp = np.cumsum(is_tp)
    recall = n_tp / len(gt)
    precision = n_tp / np.arange(1, len(is_tp) + 1)
    return recall, precision


pr_rows = []
fig, axes = plt.subplots(len(PR_IOUS), len(PR_CATS), figsize=(18, 7.8), layout="constrained", sharex=True, sharey=True)
for row, iou_thr in enumerate(PR_IOUS):
    for col, cat in enumerate(PR_CATS):
        ax = axes[row, col]
        for seed in SEEDS:
            recall, precision = pr_curve(seed, cat, iou_thr)
            ap_value = float(ultralytics.utils.metrics.compute_ap(recall, precision)[0])  # 6-2와 같은 AP 계산식
            pr_rows.append({"약 번호": cat, "IoU": iou_thr, "seed": seed, "AP": ap_value, "최대 recall": float(recall[-1])})
            ax.plot(np.r_[0, recall], np.r_[1, precision], color=SEED_COLORS[seed], lw=SEED_LW[seed],
                    label=f"seed {seed} (AP {ap_value:.3f})")
        n_gt = int((val_gt["category_id"] == cat).sum())
        ax.set_title(f"{cat} {CAT_NAME.get(cat)}\nIoU {iou_thr}, 정답 {n_gt}개", fontsize=9)
        ax.set_xlim(0, 1.02)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7.5, loc="best")
        if row == len(PR_IOUS) - 1:
            ax.set_xlabel("recall")
        if col == 0:
            ax.set_ylabel(f"IoU {iou_thr}\nprecision")
fig.suptitle("val PR 곡선 (위: IoU 0.75, 아래: IoU 0.95 / 점수 0.01 이상 예측을 점수 높은 순으로)", fontsize=11)
plt.show()

pr_table = (pd.DataFrame(pr_rows).set_index(["약 번호", "IoU", "seed"]).unstack("seed")
            .reindex(pd.MultiIndex.from_product([PR_CATS, PR_IOUS], names=["약 번호", "IoU"])).round(3))
print("약과 IoU 기준별 AP와 최대 recall (점수 0.01 이상 예측을 모두 받아들였을 때)")
display(pr_table)

### 6번 결과

val 예측과 정답 (6-1)

- 정답은 159개다. val 47장 중 29장은 알약이 3개, 18장은 4개다.
- 세 seed 모두 예측이 14,100개(47장 × 300개)다. 사진마다 RT-DETR이 낼 수 있는 최대치인 300개가 모두 점수 0.01 이상으로 나왔다. test(5-2)와 같다.
- 점수 0.5 이상 예측은 165 / 163 / 169개(seed 0 / 1 / 2)로, 정답 159개보다 6 / 4 / 10개 많다.
- 사진 단위로 보면 47장 중 38 / 44 / 39장은 점수 0.5 이상 예측 수가 정답 수와 같고, 7 / 3 / 8장은 정답보다 많다. 정답보다 적은 사진은 seed 0의 2장뿐이다.
- val에서도 점수 0.5 이상 박스가 정답보다 많이 나오는 사진이 있다. 남는 박스가 같은 알약에 겹친 것인지, 정답이 없는 곳에 나온 것인지는 6-3에서 본다. 수가 같은 사진도 약 번호나 위치가 틀렸을 수 있다.

클래스별 AP (6-2)

- 30종의 AP 평균이 세 seed 모두 4-7의 val 줄과 소수 넷째 자리까지 같다. 클래스별 값을 제대로 꺼냈다.
- 22종은 세 seed 모두 mAP50과 mAP75-95가 최고값(0.995)이다. 0.995보다 낮은 seed가 있는 약은 아래 8종이다.

| 약 번호 | 약 이름 | val 정답 | train 라벨 | 같은 모양·색 약 | mAP50 (s0 / s1 / s2) | mAP75-95 (s0 / s1 / s2) |
|---|---|---|---|---|---|---|
| 30308 | 트라젠타듀오정 2.5/850mg | 6 | 3 | 2 | 0.995 / 0.924 / 0.538 | 0.995 / 0.924 / 0.538 |
| 18147 | 리리카캡슐 150mg | 5 | 10 | 6 | 0.995 / 0.995 / 0.995 | 0.806 / 0.915 / 0.939 |
| 16262 | 크레스토정 20mg | 5 | 18 | 4 | 0.995 / 0.995 / 0.995 | 0.931 / 0.931 / 0.895 |
| 41768 | 카발린캡슐 25mg | 4 | 12 | 0 | 0.995 / 0.995 / 0.995 | 0.921 / 0.932 / 0.995 |
| 21325 | 아토르바정 10mg | 6 | 16 | 3 | 0.995 / 0.995 / 0.995 | 0.995 / 0.885 / 0.995 |
| 25469 | 아모잘탄정 5/100mg | 9 | 3 | 2 | 0.995 / 0.995 / 0.995 | 0.971 / 0.948 / 0.961 |
| 13900 | 에빅사정(메만틴염산염)(비매품) | 4 | 11 | 5 | 0.995 / 0.995 / 0.995 | 0.945 / 0.995 / 0.995 |
| 3351 | 일양하이트린정 2mg | 29 | 128 | 1 | 0.995 / 0.995 / 0.995 | 0.995 / 0.988 / 0.986 |

- 낮은 8종은 두 가지로 나뉜다.
    - 30308: seed마다 mAP50과 mAP75-95가 같다. IoU 0.5에서 이미 점수가 깎이고, IoU 기준을 0.95까지 높여도 더 깎이지 않는다. 그래서 찾은 박스의 위치는 정확하고, 점수가 깎인 이유는 놓쳤거나 다른 약으로 예측했거나 점수가 높은 오답이 있는 것으로 보인다. 4번 결과에서 숫자만 보고 한 추정과 같다. 어느 쪽인지는 6-3, 6-4에서 본다.
    - 나머지 7종: 세 seed 모두 mAP50은 0.995인데 mAP75-95만 낮다. 약을 찾고 약 번호도 맞혔지만, 높은 IoU 기준에서는 박스가 정답과 덜 맞는 것으로 보인다. 4번 결과에서 16262, 18147, 25469에 대해 한 추정과 같고, 41768, 21325, 13900, 3351이 더 나왔다. 4번 결과에서 본 mAP50-95보다 mAP75-95에서 차이가 더 크게 드러난다.
- 4번 결과에서 recall이 낮았던 20238(0.23 / 0.66 / 0.38)과 precision이 낮았던 22347(0.41 / 0.74 / 0.50)은 세 seed 모두 mAP50과 mAP75-95가 0.995다. Ultralytics의 precision과 recall은 신뢰도 기준값 하나에서 잰 값이고, AP는 점수 순위 전체로 잰 값이라 다르게 나올 수 있다. 점수 0.5 기준으로 이 두 약의 박스가 어떻게 나왔는지는 6-3에서 본다.

약의 성질과 점수 (6-2 표)

- train 라벨 수: 낮은 8종의 train 라벨은 3~128개로 퍼져 있다. val에 있는 약 중 train 라벨이 3개뿐인 약은 4종(30308, 25469, 19861, 28763)이다. 그중 30308과 25469는 낮은 쪽에 있고, 19861과 28763은 세 seed 모두 0.995다. 라벨이 적을수록 점수가 낮다고 말하기에는 예외가 있고, 약의 수도 적다.
- 같은 모양·색 약 수: 낮은 8종은 0~6개이고, 0.995인 22종도 0~6개다. 이 표에서는 닮은 약이 많을수록 점수가 낮다는 경향이 보이지 않는다.
- 모양: 박스가 덜 맞는 7종 중 18147(리리카캡슐)과 41768(카발린캡슐)은 이름이 캡슐이고 모양이 장방형이다. val에 있는 다른 캡슐 4종(16688, 18357, 19232, 32310, 모두 연질캡슐)은 0.995다. 두 약만으로는 캡슐이라서 박스가 덜 맞는다고 할 수 없다. 사진은 7번에서 본다.
- val 정답 수: 낮은 약 중 30308(6개), 18147(5개), 16262(5개), 41768(4개), 13900(4개)은 정답이 4~6개뿐이라, 박스 하나에 따라 AP가 크게 바뀔 수 있다. 정답이 29개인 3351은 0.986~0.995로 작게 흔들린다.

박스 단위 에러 분류 (6-3)

| seed | 맞음 | 위치 부정확 | 다른 약으로 착각 | 놓침 | 겹침 (같은 약) | 겹침 (다른 약) | 정답 없는 곳 |
|---|---|---|---|---|---|---|---|
| 0 | 152 | 0 | 4 | 3 | 1 | 3 | 5 |
| 1 | 155 | 0 | 4 | 0 | 1 | 3 | 0 |
| 2 | 153 | 0 | 6 | 0 | 2 | 2 | 6 |

- 앞의 넷은 정답 박스(seed마다 159개)의 분류이고, 뒤의 셋은 짝 없는 예측의 분류다. 정답 박스와 짝지은 예측과 짝 없는 예측을 더하면 점수 0.5 이상 예측 수(165 / 163 / 169)와 같다.

박스 위치

- 위치 부정확은 세 seed 모두 0개다. 같은 약으로 찾은 박스는 모두 IoU 0.75 이상이다. 클래스별 표에 나온 val의 약 13종은 같은 약 짝의 IoU 최소가 0.932~0.976이다.
- 6-2에서 mAP50은 0.995인데 mAP75-95만 낮았던 7종을 보면, mAP75-95가 낮았던 seed에서는 같은 약 짝의 IoU 최소가 모두 0.95보다 낮다(0.932~0.949). 반대로 클래스별 표의 나머지 약과 seed는 IoU 최소가 0.95 이상이고 mAP75-95가 0.995다. 다른 약으로 착각한 30308의 seed 1, 2만 예외다.
- 이 7종은 찾은 박스의 IoU가 모두 0.93 이상이라, IoU 0.75~0.90의 네 단계에서는 맞음이고 0.95 단계에서만 틀릴 수 있다. 그래서 이 7종의 mAP75-95가 낮은 것은 가장 엄격한 0.95 단계에서 깎인 것으로 보인다.
    - 계산해 보면, 네 단계가 0.995이고 0.95 단계가 0이어도 다섯 단계의 평균은 0.796이다. 18147의 seed 0(0.806)은 이 값에 가까워서, 0.95 단계에서 거의 모든 박스가 틀린 것으로 보인다.
    - 단, 이 표는 점수 0.5 이상인 박스로 센 것이고, AP는 점수가 낮은 박스까지 포함해 계산한다.
- 박스 위치 문제는 크게 어긋난 것이 아니다. 정답과 IoU 0.93~0.95 정도로 아주 조금 어긋난 것이다. 대회 지표는 IoU 0.95까지 보기 때문에 이 작은 차이도 점수에 들어간다.

약을 틀린 정답 박스 (다른 약으로 착각, 놓침)

- 맞음이 아닌 정답 박스는 세 seed를 합해 17개(7 / 4 / 6)이고, 20238과 30308 두 약뿐이다.
- 17개 중 16개가 한 조합(K-003483-019861-020238-030308, 이하 조합 A)의 사진 3장(촬영 각도 70, 75, 90)에서 나왔다. 나머지 1개는 seed 0의 K-003483-020238-027733-028763 각도 75 사진이다.
    - 같은 조합의 사진은 같은 알약을 각도만 바꿔 찍은 것이다. 그래서 17개는 서로 독립된 실수라기보다, 거의 한 장면에서 반복된 실수다.

| 정답 약 | seed 0 | seed 1 | seed 2 |
|---|---|---|---|
| 20238 플라빅스정 75mg | 22347로 착각 4개 | 22347로 착각 1개 | 22347로 착각 3개 |
| 30308 트라젠타듀오정 2.5/850mg | 놓침 3개 | 27926으로 착각 3개 | 27926으로 착각 3개 |

- 착각하거나 놓친 17개 모두, 점수 0.5 미만으로는 같은 약 박스(IoU 0.5 이상)가 있었다. 모델이 이 알약들을 맞는 약 번호로도 냈지만 점수가 낮았다는 뜻이다.
    - 20238: 0.124~0.360
    - 30308: seed 0은 0.219~0.286, seed 1은 0.359~0.393, seed 2는 0.045~0.083
- 20238과 22347 (22347은 자누비아정 50mg)
    - 점수 0.5 이상으로 22347을 낸 오답 박스(6 / 3 / 3개)는 모두 20238 알약 위에 나왔다. 착각이 4 / 1 / 3개이고, 20238을 맞게 찾은 알약 위에 겹친 것이 2 / 2 / 0개다. 반대로 22347 알약을 20238로 본 박스는 없다.
    - 점수 0.5 기준으로 세면, 20238은 정답 7개 중 3 / 6 / 4개만 같은 약으로 찾았다. 22347로 낸 박스는 9 / 6 / 6개 중 3개만 맞았다.
    - 그런데도 6-2에서 두 약의 AP는 0.995다. Ultralytics에서 AP가 0.995라는 것은, 모든 정답에 맞은 박스가 있고, 그 약 번호로 낸 박스를 점수 순으로 세웠을 때 맞은 박스가 모두 오답보다 앞에 있다는 뜻이다.
        - 20238은 0.5 미만으로 맞힌 박스보다 점수가 높은 20238 오답이 없었다.
        - 22347은 오답 박스보다 진짜 22347 알약의 박스 점수가 더 높았다.
    - 4번 결과의 추정(20238은 낮은 점수로 맞혔다, 22347은 다른 약을 22347로 봤을 수 있다)과 맞는다. 5-6에서 본 test 945번 사진(20238 알약에 22347 0.71, 20238 0.53)과 같은 현상이다.
    - JSON의 외형 정보로는 둘 다 원형이고 장축이 8.50mm, 8.02mm로 비슷하다. 색은 분홍과 주황으로 적혀 있다. 사진에서 어떻게 보이는지는 7번에서 본다.
- 30308과 27926 (27926은 울트라셋이알서방정)
    - 27926은 train에만 있는 약이다(train 라벨 3개, 5-5). seed 1, 2는 조합 A의 30308 알약 3개를 모두 27926으로 봤다(점수 0.661~0.828). seed 0은 이 알약들에 점수 0.5 이상 박스를 하나도 내지 않았다.
    - JSON의 외형 정보로는 둘 다 타원형이고 크기가 같다(장축 19.0mm, 단축 9.0mm). 색은 주황과 노랑으로 적혀 있다.
    - 30308의 AP는 0.995 / 0.924 / 0.538이다. seed 2는 조합 A에서 맞은 박스의 점수가 0.045~0.083으로 아주 낮다. 그래서 점수 순으로 세웠을 때 그보다 점수가 높은 30308 오답(점수 0.5 미만 포함)이 많아, 정답 6개 중 3개를 찾은 뒤부터 precision이 크게 떨어진 것으로 보인다. 6-4의 PR 곡선으로 확인한다.
    - 5번 결과에서 "seed 2가 30308을 29345와 헷갈리는 것으로 보인다"고 추정했다. val에서 seed 2가 30308 알약을 주로 본 약 번호는 27926이고, 29345는 조합 A의 각도 70 사진에서 한 번(0.586) 겹쳐 나왔다.
    - test 5-4에서 seed 1에 14번 나온 16551 - 27926 쌍이, val에서는 조합 A의 각도 75 사진에서 30308 알약 위에 한 번 나왔다(seed 1, 27926 0.661, 16551 0.543). test의 16551 - 27926 쌍이 실제로 어떤 알약인지는 test에 정답이 없어 알 수 없다.

짝 없는 예측

- 같은 알약에 겹침 (다른 약): 3 / 3 / 2개다. 그중 4개가 20238 알약 위의 22347이다. 나머지는 16232 위의 27993, 30308 위의 16551과 29345, 20238 위의 28763이다.
    - test 5-4에서는 사진 842장에 다른 약끼리 겹친 쌍이 111 / 102 / 193개(사진 100장당 약 12~23개)였다. val은 47장에 3 / 3 / 2개(사진 100장당 약 4~6개)다.
    - 5-4는 예측끼리 겹친 것을 세고, 6-3은 정답을 기준으로 세서 기준이 정확히 같지는 않다. 그래도 val에서 훨씬 적게 나온다.
- 같은 알약에 겹침 (같은 약): 1 / 1 / 2개다. 한 알약에 같은 약 번호 박스를 두 번 낸 것이다(19861 두 번, 18147, 30308).
- 정답 없는 곳: 5 / 0 / 6개다.
    - 11개 모두 IoU가 0이라, 라벨이 있는 어떤 알약과도 전혀 겹치지 않는다.
    - 약 번호는 seed 0이 18147, 41768이고, seed 2가 35206, 31885, 41768이다. 점수는 0.513~0.764다.
    - 사진 7장에서 나왔고, 모두 3351이 들어 있는 조합이다. 그중 2장(K-003351-016232-032310 각도 90, K-003351-029667-032310 각도 90)은 seed 0과 2에서 모두 나왔다.
    - 배경이나 그림자, 반사일 수도 있고, 라벨이 빠진 알약일 수도 있다. 이 표만으로는 알 수 없어서 7번에서 사진으로 본다.
    - 6-2에서 이 약들의 mAP50은 0.995라서, 이 오답들은 같은 약 번호의 맞은 박스보다 점수가 낮아 val AP를 깎지 않았다.

6-3 정리

- val에서 점수를 깎는 것은 두 가지로 보인다.
    - 박스가 아주 조금 어긋남: 같은 약으로 찾은 박스가 정답과 IoU 0.93~0.95라 IoU 0.95 단계에서 틀린다(7종).
    - 약 번호를 틀림: 조합 A에서 20238과 30308을 다른 약 번호로 높게 보고, 맞는 약 번호는 낮은 점수로만 냈다. AP가 크게 깎인 것은 30308의 seed 1, 2뿐이다.
- 크게 틀린 것이 거의 한 조합에 몰려 있어서, 47장의 val로는 이런 실수가 얼마나 자주 일어나는지 말하기 어렵다.

PR 곡선 (6-4)

- 계산한 AP가 6-4 설명에 적은 기대값과 모두 맞았다. 짝짓는 방법이 Ultralytics와 달라도(점수 순과 IoU 순), 이 다섯 약에서는 같은 값이 나왔다.

| 약 번호 | IoU | AP (s0 / s1 / s2) | 최대 recall (s0 / s1 / s2) | 기대값 |
|---|---|---|---|---|
| 30308 | 0.75, 0.95 | 0.995 / 0.924 / 0.538 | 1.0 / 1.0 / 1.0 | 0.995 / 0.924 / 0.538 |
| 20238 | 0.75, 0.95 | 0.995 / 0.995 / 0.995 | 1.0 / 1.0 / 1.0 | 0.995 |
| 22347 | 0.75, 0.95 | 0.995 / 0.995 / 0.995 | 1.0 / 1.0 / 1.0 | 0.995 |
| 18147 | 0.75 | 0.995 / 0.995 / 0.995 | 1.0 / 1.0 / 1.0 | 0.995 |
| 18147 | 0.95 | 0.052 / 0.595 / 0.715 | 0.6 / 0.6 / 0.8 | 약 0.05 / 0.60 / 0.72 |
| 16262 | 0.75 | 0.995 / 0.995 / 0.995 | 1.0 / 1.0 / 1.0 | 0.995 |
| 16262 | 0.95 | 0.675 / 0.675 / 0.495 | 0.8 / 0.8 / 0.6 | 약 0.68 / 0.68 / 0.50 |

약 번호를 틀린 약 (30308, 20238, 22347)

- 세 약 모두 IoU 0.75와 0.95의 곡선이 같다. 맞은 박스는 IoU 0.95 기준도 넘는다는 뜻이다. 이 약들의 문제는 박스 위치가 아니라 점수 순서다.
- 30308
    - 세 seed 모두 최대 recall이 1이다. 점수 0.01 이상으로는 정답 6개를 모두 맞는 약 번호로 찾았다. 못 찾은 것(recall 문제)이 아니다.
    - seed 0: precision 1로 recall 1까지 간다. 조합 A에서 맞은 박스의 점수가 0.22~0.29로 낮았지만, 그보다 점수가 높은 30308 오답이 없었다.
    - seed 1: recall 0.5(정답 3개)까지 precision 1이다가, 네 번째 정답 앞에 오답이 하나 끼어 0.75로 떨어진다. 그 뒤로 정답 3개를 이어 받아 0.86까지 오른다. 6-3에서 seed 1은 점수 0.5 이상의 30308 오답이 없었으므로, 이 오답의 점수는 0.393(조합 A에서 맞은 박스의 최고 점수)과 0.5 사이다.
    - seed 2: recall 0.5까지 precision 1이다가 0.1 아래로 떨어진다. 남은 정답 3개(조합 A, 점수 0.045~0.083)를 받기 전에 30308 오답을 수십 개 먼저 받아들였다는 뜻이다. 6-3에서 추정한 그대로다. AP 0.538은 "정답 절반은 precision 1, 나머지 절반은 precision 0.1 아래"에서 나온 값이다.
    - 즉 30308의 AP가 깎인 것은 놓쳐서가 아니라, 맞는 약 번호의 점수가 낮아 그보다 점수가 높은 오답 뒤로 밀렸기 때문이다. 4번 결과에서 "놓쳤거나 다른 약으로 예측한 것으로 보인다"고 추정했는데, 정확히는 다른 약 번호(27926)를 높은 점수로 내고 맞는 약 번호는 낮은 점수로 낸 것이다.
- 20238, 22347: 세 seed와 두 기준 모두 precision 1로 recall 1까지 간다. 6-3에서 점수 0.5 기준으로는 20238을 22347로 많이 봤지만, 점수 순서로는 맞은 박스가 모두 오답보다 앞에 있다. 6-3의 설명과 맞는다.

박스가 조금 어긋난 약 (18147, 16262)

- IoU 0.75에서는 세 seed 모두 AP 0.995, 최대 recall 1이다.
- IoU 0.95에서는 최대 recall이 1에 닿지 못한다. 점수가 낮은 박스까지 다 봐도, 사진마다 300개인 예측 중 어느 박스도 정답과 IoU 0.95를 넘지 못한 정답이 있다. 18147은 정답 5개 중 2 / 2 / 1개, 16262는 1 / 1 / 2개다.
- precision 쪽 손실도 있다. 점수가 높은 박스가 IoU 0.95를 넘지 못하고, 점수가 낮은 박스가 넘은 경우다.
    - 18147 seed 0은 첫 예측부터 오답이고, 맞은 박스는 다섯 번째에야 나온다(recall 0.2에서 precision 0.2). 나머지 맞은 박스는 훨씬 뒤에 나와서 AP가 0.052까지 떨어진다.
    - 18147 seed 2는 recall 0.4 뒤에 오답이 하나 끼어 precision이 0.67로 떨어진다.
    - 16262는 세 seed 모두 두 번째 예측이 오답이라 recall 0.2에서 precision이 0.5로 떨어진다.
- 이 두 약은 알약을 찾고 약 번호도 맞히지만, 정답 박스와 IoU 0.95 이상으로 맞는 박스를 내지 못하는 알약이 있다. 6-3에서 같은 약 짝의 IoU 최소가 0.934~0.945였던 것과 맞는다.
- 예측 박스가 조금 어긋난 것인지, 정답 라벨 박스가 알약보다 조금 크거나 작게 그려진 것인지는 숫자로는 구분할 수 없다. 7번에서 사진에 정답과 예측을 함께 그려 본다.

6번 정리

- val 30종 중 22종은 세 seed 모두 mAP75-95가 최고값(0.995)이다. 남은 점수 손실은 두 가지다.
    - 박스가 정답과 IoU 0.93~0.95로 아주 조금 어긋남(7종): IoU 0.95 단계에서만 깎인다. 18147과 16262에는 IoU 0.95를 넘는 박스가 아예 없는 알약이 있다.
    - 맞는 약 번호의 점수가 낮음(30308, 20238): 거의 조합 A 한 장면에서, 다른 약 번호(27926, 22347)를 높은 점수로 내고 맞는 약 번호는 낮은 점수로 냈다. AP가 깎인 것은 그보다 점수가 높은 오답이 있었던 30308의 seed 1, 2뿐이다.
- 4번, 5번에서 숫자로 한 추정과 비교
    - 맞았던 것: 16262, 18147, 25469는 박스 위치 쪽 문제다(4번). 20238은 낮은 점수로 맞혔다(4번). 22347은 다른 약을 22347로 봤다(4번, 모두 20238 알약).
    - 더 정확해진 것: seed 2의 30308은 놓친 것이 아니라, 다른 약 번호(27926)를 높게 내고 맞는 약 번호를 아주 낮은 점수로 낸 것이다(4번).
    - 달랐던 것: "seed 2가 30308을 29345와 헷갈린다"(5번)는, val에서는 주로 27926이었고 29345는 한 번 나왔다.
- test와의 관계: val의 mAP75-95는 0.973~0.982라, val에서 남은 손실은 0.02 안팎이다. val과 test 점수의 차이(약 0.56)는 val에서 본 이 문제들만으로는 설명되지 않는다. 5번에서 본 배우지 않은 약의 영향은 val로는 확인할 수 없다.
- 7번에서 사진으로 볼 것: 조합 A의 사진 3장(20238, 30308), 18147과 16262에서 IoU 0.95를 넘지 못한 박스, "정답 없는 곳" 예측이 나온 사진 7장.

## 7. 예측 보기

- 왜 하는가: 6번에서 숫자로 찾은 문제를 val 사진에 정답과 예측을 함께 그려서 직접 본다. 숫자만으로는 왜 틀렸는지 알 수 없기 때문이다. 예를 들어 닮은 약이라서인지, 박스가 어긋난 것인지 라벨이 어긋난 것인지, 정답이 없는 곳에 무엇이 있는지는 사진을 봐야 안다.
- 그림만 그리고, 판단은 그림을 보고 사람이 한다. 판단한 내용은 7번 결과에 적는다.
- 실행: 커널을 다시 시작했다면 0번 두 셀, 3-1, 5-1, 5-4, 5-5, 5-6, 6-1, 6-2, 6-3을 먼저 실행한다. 7번은 5-6의 train 예시 함수(`draw_train`)와 6-1, 6-3의 표(`val_gt`, `val_pred`, `gt_eval`)를 쓴다.

| 단계 | 볼 것 |
|---|---|
| 7-1 | 약 번호를 틀린 장면: 조합 A의 사진 3장 |
| 7-1 추가 | 30308의 앞면과 뒷면: train 박스 3개와 val 박스 6개 |
| 7-2 | 박스가 조금 어긋난 약: 18147, 16262의 val 정답 10개 |
| 7-3 | 정답 없는 곳: 사진 7장 |

### 7-1. 약 번호를 틀린 장면 (조합 A)

- 볼 것: 6-3에서 맞음이 아닌 정답 박스 17개 중 16개가 나온 조합 A(K-003483-019861-020238-030308)의 사진 3장(각도 70, 75, 90)이다. 이 조합에는 3483, 19861, 20238, 30308 네 알약이 있다. seed 0에만 있던 다른 조합(K-003483-020238-027733-028763)의 착각 1개는 넣지 않는다.
- 그림 읽는 법
    - 첫 그림: 줄은 seed(0, 1, 2), 칸은 사진(각도 70, 75, 90)이다. 알약이 잘 보이도록 정답과 예측이 있는 부분만 잘라서 크게 보여 준다. 같은 사진은 세 seed에서 같은 범위로 자른다.
    - 초록 점선: 정답 박스다. 박스 아래에 "정답 약 번호"를 적는다. 다른 약으로 착각했거나 놓친 정답에는, 점수 0.5 미만으로 낸 같은 약 박스의 최고 점수(6-3)를 괄호에 적는다.
    - 파랑 실선: 점수 0.5 이상 예측 중 같은 약으로 맞은 박스(6-3의 맞음)다.
    - 빨강 실선: 점수 0.5 이상 예측 중 나머지(다른 약으로 착각, 같은 알약에 겹침)다.
    - 예측 박스 위에 "약 번호 점수"를 적는다. 한 알약에 박스가 여러 개면 라벨을 위로 쌓는다. 정답 박스와 예측 박스가 거의 겹치므로, 정답은 점선으로 그린다.
    - 둘째 그림: 20238(플라빅스정 75mg), 22347(자누비아정 50mg), 30308(트라젠타듀오정 2.5/850mg), 27926(울트라셋이알서방정)의 train 예시다. 5-6과 같이 약마다 첫 번째 train 라벨 박스를 잘라 낸 것이다.
- 판단 기준 (그림을 보고 직접 판단한다)
    - 20238 알약이 22347 train 예시와, 30308 알약이 27926 train 예시와 닮아 보이는지
    - 조합 A 사진에 알약끼리 붙어 있거나 가려진 곳, 조명이나 초점 문제가 있는지
    - 같은 알약인데 seed마다 예측이 어떻게 다른지
- 한계: train 예시는 약마다 한 장뿐이고, 사진마다 각도와 조명이 달라 같은 약도 달라 보일 수 있다.

In [ ]:
COMBO_A = "K-003483-019861-020238-030308"  # 6-3에서 틀린 정답 박스가 몰린 조합
TRAIN_LOOK = [20238, 22347, 30308, 27926]   # 틀린 정답 약과, 대신 낸 약 번호 (6-3)
CROP_MARGIN = 80                             # 잘라 낼 때 알약 주변으로 남길 여백(픽셀)
GT_COLOR, OK_COLOR, NG_COLOR = "limegreen", "deepskyblue", "red"

combo_files = sorted(f for f in val_gt["file_name"].unique() if f.startswith(COMBO_A + "_"))
assert len(combo_files) == 3, f"조합 A 사진이 3장이 아닙니다: {len(combo_files)}"
print("조합 A 사진:", combo_files)

# 같은 약으로 맞은 예측 (6-3의 맞음, 위치 부정확): (seed, 사진, 약 번호, 점수)
matched_rows = gt_eval[gt_eval["분류"].isin(["맞음", "위치 부정확"])]
ok_pred = set(zip(matched_rows["seed"], matched_rows["file_name"], matched_rows["예측 약"].astype(int), matched_rows["점수"]))


def crop_range(file_name):
    """정답과 점수 0.5 이상 예측(세 seed 모두)이 다 들어가는 범위 + 여백 → (x0, y0, x1, y1) 픽셀"""
    b = pd.concat([val_gt.loc[val_gt["file_name"] == file_name, BOX_COLS],
                   val_pred.loc[(val_pred["file_name"] == file_name) & (val_pred["score"] >= CONF_HI), BOX_COLS]])
    with Image.open(VAL_IMG_DIR / file_name) as im:
        W, H = im.size
    x0, y0 = max(0, b["x"].min() - CROP_MARGIN), max(0, b["y"].min() - CROP_MARGIN)
    x1, y1 = min(W, (b["x"] + b["w"]).max() + CROP_MARGIN), min(H, (b["y"] + b["h"]).max() + CROP_MARGIN)
    return int(x0), int(y0), int(np.ceil(x1)), int(np.ceil(y1))


def draw_val(ax, seed, file_name, crop):
    """val 사진의 crop 범위에 정답(초록 점선)과 점수 0.5 이상 예측(맞음 파랑, 나머지 빨강)을 그린다."""
    x0, y0, x1, y1 = crop
    with Image.open(VAL_IMG_DIR / file_name) as im:
        ax.imshow(np.asarray(im.convert("RGB").crop(crop)), extent=(x0, x1, y1, y0))  # 원본 픽셀 좌표 그대로 그린다

    ev = gt_eval[(gt_eval["seed"] == seed) & (gt_eval["file_name"] == file_name)].set_index("정답 약")
    for r in val_gt[val_gt["file_name"] == file_name].itertuples():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, edgecolor=GT_COLOR, lw=2, ls="--"))
        text = f"정답 {r.category_id}"
        low = ev.loc[r.category_id, "0.5 미만 점수의 같은 약 박스"]
        if pd.notna(low):
            text += f" (같은 약 {low:.2f})"
        ax.annotate(text, (r.x, r.y + r.h), xytext=(0, -3), textcoords="offset points", va="top", fontsize=8,
                    color="black", bbox=dict(facecolor=GT_COLOR, edgecolor="none", pad=1))

    p = val_pred[(val_pred["seed"] == seed) & (val_pred["file_name"] == file_name) & (val_pred["score"] >= CONF_HI)]
    placed = []  # 이미 라벨을 적은 박스
    for r in p.sort_values("score", ascending=False).itertuples():
        color = OK_COLOR if (seed, file_name, r.category_id, r.score) in ok_pred else NG_COLOR
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, edgecolor=color, lw=1.8))
        box = np.array([[r.x, r.y, r.w, r.h]])
        k = sum(cross_iou(box, b)[0, 0] >= IOU_SAME for b in placed)  # 같은 알약에 이미 적은 라벨 수만큼 위로 올린다
        ax.annotate(f"{r.category_id} {r.score:.2f}", (r.x, r.y), xytext=(0, 3 + 12 * k), textcoords="offset points",
                    va="bottom", fontsize=8, color="white", bbox=dict(facecolor=color, edgecolor="none", pad=1))
        placed.append(box)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y1, y0)
    ax.axis("off")


crops = {f: crop_range(f) for f in combo_files}
fig, axes = plt.subplots(len(SEEDS), len(combo_files), figsize=(15, 14), layout="constrained")
for i, seed in enumerate(SEEDS):
    for j, f in enumerate(combo_files):
        draw_val(axes[i, j], seed, f, crops[f])
        axes[i, j].set_title(f"seed {seed} / 각도 {f.split('_')[5]}", fontsize=10)
fig.suptitle(f"조합 A val 사진 (초록 점선: 정답, 파랑: 맞은 예측, 빨강: 나머지 예측, 점수 {CONF_HI} 이상)", fontsize=12)
plt.show()

fig, axes = plt.subplots(1, len(TRAIN_LOOK), figsize=(3 * len(TRAIN_LOOK), 3.8), layout="constrained")
for ax, cat in zip(axes, TRAIN_LOOK):
    draw_train(ax, cat)  # 5-6의 함수
fig.suptitle("train 예시 (약마다 첫 번째 train 라벨 박스를 잘라 낸 것)", fontsize=11)
plt.show()

#### 7-1 추가. 30308의 앞면과 뒷면

- 왜 보는가: 7-1에서 조합 A의 30308은 "D2/850" 글자가 있는 뒷면이, train 예시는 문양이 있는 앞면이 찍혀 있었다. 30308은 train 라벨이 3개뿐이라, 모델이 학습 때 뒷면을 본 적이 있는지 확인한다. 뒷면을 본 적이 없다면, 조합 A에서 맞는 약 번호의 점수가 낮았던 이유가 될 수 있다.
- 볼 것: 30308의 train 라벨 박스 3개 전부(위 줄)와 val 정답 박스 6개 전부(아래 줄)를 잘라 나란히 놓는다.
- 칸 제목: train인지 val인지, 사진(조합과 각도)을 적는다. val 칸에는 seed 0 / 1 / 2마다 그 알약에 낸 30308 박스(IoU 0.5 이상, 점수 0.01 이상)의 최고 점수를 적는다. 6-3의 `best_low_score`로 계산한다.
- 판단 기준 (그림을 보고 직접 판단한다)
    - train 3개가 모두 앞면(문양)이면, 모델은 학습 때 30308의 뒷면을 본 적이 없다.
    - val에서 점수가 높은 알약(조합 A가 아닌 3개)이 앞면이고 조합 A만 뒷면이면, "뒷면이라서 맞는 약 번호의 점수가 낮았다"는 설명과 맞는다.
    - train에 뒷면이 있거나, 점수가 높은 val 알약도 뒷면이면 이 설명은 맞지 않는다.
- 한계: 30308 한 약만 본다. 다른 약도 한쪽 면만 학습했는지는 이것으로 알 수 없다.
- 실행: 7-1을 실행한 뒤 실행한다.

In [ ]:
FACE_CAT = 30308   # 면을 확인할 약
FACE_MARGIN = 10   # 박스 주변 여백(픽셀)

face_items = []  # 잘라 볼 박스: 구분(train/val), 사진 경로, 박스 [x, y, w, h], seed별 30308 점수
for p in sorted((YOLO_DIR / "labels" / "train").glob("*.txt")):
    for line in p.read_text().splitlines():
        k, cx, cy, w, h = line.split()
        if int(yolo_names[int(k)]) != FACE_CAT:
            continue
        img_path = YOLO_DIR / "images" / "train" / f"{p.stem}.png"
        with Image.open(img_path) as im:
            W, H = im.size
        bw, bh = float(w) * W, float(h) * H
        face_items.append({"구분": "train", "path": img_path, "box": (float(cx) * W - bw / 2, float(cy) * H - bh / 2, bw, bh),
                           "scores": None})
for gt in val_gt[val_gt["category_id"] == FACE_CAT].sort_values("file_name").itertuples():
    scores = [best_low_score(val_pred[(val_pred["seed"] == s) & (val_pred["file_name"] == gt.file_name)], gt) for s in SEEDS]
    face_items.append({"구분": "val", "path": VAL_IMG_DIR / gt.file_name, "box": (gt.x, gt.y, gt.w, gt.h), "scores": scores})
by_split = {s: [it for it in face_items if it["구분"] == s] for s in ["train", "val"]}
assert len(by_split["train"]) == label_cnt["train"][FACE_CAT] and len(by_split["val"]) == label_cnt["val"][FACE_CAT], "30308 박스 수가 5-5와 다릅니다."


def photo_name(path):
    """사진 이름 → '조합 각도 N' (조합 A면 표시)"""
    combo, angle = path.name.split("_")[0], path.name.split("_")[5]
    return f"{combo} 각도 {angle}" + (" (조합 A)" if combo == COMBO_A else "")


n_col = max(len(v) for v in by_split.values())
fig, axes = plt.subplots(2, n_col, figsize=(2.8 * n_col, 6.6), layout="constrained", squeeze=False)
for ax in axes.flat:
    ax.axis("off")
face_rows = []
for r, split_name in enumerate(["train", "val"]):
    for ax, it in zip(axes[r], by_split[split_name]):
        x, y, w, h = it["box"]
        with Image.open(it["path"]) as im:
            W, H = im.size
            crop = im.convert("RGB").crop((int(max(0, x - FACE_MARGIN)), int(max(0, y - FACE_MARGIN)),
                                           int(min(W, np.ceil(x + w + FACE_MARGIN))), int(min(H, np.ceil(y + h + FACE_MARGIN)))))
        ax.imshow(np.asarray(crop))
        title = f"{split_name}\n{photo_name(it['path'])}"
        if it["scores"] is not None:
            score_text = " / ".join("-" if s is None else f"{s:.2f}" for s in it["scores"])
            title += f"\n30308 점수 {score_text}"
            face_rows.append({"사진": photo_name(it["path"]), "30308 점수 (s0 / s1 / s2)": score_text})
        ax.set_title(title, fontsize=8)
fig.suptitle(f"{FACE_CAT} {CAT_NAME.get(FACE_CAT)} 박스 모음 (위: train 라벨 전체, 아래: val 정답 전체 / "
             "val 점수: seed별로 그 알약에 낸 30308 박스의 최고 점수)", fontsize=10)
plt.show()

print("val 30308 알약별 30308 박스의 최고 점수 (IoU 0.5 이상, 점수 0.01 이상 예측 중)")
display(pd.DataFrame(face_rows).set_index("사진"))

### 7-2. 박스가 조금 어긋난 약 (18147, 16262)

- 볼 것: 6-4에서 IoU 0.95 기준으로는 맞는 박스가 아예 없는 정답이 있었던 18147(리리카캡슐 150mg)과 16262(크레스토정 20mg)의 val 정답 박스 10개(약마다 5개)다.
- 왜 보는가: 이 약들은 예측 박스가 정답과 IoU 0.93~0.95로 아주 조금 어긋난다. 예측 박스가 알약에서 어긋난 것인지, 정답 라벨 박스가 알약보다 조금 크거나 작게 그려진 것인지는 사진에서 알약 가장자리와 두 박스를 비교해야 알 수 있다.
- 그림 읽는 법
    - 줄은 약(18147, 16262), 칸은 정답 박스 하나다. 정답 박스 주변(여백 20픽셀)만 잘라 크게 보여 준다.
    - 초록 점선: 정답 박스(7-1과 같다).
    - 노랑, 빨강, 자홍 실선: seed 0, 1, 2의 같은 약 예측(점수 0.01 이상) 중 정답과 IoU가 가장 큰 박스다. 6-4에서 "IoU 0.95를 넘는 박스가 있는가"를 정한 박스다. 파란 배경의 사진 위에서 잘 보이도록 4-6과 다른 색을 쓴다.
    - 칸 제목: 약 번호와 이름, 사진(조합과 각도), seed별 IoU다. IoU가 0.95 미만이면 그 seed는 IoU 0.95 기준에서 이 정답을 찾지 못한 것이다.
- 표: 정답마다 seed별 IoU 최고값과 그 박스의 점수, 6-3에서 점수 0.5 이상으로 짝지은 박스의 IoU를 낸다. 두 박스가 다를 수 있다.
- 판단 기준 (그림을 보고 직접 판단한다)
    - 초록 점선이 알약 가장자리에 딱 맞는지, 알약보다 크거나 작거나 한쪽으로 치우쳤는지
    - 예측 박스(세 seed)가 알약 가장자리에 더 잘 맞는지
    - 세 seed의 예측 박스가 같은 쪽으로 어긋나는지. 그렇다면 모델이 일관되게 그렇게 보는 것이고, 정답 박스 쪽이 다를 가능성도 있다.
- 한계: 차이가 몇 픽셀이라 그림으로는 잘 안 보일 수 있다.

In [ ]:
BOX_CATS = [18147, 16262]                              # 6-4에서 IoU 0.95를 넘는 박스가 없는 정답이 있었던 약
BOX_COLORS = {0: "yellow", 1: "red", 2: "magenta"}     # 사진 위에서 잘 보이는 색
ZOOM_MARGIN = 20                                       # 정답 박스 주변 여백(픽셀)


def best_box(seed, gt):
    """seed의 같은 약 예측(점수 0.01 이상) 중 정답과 IoU가 가장 큰 박스 → (IoU, 점수, [x, y, w, h])"""
    p = val_pred[(val_pred["seed"] == seed) & (val_pred["file_name"] == gt.file_name) & (val_pred["category_id"] == gt.category_id)]
    iou = cross_iou(p[BOX_COLS].to_numpy(float), np.array([[gt.x, gt.y, gt.w, gt.h]]))[:, 0]
    i = int(iou.argmax())
    return float(iou[i]), float(p["score"].iloc[i]), p[BOX_COLS].iloc[i].to_numpy(float)


def pair_iou_text(seed, gt):
    """6-3에서 점수 0.5 이상으로 짝지은 같은 약 박스의 IoU. 짝이 없으면 '-'"""
    e = gt_eval[(gt_eval["seed"] == seed) & (gt_eval["file_name"] == gt.file_name) & (gt_eval["정답 약"] == gt.category_id)].iloc[0]
    return f"{e['IoU']:.3f}" if e["분류"] in ("맞음", "위치 부정확") else "-"


box_gts = val_gt[val_gt["category_id"].isin(BOX_CATS)].sort_values(["category_id", "file_name"])
n_col = int(box_gts["category_id"].value_counts().max())
fig, axes = plt.subplots(len(BOX_CATS), n_col, figsize=(3.4 * n_col, 4.2 * len(BOX_CATS)), layout="constrained", squeeze=False)
for ax in axes.flat:
    ax.axis("off")
box_rows = []
for i, cat in enumerate(BOX_CATS):
    for ax, gt in zip(axes[i], box_gts[box_gts["category_id"] == cat].itertuples()):
        with Image.open(VAL_IMG_DIR / gt.file_name) as im:
            W, H = im.size
            x0, y0 = int(max(0, gt.x - ZOOM_MARGIN)), int(max(0, gt.y - ZOOM_MARGIN))
            x1, y1 = int(min(W, np.ceil(gt.x + gt.w + ZOOM_MARGIN))), int(min(H, np.ceil(gt.y + gt.h + ZOOM_MARGIN)))
            ax.imshow(np.asarray(im.convert("RGB").crop((x0, y0, x1, y1))), extent=(x0, x1, y1, y0))
        ax.add_patch(Rectangle((gt.x, gt.y), gt.w, gt.h, fill=False, edgecolor=GT_COLOR, lw=1.6, ls="--"))
        best = {}
        for seed in SEEDS:
            iou, score, (bx, by, bw, bh) = best_box(seed, gt)
            best[seed] = (iou, score)
            ax.add_patch(Rectangle((bx, by), bw, bh, fill=False, edgecolor=BOX_COLORS[seed], lw=1.2))
        ax.set_xlim(x0, x1)
        ax.set_ylim(y1, y0)
        combo, angle = gt.file_name.split("_")[0], gt.file_name.split("_")[5]
        ax.set_title(f"{cat} {CAT_NAME.get(cat)}\n{combo} 각도 {angle}\nIoU " + " / ".join(f"{best[s][0]:.3f}" for s in SEEDS),
                     fontsize=8)
        box_rows.append({"약 번호": cat, "사진": f"{combo} 각도 {angle}",
                         "IoU 최고 (s0 / s1 / s2)": " / ".join(f"{best[s][0]:.3f}" for s in SEEDS),
                         "그 박스의 점수": " / ".join(f"{best[s][1]:.2f}" for s in SEEDS),
                         "6-3 짝의 IoU": " / ".join(pair_iou_text(s, gt) for s in SEEDS)})
handles = [Rectangle((0, 0), 1, 1, fill=False, edgecolor=GT_COLOR, lw=1.6, ls="--")]
handles += [Rectangle((0, 0), 1, 1, fill=False, edgecolor=BOX_COLORS[s], lw=1.2) for s in SEEDS]
fig.legend(handles=handles, labels=["정답", *[f"seed {s} (IoU 가장 큰 같은 약 예측)" for s in SEEDS]],
           loc="outside lower center", ncol=4, fontsize=9)
fig.suptitle("정답 박스 주변 확대 (초록 점선: 정답 / 실선: seed별로 정답과 IoU가 가장 큰 같은 약 예측)", fontsize=11)
plt.show()

print("정답 박스별 IoU (IoU 0.95 미만이면 그 seed는 IoU 0.95 기준에서 이 정답을 찾지 못했다)")
display(pd.DataFrame(box_rows).set_index(["약 번호", "사진"]))

### 7-3. 정답 없는 곳 (사진 7장)

- 볼 것: 6-3에서 "정답 없는 곳"으로 나온 예측 11개(seed 0에서 5개, seed 2에서 6개)가 있는 val 사진 7장이다. 이 예측들은 라벨이 있는 어떤 알약과도 IoU가 0이었다. 즉 정답 박스와 전혀 겹치지 않는다.
- 그림 읽는 법
    - 칸마다 사진 한 장 전체를 그린다. 예측이 사진의 어디에 있는지 보려는 것이라 자르지 않는다.
    - 초록 점선: 정답 박스다. 아래에 "정답 약 번호"를 적는다.
    - 빨강 실선: "정답 없는 곳" 예측이다. 위에 "seed 약 번호 점수"를 적는다(예: s0 41768 0.76). 같은 곳에 여러 seed가 냈으면 라벨을 위로 쌓는다.
    - 정답과 짝지어진 예측 등 다른 예측은 그리지 않는다.
- 표: 예측마다 seed, 사진, 예측한 약 번호와 이름, 점수를 낸다.
- 판단 기준 (그림을 보고 직접 판단한다)
    - 빨간 박스 아래에 알약이 있는지. 있다면 라벨이 빠진 알약이다(라벨 누락).
    - 알약이 없다면 무엇이 있는지(그림자, 반사, 배경 무늬, 먼지 등)
- 실행: 6-3, 7-1, 7-1 추가를 실행한 뒤 실행한다(7-1의 색, 7-1 추가의 `photo_name`을 쓴다).

In [ ]:
# 6-3의 "정답 없는 곳" 예측에 박스 좌표를 붙인다 (val_pred에서 seed, 사진, 약 번호, 점수가 같은 줄)
nowhere = extra_eval[extra_eval["분류"] == "정답 없는 곳"].merge(
    val_pred, left_on=["seed", "file_name", "예측 약", "점수"], right_on=["seed", "file_name", "category_id", "score"])
assert len(nowhere) == int((extra_eval["분류"] == "정답 없는 곳").sum()), "박스 좌표를 찾지 못한 예측이 있습니다."
nowhere_files = sorted(nowhere["file_name"].unique())
print(f"정답 없는 곳 예측: {len(nowhere)}개 / 사진 {len(nowhere_files)}장")

n_col = 4
n_row = int(np.ceil(len(nowhere_files) / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(4.2 * n_col, 5.6 * n_row), layout="constrained", squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, f in zip(axes.flat, nowhere_files):
    with Image.open(VAL_IMG_DIR / f) as im:
        ax.imshow(np.asarray(im.convert("RGB")))
    for r in val_gt[val_gt["file_name"] == f].itertuples():
        ax.add_patch(Rectangle((r.x, r.y), r.w, r.h, fill=False, edgecolor=GT_COLOR, lw=1.8, ls="--"))
        ax.annotate(f"정답 {r.category_id}", (r.x, r.y + r.h), xytext=(0, -3), textcoords="offset points", va="top",
                    fontsize=7.5, color="black", bbox=dict(facecolor=GT_COLOR, edgecolor="none", pad=1))
    placed = []  # 이미 라벨을 적은 박스
    for _, r in nowhere[nowhere["file_name"] == f].sort_values(["seed", "score"], ascending=[True, False]).iterrows():
        box = np.array([[r["x"], r["y"], r["w"], r["h"]]])
        ax.add_patch(Rectangle((r["x"], r["y"]), r["w"], r["h"], fill=False, edgecolor=NG_COLOR, lw=1.8))
        k = sum(cross_iou(box, b)[0, 0] >= IOU_SAME for b in placed)  # 같은 곳에 이미 적은 라벨 수만큼 위로 올린다
        ax.annotate(f"s{r['seed']} {r['category_id']} {r['score']:.2f}", (r["x"], r["y"]), xytext=(0, 3 + 12 * k),
                    textcoords="offset points", va="bottom", fontsize=7.5, color="white",
                    bbox=dict(facecolor=NG_COLOR, edgecolor="none", pad=1))
        placed.append(box)
    ax.set_title(photo_name(VAL_IMG_DIR / f), fontsize=9)
fig.suptitle(f"정답 없는 곳 예측이 나온 val 사진 (초록 점선: 정답, 빨강: 정답 없는 곳 예측, 점수 {CONF_HI} 이상)", fontsize=11)
plt.show()

nowhere_table = pd.DataFrame({"seed": nowhere["seed"], "사진": [photo_name(VAL_IMG_DIR / f) for f in nowhere["file_name"]],
                              "예측 약": nowhere["category_id"], "약 이름": [CAT_NAME.get(c) for c in nowhere["category_id"]],
                              "점수": nowhere["score"].round(3)})
print("정답 없는 곳 예측 목록")
display(nowhere_table.sort_values(["사진", "seed"]).set_index(["사진", "seed"]))

### 7번 결과

조합 A (7-1)

그림을 보고 판단한 것

- 20238(플라빅스정 75mg) 알약은 22347(자누비아정 50mg)의 train 예시와 비슷해 보인다. 6-3에서 20238을 22347로 본 것은, 생김새가 닮은 배운 약끼리 헷갈린 것으로 볼 수 있다. 5-6의 test 945번 사진에서 본 것과 같은 현상이다.
- 30308(트라젠타듀오정 2.5/850mg) 알약은 27926(울트라셋이알서방정)의 train 예시와 색은 약간 비슷하지만 다르고, 각인은 확실히 다르다. 닮은 약이라서 27926으로 봤다고 보기는 어렵다. 왜 27926으로 봤는지는 이 그림으로는 알 수 없다.
- 조합 A의 30308과 train 예시의 30308은 같은 약이지만 찍힌 면이 다르다. 조합 A에서는 "D2/850" 글자가 있는 뒷면이, train 예시에서는 문양이 있는 앞면이 찍혔다. JSON의 식별 문자도 앞면은 마크, 뒷면은 "D2/850"이다.
    - 모델이 30308의 두 면을 모두 배웠는지는 이 그림으로는 알 수 없다. train 예시는 약마다 한 장뿐이고, val에서 30308을 맞힌 다른 사진이 어느 면인지도 이 그림에는 없다.
- 조합 A 사진 3장은 알약이 서로 붙어 있거나 가려지지 않았고, 조명과 초점도 크게 문제 없어 보인다. 사진 상태 때문에 틀린 것은 아닌 것으로 보인다.

그림의 라벨로 본 seed별 차이 (6-3의 숫자와 같다)

- 20238 알약
    - seed 0: 세 장 모두 22347만 냈다(각도 70, 75, 90에서 0.75, 0.87, 0.82).
    - seed 1: 각도 70과 90에서는 20238(0.64, 0.71)과 22347(0.62, 0.54)을 함께 냈고, 각도 75에서는 22347(0.77)만 냈다.
    - seed 2: 세 장 모두 22347(0.78, 0.86, 0.85)을 냈고, 각도 70에서는 28763(0.51)도 냈다.
- 30308 알약
    - seed 0: 세 장 모두 점수 0.5 이상 박스가 없다.
    - seed 1: 27926(0.73, 0.66, 0.74)을 냈고, 각도 75에서는 16551(0.54)도 냈다.
    - seed 2: 27926(0.75, 0.79, 0.83)을 냈고, 각도 70에서는 29345(0.59)도 냈다.
- 3483과 19861은 세 seed 모두 맞혔다. 19861은 seed 0의 각도 90과 seed 1의 각도 75에서 같은 약 박스를 한 번 더 냈다(0.83, 0.52).
- 같은 사진의 같은 알약이라도, 맞는 약 번호를 점수 0.5 이상으로 내는지는 seed에 따라 달라진다.

30308의 앞면과 뒷면 (7-1 추가)

그림을 보고 판단한 것

- train의 30308 박스 3개(K-003483-025469-030308-035206의 각도 70, 75, 90)는 모두 문양만 보이는 앞면이다. 모델은 학습 때 30308의 뒷면("D2/850" 글자 면)을 본 적이 없다.
- val의 30308 6개는 앞면 3개(K-003483-016232-020877-030308)와 뒷면 3개(조합 A)다. 조합 A의 30308은 3장 모두 뒷면이다.
- 뒷면에서 30308로 낸 낮은 점수는, 각인을 보고 맞힌 것이 아니라 색과 모양만 보고 낸 점수가 우연히 맞았을 가능성이 있다.

표로 본 것 (seed 0 / 1 / 2에서 그 알약에 낸 30308 박스의 최고 점수)

| 면 | 사진 | 30308 점수 |
|---|---|---|
| 앞면 | K-003483-016232-020877-030308 각도 70 | 0.88 / 0.80 / 0.84 |
| 앞면 | K-003483-016232-020877-030308 각도 75 | 0.89 / 0.82 / 0.85 |
| 앞면 | K-003483-016232-020877-030308 각도 90 | 0.88 / 0.81 / 0.84 |
| 뒷면 | 조합 A 각도 70 | 0.29 / 0.36 / 0.08 |
| 뒷면 | 조합 A 각도 75 | 0.28 / 0.39 / 0.05 |
| 뒷면 | 조합 A 각도 90 | 0.22 / 0.36 / 0.05 |

- 학습한 앞면은 train과 다른 조합의 사진인데도 세 seed 모두 0.80~0.89로 높고, 학습하지 않은 뒷면은 0.05~0.39로 낮다.

정리

- 조합 A에서 30308을 틀린 것은, train에 한 면(앞면)만 있는 약을 다른 면(뒷면)으로 보여 준 경우다. 4번과 6번에서 숫자만으로는 알 수 없던 이유다.
- train 라벨이 3개뿐인(한 조합의 사진 3장뿐인) 다른 약도 30308처럼 한 면만 학습했을 수 있다. 다만 7-1 추가는 30308 한 약만 본 것이라, 다른 약도 그런지는 알 수 없다.

박스가 조금 어긋난 약 (7-2)

그림을 보고 판단한 것

- 정답 박스(초록 점선)는 완전히 같지는 않지만, 알약 가장자리에 얼추 맞게 그려져 있다.
- 예측 박스가 정답 박스보다 더 잘 맞는지는 알 수 없다. 예측 박스도 정답 박스와 비슷하게 맞췄다.
- 세 seed의 예측 박스도 서로 큰 차이가 없다.

표로 본 것

- IoU 0.95를 넘는 박스가 없는 정답 수는 18147이 2 / 2 / 1개, 16262가 1 / 1 / 2개로, 6-4의 최대 recall과 같다.
- 못 넘는 정답이 몇 장의 사진에 몰려 있고, 세 seed가 같은 사진에서 비슷한 값으로 못 넘는다.
    - 18147: K-003351-016688-018147의 각도 70(0.948 / 0.946 / 0.951)과 각도 75(0.944 / 0.934 / 0.936)
    - 16262: K-003351-016262-033880의 각도 90(0.942 / 0.941 / 0.944). seed 2는 K-003351-016262-018147의 각도 90(0.949)도 못 넘는다.
- 18147의 seed 0은 점수 0.5 이상으로 짝지은 박스의 IoU가 5개 중 4개에서 0.938~0.948로, 0.95에 조금 못 미친다. 그중 2개는 점수 0.02인 박스가 0.95를 넘었다(0.978, 0.967). 그래서 IoU 0.95 기준에서는 맞은 박스가 많은 오답 뒤로 밀려, 6-4에서 AP가 0.052까지 떨어졌다.
- 18147의 seed 1, 2와 16262의 세 seed는 점수 0.5 이상으로 짝지은 박스가 IoU가 가장 큰 박스와 같다.

정리

- 18147과 16262의 mAP75-95가 낮은 것은, 정답 박스와 예측 박스가 모두 알약을 얼추 감싸지만 몇 픽셀 차이로 IoU 0.95에 못 미치는 사진이 있기 때문으로 보인다. 어느 쪽 박스가 더 정확한지는 그림으로 가릴 수 없었다.
- 세 seed가 같은 사진에서 비슷한 값으로 못 넘으므로, seed에 따른 우연이라기보다 모델이 그 사진에서 일관되게 비슷한 박스를 내는 것으로 보인다.

정답 없는 곳 (7-3)

그림을 보고 판단한 것

- 빨간 박스 아래에는 알약이 없고, 배경 부분이다. 라벨이 빠진 알약은 아니다.
- 대부분 둥근 배경의 가장자리에 하얀색이 살짝 보이는 곳을 알약으로 착각한 것으로 보인다. 빛이 반사된 부분인지는 알 수 없다.

그림과 표로 본 것

- 빨간 박스 11개는 모두 사진의 모서리, 둥근 배경의 바깥 가장자리에 있다.
- 예측한 약 번호는 41768(카발린캡슐 25mg) 3개, 18147(리리카캡슐 150mg) 3개, 35206(아토젯정 10/40mg) 4개, 31885(자누메트엑스알서방정 100/1000mg) 1개다. JSON의 색은 41768이 노랑과 하양, 18147과 35206이 하양, 31885가 파랑이다. 4종 중 3종이 하얀색이 들어간 약이다.
- 점수는 0.513~0.764다. 6-3에서 본 것처럼, 같은 약 번호의 맞은 박스보다 점수가 낮아 val AP는 깎지 않았다.

7번 정리

- 약 번호를 틀린 장면(7-1)
    - 20238을 22347로 본 것은, 생김새가 닮은 배운 약끼리 헷갈린 것이다.
    - 30308을 27926으로 본 것은 두 약이 닮아서가 아니다. train에 앞면만 있는 30308을 뒷면으로 보여 줘서, 맞는 약 번호의 점수가 낮게(0.05~0.39) 나왔다(7-1 추가). 같은 약이라도 학습하지 않은 면은 잘 알아보지 못한다.
- 박스가 조금 어긋난 약(7-2): 정답 박스와 예측 박스가 모두 알약을 얼추 감싸고, 몇 픽셀 차이로 IoU 0.95에 못 미친다. 어느 쪽 박스가 더 정확한지는 그림으로 가릴 수 없었다.
- 정답 없는 곳(7-3): 라벨 누락이 아니라, 사진 모서리의 하얗게 보이는 배경을 알약으로 본 오답이다.
- 사진 상태(가림, 겹침, 조명, 초점)가 나빠서 틀린 경우는 이번에 본 사진에서는 없었다.

## 8. 정리

- 1차 실험(RT-DETR-L, 라벨 수정 반영, Ultralytics 기본 설정, seed 0, 1, 2)의 결과를 정리하고(8-1, 8-2), 그 결과에서 나온 2차 실험의 방향을 적는다(8-3).
- 새로 계산한 것은 없고, 앞 섹션의 결과를 옮겨 적는다. 괄호 안은 근거가 된 셀이다.

### 8-1. 결과 요약

| seed | val mAP50 | val mAP50-95 | val mAP75-95 | Kaggle public mAP@[0.75:0.95] | val mAP75-95 − Kaggle |
|---|---|---|---|---|---|
| 0 | 0.995 | 0.988 | 0.982 | 0.4215 | 0.560 |
| 1 | 0.993 | 0.987 | 0.980 | 0.4176 | 0.563 |
| 2 | 0.980 | 0.976 | 0.973 | 0.4201 | 0.553 |
| 평균 | 0.989 | 0.984 | 0.978 | 0.4198 | 0.559 |

- val 점수는 best.pt로 val 47장을 다시 잰 값이다(4-7, 6-2에서 같은 값이 두 번 나왔다). 4번 결과 첫 표는 학습 직후 Ultralytics가 잰 값이라 소수 셋째 자리가 조금 다르고, 5번 결과의 차이(평균 0.557)는 그 값으로 계산했다.
- Kaggle 점수는 웹사이트에 CSV를 올려 받은 public 점수다(5-3).

섹션별로 알아낸 것

| 섹션 | 알아낸 것 |
|---|---|
| 4. 학습 | 세 seed 모두 100 epoch를 끝까지 돌았고, val mAP는 약 40 epoch부터 거의 평평하다(4-6). 증강 없이 잰 train 점수가 0.98~0.99라 과소적합은 아니다. 같은 30종에서 train과 val의 차이는 0.007~0.020으로 작다(4-7). 과적합은 단정하지 않았다. 4번에서 과적합 신호로 본 30308은, 7번에서 train에 한 면만 있었던 것으로 확인했다(7-1 추가). |
| 5. test | Kaggle 점수는 0.418~0.422로 val보다 약 0.56 낮고, seed에 따라 거의 흔들리지 않는다(5-3). test 사진의 14~20%에서 점수 0.5 이상 박스가 5개 이상 나왔고, 대부분 한 알약에 다른 약 번호 박스가 겹친 것이다(5-2, 5-4). 사진으로 판단한 알약 20개 중 8개가 train에 없는 약이었고, 모델은 이 8개를 모두 배운 약 번호로 0.54~0.91의 점수로 예측했다(5-6). |
| 6. val 분석 | val 30종 중 22종은 세 seed 모두 mAP75-95가 최고값(0.995)이다(6-2). 남은 손실은 두 가지다. 박스가 정답과 IoU 0.93~0.95로 조금 어긋나 IoU 0.95 단계에서 깎이는 것(7종)과, 한 조합(조합 A)에서 20238과 30308을 다른 약 번호로 높게 본 것이다(6-3). AP가 크게 깎인 것은 30308의 seed 1, 2뿐이다(6-4). |
| 7. 사진 확인 | 20238은 생김새가 닮은 22347과 헷갈렸다. 30308은 train에 앞면만 있어 뒷면을 알아보지 못했다(7-1, 7-1 추가). 박스 차이는 몇 픽셀이라 예측과 정답 중 어느 쪽이 정확한지 가릴 수 없었다(7-2). 정답 없는 곳 예측은 사진 모서리의 하얀 배경이었다(7-3). |

### 8-2. 점수를 깎은 원인

- 확실한 정도는 세 가지로 적는다.
    - 사진으로 확인: 그림을 보고 사람이 판단했다.
    - 숫자로 확인: 정답과 비교한 계산으로 확인했다.
    - 추정: 정답이 없거나 계산 방식을 몰라 확인하지 못했다.

test 쪽 (val과 test의 차이, 약 0.56)

| 원인 | 내용 | 근거 | 확실한 정도 |
|---|---|---|---|
| 배우지 않은 약 | test에는 train에 없는 약이 있다. 모델은 학습한 56종의 번호만 낼 수 있어서, 그 약은 맞힐 수 없다. 게다가 닮은 배운 약의 번호로 점수가 높은 오답을 내서(0.54~0.91), 그 배운 약의 AP도 깎는다 | 대회 설명, 5번 결과의 계산, 5-6 | 있다는 것은 사진으로 확인(알약 20개 중 8개). 점수를 얼마나 깎았는지는 추정 |
| 배운 약끼리 헷갈림 | 한 알약에 맞는 약과 닮은 다른 약 번호를 함께 높은 점수로 낸다. 틀린 약 번호의 AP를 깎는다 | 5-4, 5-6 | 사례는 사진으로 확인(겹친 쌍 8개 중 7개가 배운 약). 전체 비율은 추정 |

val 쪽 (val에서 남은 손실, 0.02 안팎)

| 원인 | 내용 | 근거 | 확실한 정도 |
|---|---|---|---|
| 한 면만 학습한 약 | 30308은 train에 앞면만 있다. val의 뒷면 3개(조합 A)에서는 맞는 약 번호의 점수가 0.05~0.39로 낮고, seed 1, 2는 27926을 0.66~0.83으로 냈다. 그래서 30308의 AP가 0.924, 0.538로 떨어졌다 | 6-3, 6-4, 7-1 추가 | 사진과 숫자로 확인. 다른 약도 한 면만 학습했는지는 모름 |
| 닮은 배운 약 | 20238 알약에 22347을 높은 점수로 냈다. 점수 0.5 기준으로 20238은 정답 7개 중 3 / 6 / 4개만 찾았다. 다만 점수 순서로는 맞은 박스가 오답보다 앞이라 val AP는 깎이지 않았다. test에서도 같은 현상이 있었다(5-6의 945번 사진) | 6-3, 6-4, 7-1 | 사진과 숫자로 확인 |
| 박스의 몇 픽셀 차이 | 같은 약으로 찾은 박스가 정답과 IoU 0.93~0.95라, 대회 지표에서 가장 엄격한 IoU 0.95 단계에서 깎인다. 7종의 mAP75-95가 0.995보다 낮고, 18147과 16262는 IoU 0.95의 AP가 0.052~0.715다 | 6-2, 6-3, 6-4, 7-2 | 숫자로 확인. 예측과 정답 중 어느 쪽 박스가 어긋났는지는 모름 |
| 배경 모서리 오답 | 사진 모서리의 하얀 배경을 알약(주로 하얀색이 들어간 약 번호)으로 봤다(seed 0에서 5개, seed 2에서 6개, 점수 0.51~0.76). 맞은 박스보다 점수가 낮아 val AP는 깎이지 않았다 | 6-3, 7-3 | 사진으로 확인. test에서 얼마나 나오는지는 모름 |

크기 비교

- val에서 남은 손실은 0.02 안팎이고, val과 test의 차이는 약 0.56이다. 그래서 test 점수를 낮춘 주된 원인은 val로는 볼 수 없는 것, 즉 test가 train, val과 다르다는 점(배우지 않은 약 등)으로 보인다. val에서 찾은 원인들은 test에서도 일어날 수 있지만, 그것만으로는 0.56의 차이를 설명할 수 없다.

이 실험에서 확인하지 못한 것

- Kaggle 지표의 정확한 계산 방식, public 점수가 test의 몇 %로 매겨지는지(5-3, 5번 결과)
- test의 실제 약 구성과 배우지 않은 약의 종류 수(5번 결과, 5-6)
- 7-2에서 예측 박스와 정답 박스 중 어느 쪽이 정확한지
- 30308 말고 다른 약도 한 면만 학습했는지(7-1 추가)
- 30308의 뒷면을 왜 하필 27926으로 봤는지(7-1)

결과를 읽을 때 주의할 점

- val은 47장, 30종뿐이다. val에서 약 번호를 틀린 정답 박스 17개 중 16개가 한 조합(사진 3장)에서 나왔다(6-3). 그래서 val에서 본 실수들이 얼마나 자주 일어나는지는 말하기 어렵다.
- train에만 있는 26종은 val에 없어서, 처음 보는 사진에서도 잘 맞히는지 val로는 확인하지 못했다.

### 8-3. 2차 실험의 방향

- 2차 실험은 AI Hub에서 데이터를 더 가져와 학습한다. 여기서는 1차 결과에서 나온, 2차 실험에서 확인할 것과 볼 것을 적는다. 정해진 계획이 아니라 후보다.
- 괄호 안은 근거가 된 셀이다. 확인하지 못한 것은 추정이라고 적는다.

데이터를 넣을 때 확인할 것

1. 약 번호를 대회와 같은 방식으로 매기기
    - 대회 설명에 따르면 원본 데이터는 category_id가 모두 1이었고, 대회 데이터는 dl_idx로 다시 매겼다. AI Hub에서 가져온 데이터도 같은 규칙으로 바꿔야 제출 CSV의 약 번호가 Kaggle 정답과 맞는다. bbox 유효성 규칙(bbox 키가 있고 길이가 4)도 같게 맞춘다.
    - 모델은 학습한 번호만 낼 수 있으므로(5번 결과), 이 단계가 틀리면 데이터를 넣어도 점수에 반영되지 않는다.
2. 약 종류 늘리기
    - val에서 남은 손실은 0.02 안팎이고 val과 test의 차이는 약 0.56이라, test 점수를 낮춘 주된 원인은 배우지 않은 약으로 보인다(8-2, 추정).
    - 배우지 않은 약은 자기 AP가 0인 데서 끝나지 않는다. 닮은 배운 약 번호로 0.54~0.91의 점수를 내서 그 배운 약의 AP까지 깎는다(5-6).
    - 어떤 약을 넣을지는 test에서 알아본 약 이름과 무관한 기준으로 정한다. 예를 들어 허용된 데이터의 약 전부를 넣거나, 미리 정한 규칙을 쓴다. 사용이 금지된 TL_2_조합.zip, TS_2_조합.zip은 쓰지 않는다.
3. 약마다 앞면과 뒷면이 모두 있는지 확인
    - 30308은 train에 앞면만 있었고, 뒷면에서는 맞는 약 번호의 점수가 0.05~0.39로 떨어졌다(7-1 추가).
    - JSON의 `drug_dir`(알약방향)로 약마다 면별 사진 수를 세서, 한쪽 면만 있는 약이 없는지 본다. 1차 train에서 라벨이 3개뿐인 17종(한 조합의 사진 3장)도 같은 상태일 수 있다.
4. 약마다 라벨 수 확인
    - 1차에서는 라벨이 3개뿐인 약이 test에서 라벨 비율보다 최대 8.2배 많이 예측됐다(5-5).
    - 추가한 뒤 약별 라벨 수 분포를 보고, 극단적으로 치우치면 샘플링을 고려한다.
5. 사진 조건을 대회 사진과 비교
    - 대회 사진은 알약 3~4개가 한 장에 있고, 둥근 배경의 모서리가 밝게 보인다(7-3).
    - 추가 데이터의 사진당 알약 수, `back_color`(촬영배경), `light_color`(촬영조명)를 대회 데이터와 비교한다.
    - 데이터가 지금의 약 52배가 되면 대회 사진은 학습 데이터의 약 2%가 된다. 대회 사진의 비중을 유지하는 방법도 후보다. 예를 들어 대회 사진을 여러 번 쓰거나, AI Hub 데이터로 먼저 학습한 뒤 대회 데이터로 이어서 학습한다.
6. 박스를 그리는 기준이 같은지 확인
    - 대회 지표는 IoU 0.95까지 보기 때문에 몇 픽셀 차이도 점수에 들어간다(6-4, 7-2).
    - AI Hub 라벨이 알약에 딱 맞게 그렸는지, 여백을 두고 그렸는지가 대회 라벨과 다르면, 학습한 박스가 대회 정답과 어긋난다.
    - 같은 약끼리 bbox 크기와 알약 크기(`leng_long`, `leng_short`)의 비율을 두 데이터에서 비교하면 확인할 수 있다(1차 EDA 5-2와 같은 방식).

학습 설정에서 조심할 것

1. `optimizer=auto`가 데이터 크기에 따라 저절로 바뀐다 (Ultralytics 8.4.183 원본 코드로 확인)
    - 규칙: 반복 수 = ceil(train 사진 수 ÷ 64) × epoch다.
        - 반복 수가 10,000을 넘으면 MuSGD(lr 0.01)를 고른다.
        - 넘지 않으면 AdamW를 고르고, lr = 0.01 ÷ (4 + 클래스 수)다.
    - 1차: 185장 → 3 × 100 = 300 → AdamW(lr 0.000167)
    - 지금의 약 52배(약 9,600장)로 100 epoch를 돌리면 151 × 100 = 15,100 → MuSGD(lr 0.01)
    - 클래스 수가 늘면 AdamW의 lr도 바뀐다. 그대로 두면 데이터를 바꾼 효과와 optimizer가 바뀐 효과가 섞인다. optimizer와 lr0을 직접 정해 두거나, 적어도 학습 로그의 optimizer 줄을 확인해 기록한다.
2. epoch
    - 1차는 185장으로 약 40 epoch에 수렴했다(4-6).
    - 데이터가 많아지면 epoch 하나에 걸리는 시간도 그만큼 늘어난다. 1차는 seed마다 1~2.8시간이었다.
    - 짧게 돌려 학습 곡선을 보고 정하는 것이 현실적이다. epoch를 바꾸면 위 optimizer 규칙의 결과도 바뀐다.
3. 입력 크기
    - 1차는 사진(976×1280)을 640×640으로 늘려 맞췄다. 640에서 1픽셀은 원본의 약 1.5~2픽셀이다.
    - IoU 0.95 손실이 몇 픽셀 차이에서 나오므로, 입력 크기를 키우면 박스가 더 정확해질 수 있다(가설).
    - 1차에서도 batch 16에서 GPU 메모리(12GB)를 넘었으므로, 실험으로 확인할 문제다.
4. seed와 비교 방법
    - test 점수의 seed 차이는 0.004였고, val 순위와 test 순위가 달랐다(5번 결과).
    - 실험끼리는 seed 평균으로 비교한다. Kaggle 점수 차이가 0.005보다 작으면 개선이라고 보기 어렵다.

2차 실험 뒤에 볼 것

- 1차 노트북의 분석 셀을 그대로 다시 돌리면 같은 기준으로 비교할 수 있다.

| 볼 것 | 1차 값 | 좋아졌다면 | 셀 |
|---|---|---|---|
| Kaggle 점수 (seed 평균) | 0.4198 | 오른다 | 5-3 |
| val mAP75-95 − Kaggle | 0.559 | 줄어든다. 크게 줄면 "배우지 않은 약"이라는 설명의 근거가 된다 | 8-1 |
| test에서 점수 0.5 이상 박스가 5개 이상인 사진 | 14~20% | 줄어든다 | 5-2 |
| test의 다른 약끼리 겹친 쌍 | 102~193쌍 | 줄어든다 | 5-4 |
| 라벨이 적은 약의 예측 배율 | 최대 8.2 | 1에 가까워진다 | 5-5 |
| 기존 val 47장의 mAP75-95 | 0.978 | 비슷하거나 오른다. 떨어지면 대회 사진에 덜 맞춰진 것을 의심한다 | 6-2 |
| 조합 A의 30308 뒷면 점수 | 0.05~0.39 | 오른다(뒷면을 학습한 경우) | 7-1 추가 |
| 20238을 같은 약으로 찾은 수 (점수 0.5 기준) | 7개 중 3 / 6 / 4 | 늘어난다 | 6-3 |
| IoU 0.95를 못 넘는 정답 (18147, 16262) | 2 / 2 / 1, 1 / 1 / 2 | 줄거나 그대로다. 늘어나면 박스 기준 차이를 의심한다 | 6-4, 7-2 |
| 배경 모서리 오답 | 5 / 0 / 6 | 줄어든다 | 6-3, 7-3 |

- val도 새로 설계해야 한다.
    - 지금 val은 30종 중 22종이 만점이고, 실수가 한 조합에 몰려 있어 개선을 재기 어렵다(8-2).
    - 기존 val 47장은 1차와 비교하려고 그대로 둔다.
    - 새 데이터에서도 같은 알약을 각도만 바꿔 찍은 사진이 train과 val에 나뉘지 않게 따로 떼어 두면, 늘어난 약 종류까지 잴 수 있다.

우선순위 (제안)

1. 약 번호 맞추기와 약 종류 늘리기: test 점수 차이의 주된 원인으로 보이는 쪽이다.
2. 두 면 확인과 사진 조건 비교: 늘린 데이터가 대회 사진에서도 통하게 하는 쪽이다.
3. 박스 기준 확인: IoU 0.95 쪽 손실과 이어진다.
4. 설정 고정(optimizer, epoch)과 입력 크기 실험: 이것만 바꿨을 때의 효과는 1, 2보다 작을 것으로 본다.

- 1이 가장 효과가 클 것이라는 판단은 추정이다. Kaggle의 계산 방식과 test의 실제 약 구성은 확인하지 못했다.